# Self-Contained Kaggle GPU Framework Validation

This notebook is self-contained and does **not** clone from GitHub.

It writes the required source files directly into `/kaggle/working/EECE_608`, installs the dependencies, runs the GPU-scale validation, bundles the outputs into one zip file, and prints a compact summary.


In [ ]:
from pathlib import Path
import os
import shutil

REPO_DIR = Path('/kaggle/working/EECE_608')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
REPO_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "pyproject.toml": """[build-system]
requires = ["setuptools>=69", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "dp-audit-tightness"
version = "0.1.0"
description = "Experimental scaffold for studying the tightness gap between theoretical upper bounds and empirical lower bounds on privacy loss."
readme = "README.md"
requires-python = ">=3.10"
dependencies = [
  "numpy>=1.26",
  "pandas>=2.2",
  "PyYAML>=6.0",
  "matplotlib>=3.8",
  "torch>=2.2",
  "torchvision>=0.17",
  "opacus>=1.5",
  "dp-accounting>=0.4",
]

[tool.setuptools]
package-dir = {"" = "src"}

[tool.setuptools.packages.find]
where = ["src"]

""",
    "codex/run_support_scaled_pilot.py": """from __future__ import annotations

import csv
import json
import statistics
import sys
import time
import traceback
from pathlib import Path


ROOT = Path(__file__).resolve().parents[1]
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dp_audit_tightness.auditing.canary.generation import generate_canaries, insert_canaries_into_dataset
from dp_audit_tightness.auditing.canary.one_run import (
    _infer_image_shape,
    _score_canaries,
    _score_reference_canaries,
)
from dp_audit_tightness.auditing.canary.seeding import build_canary_seed_plan
from dp_audit_tightness.auditing.passive.auditors import run_passive_audit_once
from dp_audit_tightness.config import (
    AuditRunConfig,
    CanaryAuditConfig,
    CanaryConfig,
    DatasetConfig,
    ModelConfig,
    OptimizerConfig,
    PassiveAuditConfig,
    PassiveAuditorConfig,
    PrivacyConfig,
    RunConfig,
    SaturationConfig,
    TrainExperimentConfig,
    TrainingConfig,
    config_to_dict,
)
from dp_audit_tightness.data.datasets import DatasetBundle, load_dataset_bundle
from dp_audit_tightness.models.io import load_model_for_inference, load_model_from_state_dict
from dp_audit_tightness.privacy.empirical import estimate_empirical_lower_bound
from dp_audit_tightness.training.dp_sgd import train_dp_sgd
from dp_audit_tightness.utils.logging_utils import configure_logging
from dp_audit_tightness.utils.results import save_json, save_training_result
from dp_audit_tightness.utils.seeds import set_global_seed


CODEX_DIR = ROOT / "codex"
DATA_DIR = CODEX_DIR / "data"
RESULTS_DIR = CODEX_DIR / "results" / "support_scaled_pilot"
SUMMARY_JSON = RESULTS_DIR / "support_scaled_pilot_summary.json"
SUMMARY_CSV = RESULTS_DIR / "support_scaled_pilot_summary.csv"

PASSIVE_SUPPORT_LEVELS = [
    {"support_label": "smoke", "query_budget": 64, "audit_seeds": [401]},
    {"support_label": "medium", "query_budget": 256, "audit_seeds": [401, 402, 403]},
    {"support_label": "large", "query_budget": 512, "audit_seeds": [401, 402, 403, 404, 405]},
]

CANARY_SUPPORT_LEVELS = [
    {"support_label": "smoke", "num_canaries": 8, "audit_seeds": [101]},
    {"support_label": "medium", "num_canaries": 16, "audit_seeds": [101, 102, 103]},
    {"support_label": "large", "num_canaries": 32, "audit_seeds": [101, 102, 103, 104, 105]},
]


def ensure_adult_npz(target_path: Path) -> tuple[int, int, int]:
    import numpy as np
    import pandas as pd
    from sklearn.datasets import fetch_openml

    if target_path.exists():
        cached = np.load(target_path)
        features = cached["features"]
        labels = cached["labels"]
        return int(features.shape[1]), int(labels.max()) + 1, int(features.shape[0])

    target_path.parent.mkdir(parents=True, exist_ok=True)
    adult = fetch_openml("adult", version=2, as_frame=True)
    features = adult.data.copy()
    labels = adult.target.astype(str).copy()

    features = features.replace("?", pd.NA)
    mask = ~features.isna().any(axis=1) & labels.notna()
    features = features.loc[mask]
    labels = labels.loc[mask]

    encoded = pd.get_dummies(features, dummy_na=False, drop_first=False, dtype="float32")
    label_values = sorted(labels.unique().tolist())
    label_map = {label: index for index, label in enumerate(label_values)}
    y = labels.map(label_map).astype("int64").to_numpy()
    x = encoded.to_numpy(dtype="float32")

    np.savez(target_path, features=x, labels=y)
    metadata = {
        "rows": int(x.shape[0]),
        "input_dim": int(x.shape[1]),
        "num_classes": int(len(label_values)),
        "label_map": label_map,
    }
    save_json(target_path.with_suffix(".metadata.json"), metadata)
    return int(x.shape[1]), int(len(label_values)), int(x.shape[0])


def subset_bundle(bundle: DatasetBundle, train_limit: int, eval_limit: int, seed: int) -> DatasetBundle:
    import torch
    from torch.utils.data import Subset

    generator = torch.Generator().manual_seed(seed)
    train_perm = torch.randperm(len(bundle.train_dataset), generator=generator).tolist()
    eval_perm = torch.randperm(len(bundle.eval_dataset), generator=generator).tolist()
    train_indices = train_perm[: min(train_limit, len(train_perm))]
    eval_indices = eval_perm[: min(eval_limit, len(eval_perm))]
    return DatasetBundle(
        train_dataset=Subset(bundle.train_dataset, train_indices),
        eval_dataset=Subset(bundle.eval_dataset, eval_indices),
        input_dim=bundle.input_dim,
        num_classes=bundle.num_classes,
        train_size=len(train_indices),
        eval_size=len(eval_indices),
    )


def make_train_config(
    *,
    dataset_name: str,
    data_dir: str,
    model_name: str,
    input_dim: int,
    hidden_dim: int,
    num_classes: int,
    learning_rate: float,
    momentum: float,
    split_seed: int,
    training_seed: int,
) -> TrainExperimentConfig:
    return TrainExperimentConfig(
        experiment_name=f"codex_support_scaled_{dataset_name}",
        dataset=DatasetConfig(
            name=dataset_name,
            data_dir=data_dir,
            validation_fraction=0.1,
            num_workers=0,
        ),
        model=ModelConfig(
            name=model_name,
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            num_classes=num_classes,
        ),
        training=TrainingConfig(
            batch_size=128,
            eval_batch_size=256,
            epochs=1,
            clipping_norm=1.0,
            noise_multiplier=1.1,
            optimizer=OptimizerConfig(
                name="sgd",
                learning_rate=learning_rate,
                weight_decay=0.0,
                momentum=momentum,
            ),
        ),
        privacy=PrivacyConfig(delta=1e-5, accountant="rdp"),
        run=RunConfig(
            split_seed=11,
            training_seed=training_seed,
            results_root=str(RESULTS_DIR),
            save_checkpoint=True,
            notes="Codex support-scaled pilot sidecar run.",
        ),
    )


def train_dataset_case(name: str, config: TrainExperimentConfig, bundle: DatasetBundle, logger) -> dict:
    set_global_seed(config.run.training_seed)
    started = time.time()
    outcome = train_dp_sgd(config=config, logger=logger, dataset_bundle=bundle)

    config_path = RESULTS_DIR / "training" / "configs" / f"{outcome.record.training_run_id}_config.json"
    outcome.record.config_snapshot_path = str(config_path)
    save_json(config_path, config_to_dict(config))
    training_result_path = save_training_result(outcome.record, results_root=config.run.results_root)

    return {
        "dataset": name,
        "config": config,
        "bundle": bundle,
        "record": outcome.record,
        "checkpoint_path": outcome.checkpoint_path,
        "training_result_path": str(training_result_path),
        "elapsed_seconds": round(time.time() - started, 3),
    }


def safe_ratio(lower: float | None, upper: float | None) -> float | None:
    if lower is None or upper is None or upper <= 0.0:
        return None
    return lower / upper


def safe_gap(upper: float | None, lower: float | None) -> float | None:
    if upper is None or lower is None:
        return None
    return upper - lower


def aggregate_estimate(member_scores: list[float], nonmember_scores: list[float], delta: float):
    return estimate_empirical_lower_bound(
        member_scores=member_scores,
        nonmember_scores=nonmember_scores,
        delta=delta,
        align_event_to_score_direction=True,
        require_member_favoring=True,
        report_confidence_supported_lower_bound=True,
    )


def run_passive_support_level(
    *,
    dataset_name: str,
    attack_name: str,
    score_type: str,
    training_case: dict,
    support_label: str,
    query_budget: int,
    audit_seeds: list[int],
) -> dict:
    import torch

    config: TrainExperimentConfig = training_case["config"]
    bundle: DatasetBundle = training_case["bundle"]
    record = training_case["record"]

    passive_config = PassiveAuditConfig(
        training_result_path=training_case["training_result_path"],
        audit_regime="passive_final_model_only",
        auditor_variant=attack_name,
        auditor_strength_rank=1,
        delta=config.privacy.delta,
        repeated_seeds=list(audit_seeds),
        passive=PassiveAuditorConfig(
            query_budget=query_budget,
            score_type=score_type,
            calibration_method="none",
            calibration_fraction=0.0,
        ),
        saturation=SaturationConfig(),
        run=AuditRunConfig(
            results_root=str(RESULTS_DIR),
            save_debug_artifacts=False,
        ),
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = load_model_for_inference(config.model, record.model_artifact_path, device=device)

    observations = [
        run_passive_audit_once(
            training_record=record,
            training_config=config,
            config=passive_config,
            seed=audit_seed,
            dataset_bundle=bundle,
            model=model,
        )
        for audit_seed in audit_seeds
    ]
    member_scores = [score for observation in observations for score in observation.member_scores]
    nonmember_scores = [score for observation in observations for score in observation.nonmember_scores]
    estimate = aggregate_estimate(member_scores, nonmember_scores, config.privacy.delta)

    epsilon_upper_rdp = record.epsilon_upper_theory
    epsilon_upper_tighter = record.epsilon_upper_pld
    epsilon_lower = estimate.epsilon_lower_empirical
    point_lower = estimate.epsilon_lower_empirical_point_estimate

    return {
        "dataset": dataset_name,
        "attack": attack_name,
        "support_label": support_label,
        "status": "ok",
        "audit_regime": passive_config.audit_regime,
        "score_name": score_type,
        "query_budget_per_seed": query_budget,
        "num_audit_seeds": len(audit_seeds),
        "audit_seeds": json.dumps(audit_seeds),
        "epsilon_upper_rdp": epsilon_upper_rdp,
        "epsilon_upper_tighter": epsilon_upper_tighter,
        "tighter_upper_backend": record.pld_accounting_backend,
        "epsilon_lower_conservative": epsilon_lower,
        "epsilon_lower_point": point_lower,
        "tightness_ratio_rdp": safe_ratio(epsilon_lower, epsilon_upper_rdp),
        "tightness_ratio_tighter": safe_ratio(epsilon_lower, epsilon_upper_tighter),
        "privacy_loss_gap_rdp": safe_gap(epsilon_upper_rdp, epsilon_lower),
        "privacy_loss_gap_tighter": safe_gap(epsilon_upper_tighter, epsilon_lower),
        "member_favoring": estimate.member_favoring,
        "selected_tpr": estimate.selected_tpr,
        "selected_fpr": estimate.selected_fpr,
        "warning": estimate.warning_message,
        "num_member_samples": estimate.num_member_samples,
        "num_nonmember_samples": estimate.num_nonmember_samples,
    }


def run_canary_support_level(
    *,
    dataset_name: str,
    attack_name: str,
    training_case: dict,
    support_label: str,
    num_canaries: int,
    audit_seeds: list[int],
    strategy: str = "random_canaries",
) -> dict:
    import torch

    config: TrainExperimentConfig = training_case["config"]
    base_bundle: DatasetBundle = training_case["bundle"]
    record = training_case["record"]

    if dataset_name not in {"mnist", "cifar10"}:
        return {
            "dataset": dataset_name,
            "attack": attack_name,
            "support_label": support_label,
            "status": "not_supported",
            "reason": "canary sidecar pilot currently supports image datasets only",
        }

    canary_config = CanaryAuditConfig(
        training_result_path=training_case["training_result_path"],
        audit_regime="evaluator_controlled_canary_stress_test",
        auditor_variant=attack_name,
        auditor_strength_rank=1,
        audit_mode="repeated_run",
        delta=config.privacy.delta,
        repeated_seeds=list(audit_seeds),
        canary=CanaryConfig(
            strategy=strategy,
            num_canaries=num_canaries,
            insertion_rate=0.005,
            optimize_steps=0,
        ),
        saturation=SaturationConfig(),
        run=AuditRunConfig(
            results_root=str(RESULTS_DIR),
            save_debug_artifacts=False,
        ),
    )

    epsilon_upper_rdp_values: list[float] = []
    epsilon_upper_tighter_values: list[float] = []
    member_scores: list[float] = []
    nonmember_scores: list[float] = []
    inserted_example_counts: list[int] = []
    image_shape = _infer_image_shape(config.dataset.name)

    for audit_seed in audit_seeds:
        seed_plan = build_canary_seed_plan(
            experiment_seed=record.training_seed,
            audit_seed=audit_seed,
            dataset_split_seed=record.split_seed,
        )
        canaries = generate_canaries(
            canary_config.canary,
            seed=seed_plan.canary_generation_seed,
            num_classes=config.model.num_classes,
            image_shape=image_shape,
        )
        insertion_result = insert_canaries_into_dataset(
            base_bundle.train_dataset,
            canaries,
            insertion_rate=canary_config.canary.insertion_rate,
            seed=seed_plan.canary_insertion_seed,
        )
        augmented_bundle = DatasetBundle(
            train_dataset=insertion_result.augmented_train_dataset,
            eval_dataset=base_bundle.eval_dataset,
            input_dim=base_bundle.input_dim,
            num_classes=base_bundle.num_classes,
            train_size=base_bundle.train_size + insertion_result.inserted_example_count,
            eval_size=base_bundle.eval_size,
        )
        canary_train_config = TrainExperimentConfig(
            experiment_name=f"{config.experiment_name}_{attack_name}_{support_label}",
            dataset=config.dataset,
            model=config.model,
            training=config.training,
            privacy=config.privacy,
            run=RunConfig(
                split_seed=config.run.split_seed,
                training_seed=seed_plan.retrain_seed,
                results_root=str(RESULTS_DIR),
                save_checkpoint=False,
                notes="Codex support-scaled canary retraining run.",
            ),
        )
        set_global_seed(seed_plan.retrain_seed)
        outcome = train_dp_sgd(
            config=canary_train_config,
            logger=configure_logging(),
            dataset_bundle=augmented_bundle,
            save_checkpoint=False,
            run_descriptor=f"{attack_name}_{dataset_name}_{support_label}",
            return_model_state=True,
        )
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = load_model_from_state_dict(config.model, outcome.model_state_dict, device=device)
        member_scores.extend(_score_canaries(model, insertion_result.inserted_canaries, device=device))
        nonmember_scores.extend(_score_reference_canaries(model, insertion_result.reference_canaries, device=device))
        epsilon_upper_rdp_values.append(outcome.record.epsilon_upper_theory)
        epsilon_upper_tighter_values.append(outcome.record.epsilon_upper_pld)
        inserted_example_counts.append(insertion_result.inserted_example_count)

    estimate = aggregate_estimate(member_scores, nonmember_scores, config.privacy.delta)
    epsilon_upper_rdp = statistics.fmean(epsilon_upper_rdp_values)
    epsilon_upper_tighter = statistics.fmean(epsilon_upper_tighter_values)
    epsilon_lower = estimate.epsilon_lower_empirical
    point_lower = estimate.epsilon_lower_empirical_point_estimate

    return {
        "dataset": dataset_name,
        "attack": attack_name,
        "support_label": support_label,
        "status": "ok",
        "audit_regime": canary_config.audit_regime,
        "score_name": "target_label_logit_margin",
        "num_canaries_per_seed": num_canaries,
        "num_audit_seeds": len(audit_seeds),
        "audit_seeds": json.dumps(audit_seeds),
        "mean_inserted_example_count": round(statistics.fmean(inserted_example_counts), 3),
        "epsilon_upper_rdp": epsilon_upper_rdp,
        "epsilon_upper_tighter": epsilon_upper_tighter,
        "tighter_upper_backend": "google_dp_accounting",
        "epsilon_lower_conservative": epsilon_lower,
        "epsilon_lower_point": point_lower,
        "tightness_ratio_rdp": safe_ratio(epsilon_lower, epsilon_upper_rdp),
        "tightness_ratio_tighter": safe_ratio(epsilon_lower, epsilon_upper_tighter),
        "privacy_loss_gap_rdp": safe_gap(epsilon_upper_rdp, epsilon_lower),
        "privacy_loss_gap_tighter": safe_gap(epsilon_upper_tighter, epsilon_lower),
        "member_favoring": estimate.member_favoring,
        "selected_tpr": estimate.selected_tpr,
        "selected_fpr": estimate.selected_fpr,
        "warning": estimate.warning_message,
        "num_member_samples": estimate.num_member_samples,
        "num_nonmember_samples": estimate.num_nonmember_samples,
    }


def write_summary(summary: list[dict]) -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with SUMMARY_JSON.open("w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2, sort_keys=True)

    fieldnames: list[str] = []
    for row in summary:
        for key in row.keys():
            if key not in fieldnames:
                fieldnames.append(key)
    with SUMMARY_CSV.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in summary:
            writer.writerow(row)


def main() -> None:
    logger = configure_logging()
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    adult_npz = DATA_DIR / "adult.npz"
    adult_input_dim, adult_num_classes, adult_rows = ensure_adult_npz(adult_npz)
    logger.info(
        "Prepared adult dataset rows=%s input_dim=%s classes=%s",
        adult_rows,
        adult_input_dim,
        adult_num_classes,
    )

    dataset_specs = [
        {
            "name": "mnist",
            "config": make_train_config(
                dataset_name="mnist",
                data_dir=str(ROOT / "data" / "raw"),
                model_name="simple_mlp",
                input_dim=784,
                hidden_dim=64,
                num_classes=10,
                learning_rate=0.15,
                momentum=0.0,
                split_seed=11,
                training_seed=123,
            ),
            "train_limit": 2048,
            "eval_limit": 512,
        },
        {
            "name": "cifar10",
            "config": make_train_config(
                dataset_name="cifar10",
                data_dir=str(DATA_DIR),
                model_name="cnn_cifar10",
                input_dim=3072,
                hidden_dim=128,
                num_classes=10,
                learning_rate=0.05,
                momentum=0.9,
                split_seed=11,
                training_seed=123,
            ),
            "train_limit": 1024,
            "eval_limit": 256,
        },
        {
            "name": "adult",
            "config": make_train_config(
                dataset_name="adult",
                data_dir=str(DATA_DIR),
                model_name="tabular_mlp",
                input_dim=adult_input_dim,
                hidden_dim=64,
                num_classes=adult_num_classes,
                learning_rate=0.1,
                momentum=0.0,
                split_seed=11,
                training_seed=123,
            ),
            "train_limit": 4096,
            "eval_limit": 1024,
        },
    ]

    summary: list[dict] = []
    for spec in dataset_specs:
        dataset_name = spec["name"]
        config = spec["config"]
        try:
            logger.info("Loading dataset bundle for %s", dataset_name)
            bundle = load_dataset_bundle(config.dataset, split_seed=config.run.split_seed)
            bundle = subset_bundle(bundle, spec["train_limit"], spec["eval_limit"], seed=777)
            logger.info(
                "Dataset %s subset sizes train=%s eval=%s",
                dataset_name,
                bundle.train_size,
                bundle.eval_size,
            )
            training_case = train_dataset_case(dataset_name, config, bundle, logger)
            summary.append(
                {
                    "dataset": dataset_name,
                    "attack": "__training__",
                    "support_label": "__training__",
                    "status": "ok",
                    "train_size": bundle.train_size,
                    "eval_size": bundle.eval_size,
                    "epsilon_upper_rdp": training_case["record"].epsilon_upper_theory,
                    "epsilon_upper_tighter": training_case["record"].epsilon_upper_pld,
                    "tighter_upper_backend": training_case["record"].pld_accounting_backend,
                    "accuracy": training_case["record"].utility_metrics.get("accuracy"),
                    "loss": training_case["record"].utility_metrics.get("loss"),
                    "training_elapsed_seconds": training_case["elapsed_seconds"],
                    "training_run_id": training_case["record"].training_run_id,
                    "training_result_path": training_case["training_result_path"],
                }
            )

            for attack_name, score_type in [
                ("passive_negative_loss", "negative_loss"),
                ("passive_score_fusion", "score_fusion"),
            ]:
                for support in PASSIVE_SUPPORT_LEVELS:
                    try:
                        started = time.time()
                        row = run_passive_support_level(
                            dataset_name=dataset_name,
                            attack_name=attack_name,
                            score_type=score_type,
                            training_case=training_case,
                            support_label=support["support_label"],
                            query_budget=support["query_budget"],
                            audit_seeds=support["audit_seeds"],
                        )
                        row["elapsed_seconds"] = round(time.time() - started, 3)
                        summary.append(row)
                        logger.info(
                            "Completed %s on %s support=%s eps_lower=%s",
                            attack_name,
                            dataset_name,
                            support["support_label"],
                            row.get("epsilon_lower_conservative"),
                        )
                    except Exception as exc:
                        summary.append(
                            {
                                "dataset": dataset_name,
                                "attack": attack_name,
                                "support_label": support["support_label"],
                                "status": "error",
                                "error": str(exc),
                                "traceback": traceback.format_exc(),
                            }
                        )
                        logger.exception(
                            "Attack %s failed on %s support=%s",
                            attack_name,
                            dataset_name,
                            support["support_label"],
                        )

            for support in CANARY_SUPPORT_LEVELS:
                try:
                    started = time.time()
                    row = run_canary_support_level(
                        dataset_name=dataset_name,
                        attack_name="canary_random",
                        training_case=training_case,
                        support_label=support["support_label"],
                        num_canaries=support["num_canaries"],
                        audit_seeds=support["audit_seeds"],
                    )
                    row["elapsed_seconds"] = round(time.time() - started, 3)
                    summary.append(row)
                    logger.info(
                        "Completed canary_random on %s support=%s status=%s eps_lower=%s",
                        dataset_name,
                        support["support_label"],
                        row.get("status"),
                        row.get("epsilon_lower_conservative"),
                    )
                except Exception as exc:
                    summary.append(
                        {
                            "dataset": dataset_name,
                            "attack": "canary_random",
                            "support_label": support["support_label"],
                            "status": "error",
                            "error": str(exc),
                            "traceback": traceback.format_exc(),
                        }
                    )
                    logger.exception(
                        "Canary attack failed on %s support=%s",
                        dataset_name,
                        support["support_label"],
                    )
        except Exception as exc:
            summary.append(
                {
                    "dataset": dataset_name,
                    "attack": "__dataset__",
                    "support_label": "__dataset__",
                    "status": "error",
                    "error": str(exc),
                    "traceback": traceback.format_exc(),
                }
            )
            logger.exception("Dataset case failed for %s", dataset_name)

    write_summary(summary)
    print(f"Wrote summary JSON to {SUMMARY_JSON}")
    print(f"Wrote summary CSV to {SUMMARY_CSV}")


if __name__ == "__main__":
    main()
""",
    "codex/run_raw_lira_pilot.py": """from __future__ import annotations

import csv
import json
import random
import statistics
import sys
import time
import traceback
from pathlib import Path


ROOT = Path(__file__).resolve().parents[1]
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dp_audit_tightness.config import (
    DatasetConfig,
    ModelConfig,
    OptimizerConfig,
    PrivacyConfig,
    RunConfig,
    TrainExperimentConfig,
    TrainingConfig,
    config_to_dict,
)
from dp_audit_tightness.data.datasets import DatasetBundle, load_dataset_bundle
from dp_audit_tightness.privacy.empirical import estimate_empirical_lower_bound
from dp_audit_tightness.training.dp_sgd import train_dp_sgd
from dp_audit_tightness.utils.logging_utils import configure_logging
from dp_audit_tightness.utils.results import save_json, save_training_result
from dp_audit_tightness.utils.seeds import set_global_seed


CODEX_DIR = ROOT / "codex"
DATA_DIR = CODEX_DIR / "data"
RESULTS_DIR = CODEX_DIR / "results" / "raw_lira_pilot"
SUMMARY_JSON = RESULTS_DIR / "raw_lira_pilot_summary.json"
SUMMARY_CSV = RESULTS_DIR / "raw_lira_pilot_summary.csv"

QUERY_BUDGET = 512
AUDIT_SEEDS = [401, 402, 403, 404, 405]
K_VALUES = [8, 16]
SHADOW_SUBSET_FRACTION = 0.5


def ensure_adult_npz(target_path: Path) -> tuple[int, int, int]:
    import numpy as np
    import pandas as pd
    from sklearn.datasets import fetch_openml

    if target_path.exists():
        cached = np.load(target_path)
        features = cached["features"]
        labels = cached["labels"]
        return int(features.shape[1]), int(labels.max()) + 1, int(features.shape[0])

    target_path.parent.mkdir(parents=True, exist_ok=True)
    adult = fetch_openml("adult", version=2, as_frame=True)
    features = adult.data.copy()
    labels = adult.target.astype(str).copy()

    features = features.replace("?", pd.NA)
    mask = ~features.isna().any(axis=1) & labels.notna()
    features = features.loc[mask]
    labels = labels.loc[mask]

    encoded = pd.get_dummies(features, dummy_na=False, drop_first=False, dtype="float32")
    label_values = sorted(labels.unique().tolist())
    label_map = {label: index for index, label in enumerate(label_values)}
    y = labels.map(label_map).astype("int64").to_numpy()
    x = encoded.to_numpy(dtype="float32")

    np.savez(target_path, features=x, labels=y)
    metadata = {
        "rows": int(x.shape[0]),
        "input_dim": int(x.shape[1]),
        "num_classes": int(len(label_values)),
        "label_map": label_map,
    }
    save_json(target_path.with_suffix(".metadata.json"), metadata)
    return int(x.shape[1]), int(len(label_values)), int(x.shape[0])


def subset_bundle(bundle: DatasetBundle, train_limit: int, eval_limit: int, seed: int) -> DatasetBundle:
    import torch
    from torch.utils.data import Subset

    generator = torch.Generator().manual_seed(seed)
    train_perm = torch.randperm(len(bundle.train_dataset), generator=generator).tolist()
    eval_perm = torch.randperm(len(bundle.eval_dataset), generator=generator).tolist()
    train_indices = train_perm[: min(train_limit, len(train_perm))]
    eval_indices = eval_perm[: min(eval_limit, len(eval_perm))]
    return DatasetBundle(
        train_dataset=Subset(bundle.train_dataset, train_indices),
        eval_dataset=Subset(bundle.eval_dataset, eval_indices),
        input_dim=bundle.input_dim,
        num_classes=bundle.num_classes,
        train_size=len(train_indices),
        eval_size=len(eval_indices),
    )


def make_train_config(
    *,
    dataset_name: str,
    data_dir: str,
    model_name: str,
    input_dim: int,
    hidden_dim: int,
    num_classes: int,
    learning_rate: float,
    momentum: float,
    training_seed: int,
) -> TrainExperimentConfig:
    return TrainExperimentConfig(
        experiment_name=f"codex_raw_lira_{dataset_name}",
        dataset=DatasetConfig(
            name=dataset_name,
            data_dir=data_dir,
            validation_fraction=0.1,
            num_workers=0,
        ),
        model=ModelConfig(
            name=model_name,
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            num_classes=num_classes,
        ),
        training=TrainingConfig(
            batch_size=128,
            eval_batch_size=256,
            epochs=1,
            clipping_norm=1.0,
            noise_multiplier=1.1,
            optimizer=OptimizerConfig(
                name="sgd",
                learning_rate=learning_rate,
                weight_decay=0.0,
                momentum=momentum,
            ),
        ),
        privacy=PrivacyConfig(delta=1e-5, accountant="rdp"),
        run=RunConfig(
            split_seed=11,
            training_seed=training_seed,
            results_root=str(RESULTS_DIR),
            save_checkpoint=True,
            notes="Codex raw LiRA sidecar pilot.",
        ),
    )


def train_target_case(dataset_name: str, config: TrainExperimentConfig, bundle: DatasetBundle, logger) -> dict:
    set_global_seed(config.run.training_seed)
    started = time.time()
    outcome = train_dp_sgd(config=config, logger=logger, dataset_bundle=bundle)

    config_path = RESULTS_DIR / "training" / "configs" / f"{outcome.record.training_run_id}_config.json"
    outcome.record.config_snapshot_path = str(config_path)
    save_json(config_path, config_to_dict(config))
    training_result_path = save_training_result(outcome.record, results_root=config.run.results_root)

    return {
        "dataset": dataset_name,
        "config": config,
        "bundle": bundle,
        "record": outcome.record,
        "training_result_path": str(training_result_path),
        "elapsed_seconds": round(time.time() - started, 3),
    }


def sample_query_indices(train_size: int, eval_size: int) -> tuple[list[int], list[int], dict[int, list[int]], dict[int, list[int]]]:
    all_train_indices: set[int] = set()
    all_eval_indices: set[int] = set()
    per_seed_train: dict[int, list[int]] = {}
    per_seed_eval: dict[int, list[int]] = {}
    for seed in AUDIT_SEEDS:
        rng = random.Random(seed)
        train_indices = rng.sample(range(train_size), QUERY_BUDGET // 2)
        eval_indices = rng.sample(range(eval_size), QUERY_BUDGET // 2)
        per_seed_train[seed] = train_indices
        per_seed_eval[seed] = eval_indices
        all_train_indices.update(train_indices)
        all_eval_indices.update(eval_indices)
    return sorted(all_train_indices), sorted(all_eval_indices), per_seed_train, per_seed_eval


def compute_loss_for_indices(model, dataset, indices, device, batch_size=256):
    import torch
    import torch.nn.functional as F

    if not indices:
        return []

    losses = []
    with torch.no_grad():
        for start in range(0, len(indices), batch_size):
            batch_indices = indices[start : start + batch_size]
            images = []
            labels = []
            for index in batch_indices:
                image, label = dataset[index]
                images.append(image)
                labels.append(label)
            x = torch.stack(images).to(device)
            y = torch.tensor(labels, dtype=torch.long, device=device)
            batch_losses = F.cross_entropy(model(x), y, reduction="none")
            losses.extend(float(value) for value in batch_losses.detach().cpu().tolist())
    return losses


def estimate_conservative(member_scores: list[float], nonmember_scores: list[float], delta: float):
    return estimate_empirical_lower_bound(
        member_scores=member_scores,
        nonmember_scores=nonmember_scores,
        delta=delta,
        align_event_to_score_direction=True,
        require_member_favoring=True,
        report_confidence_supported_lower_bound=True,
    )


def estimate_conservative_with_direction(
    member_scores: list[float],
    nonmember_scores: list[float],
    delta: float,
    score_direction: str,
):
    return estimate_empirical_lower_bound(
        member_scores=member_scores,
        nonmember_scores=nonmember_scores,
        delta=delta,
        score_direction=score_direction,
        align_event_to_score_direction=True,
        require_member_favoring=True,
        report_confidence_supported_lower_bound=True,
    )


def train_shadow_losses(
    *,
    training_case: dict,
    dataset_name: str,
    query_train_indices: list[int],
    query_eval_indices: list[int],
    k_max: int,
    logger,
) -> tuple[list[list[float]], list[list[float]], list[set[int]]]:
    import torch
    from torch.utils.data import Subset
    from dp_audit_tightness.models.io import load_model_from_state_dict

    config: TrainExperimentConfig = training_case["config"]
    bundle: DatasetBundle = training_case["bundle"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    shadow_train_losses: list[list[float]] = []
    shadow_eval_losses: list[list[float]] = []
    shadow_member_sets: list[set[int]] = []

    train_size = len(bundle.train_dataset)
    shadow_train_size = max(2, int(train_size * SHADOW_SUBSET_FRACTION))

    for shadow_index in range(k_max):
        shadow_seed = 5000 + shadow_index
        rng = random.Random(shadow_seed)
        all_indices = list(range(train_size))
        rng.shuffle(all_indices)
        in_indices = sorted(all_indices[:shadow_train_size])
        shadow_member_sets.append(set(in_indices))

        shadow_bundle = DatasetBundle(
            train_dataset=Subset(bundle.train_dataset, in_indices),
            eval_dataset=bundle.eval_dataset,
            input_dim=bundle.input_dim,
            num_classes=bundle.num_classes,
            train_size=len(in_indices),
            eval_size=bundle.eval_size,
        )
        shadow_config = make_train_config(
            dataset_name=config.dataset.name,
            data_dir=config.dataset.data_dir,
            model_name=config.model.name,
            input_dim=config.model.input_dim,
            hidden_dim=config.model.hidden_dim,
            num_classes=config.model.num_classes,
            learning_rate=config.training.optimizer.learning_rate,
            momentum=config.training.optimizer.momentum,
            training_seed=shadow_seed,
        )
        shadow_config.experiment_name = f"{config.experiment_name}_shadow_{shadow_index}"
        shadow_config.run.results_root = str(RESULTS_DIR)
        shadow_config.run.save_checkpoint = False
        shadow_config.run.notes = "Codex raw LiRA shadow model."

        set_global_seed(shadow_seed)
        outcome = train_dp_sgd(
            config=shadow_config,
            logger=logger,
            dataset_bundle=shadow_bundle,
            save_checkpoint=False,
            run_descriptor=f"{dataset_name}_shadow_{shadow_index}",
            return_model_state=True,
        )
        shadow_model = load_model_from_state_dict(config.model, outcome.model_state_dict, device=device)
        shadow_train_losses.append(
            compute_loss_for_indices(shadow_model, bundle.train_dataset, query_train_indices, device=device)
        )
        shadow_eval_losses.append(
            compute_loss_for_indices(shadow_model, bundle.eval_dataset, query_eval_indices, device=device)
        )
        logger.info(
            "Shadow %s/%s complete for %s",
            shadow_index + 1,
            k_max,
            dataset_name,
        )

    return shadow_train_losses, shadow_eval_losses, shadow_member_sets


def raw_lira_scores(
    *,
    query_train_indices: list[int],
    query_eval_indices: list[int],
    per_seed_train: dict[int, list[int]],
    per_seed_eval: dict[int, list[int]],
    target_train_losses: list[float],
    target_eval_losses: list[float],
    shadow_train_losses: list[list[float]],
    shadow_eval_losses: list[list[float]],
    shadow_member_sets: list[set[int]],
    k: int,
) -> tuple[list[float], list[float]]:
    train_pos = {index: pos for pos, index in enumerate(query_train_indices)}
    eval_pos = {index: pos for pos, index in enumerate(query_eval_indices)}

    member_scores: list[float] = []
    nonmember_scores: list[float] = []

    for seed in AUDIT_SEEDS:
        for index in per_seed_train[seed]:
            pos = train_pos[index]
            in_losses = []
            out_losses = []
            for shadow_index in range(k):
                loss_value = float(shadow_train_losses[shadow_index][pos])
                if index in shadow_member_sets[shadow_index]:
                    in_losses.append(loss_value)
                else:
                    out_losses.append(loss_value)
            if in_losses and out_losses:
                score = statistics.fmean(out_losses) - statistics.fmean(in_losses)
            elif out_losses:
                score = statistics.fmean(out_losses)
            else:
                score = -statistics.fmean(in_losses) if in_losses else 0.0
            member_scores.append(float(score))

        for index in per_seed_eval[seed]:
            pos = eval_pos[index]
            shadow_losses = [float(shadow_eval_losses[shadow_index][pos]) for shadow_index in range(k)]
            shadow_mean = statistics.fmean(shadow_losses) if shadow_losses else 0.0
            score = shadow_mean - float(target_eval_losses[pos])
            nonmember_scores.append(float(score))

    return member_scores, nonmember_scores


def negative_loss_scores(
    *,
    query_train_indices: list[int],
    query_eval_indices: list[int],
    per_seed_train: dict[int, list[int]],
    per_seed_eval: dict[int, list[int]],
    target_train_losses: list[float],
    target_eval_losses: list[float],
) -> tuple[list[float], list[float]]:
    train_pos = {index: pos for pos, index in enumerate(query_train_indices)}
    eval_pos = {index: pos for pos, index in enumerate(query_eval_indices)}
    member_scores: list[float] = []
    nonmember_scores: list[float] = []

    for seed in AUDIT_SEEDS:
        member_scores.extend(-float(target_train_losses[train_pos[index]]) for index in per_seed_train[seed])
        nonmember_scores.extend(-float(target_eval_losses[eval_pos[index]]) for index in per_seed_eval[seed])

    return member_scores, nonmember_scores


def build_result_row(
    *,
    dataset_name: str,
    attack_name: str,
    k: int,
    training_case: dict,
    member_scores: list[float],
    nonmember_scores: list[float],
    score_direction: str = "higher",
) -> dict:
    estimate = estimate_conservative_with_direction(
        member_scores=member_scores,
        nonmember_scores=nonmember_scores,
        delta=training_case["config"].privacy.delta,
        score_direction=score_direction,
    )
    record = training_case["record"]
    epsilon_upper_tighter = record.epsilon_upper_pld
    epsilon_lower = estimate.epsilon_lower_empirical
    return {
        "dataset": dataset_name,
        "attack": attack_name,
        "status": "ok",
        "k_shadows": k,
        "score_direction": score_direction,
        "query_budget_per_seed": QUERY_BUDGET,
        "num_audit_seeds": len(AUDIT_SEEDS),
        "audit_seeds": json.dumps(AUDIT_SEEDS),
        "epsilon_upper_rdp": record.epsilon_upper_theory,
        "epsilon_upper_tighter": epsilon_upper_tighter,
        "tighter_upper_backend": record.pld_accounting_backend,
        "epsilon_lower_conservative": epsilon_lower,
        "epsilon_lower_point": estimate.epsilon_lower_empirical_point_estimate,
        "tightness_ratio_tighter": (epsilon_lower / epsilon_upper_tighter) if epsilon_upper_tighter else None,
        "privacy_loss_gap_tighter": (epsilon_upper_tighter - epsilon_lower) if epsilon_upper_tighter is not None else None,
        "selected_tpr": estimate.selected_tpr,
        "selected_fpr": estimate.selected_fpr,
        "warning": estimate.warning_message,
        "num_member_samples": estimate.num_member_samples,
        "num_nonmember_samples": estimate.num_nonmember_samples,
        "member_score_mean": statistics.fmean(member_scores),
        "nonmember_score_mean": statistics.fmean(nonmember_scores),
        "score_gap": statistics.fmean(member_scores) - statistics.fmean(nonmember_scores),
    }


def write_summary(summary: list[dict]) -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with SUMMARY_JSON.open("w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2, sort_keys=True)

    fieldnames: list[str] = []
    for row in summary:
        for key in row.keys():
            if key not in fieldnames:
                fieldnames.append(key)
    with SUMMARY_CSV.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in summary:
            writer.writerow(row)


def main() -> None:
    import torch
    from dp_audit_tightness.models.io import load_model_for_inference

    logger = configure_logging()
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    adult_npz = DATA_DIR / "adult.npz"
    adult_input_dim, adult_num_classes, adult_rows = ensure_adult_npz(adult_npz)
    logger.info(
        "Prepared adult dataset rows=%s input_dim=%s classes=%s",
        adult_rows,
        adult_input_dim,
        adult_num_classes,
    )

    dataset_specs = [
        {
            "name": "mnist",
            "config": make_train_config(
                dataset_name="mnist",
                data_dir=str(ROOT / "data" / "raw"),
                model_name="simple_mlp",
                input_dim=784,
                hidden_dim=64,
                num_classes=10,
                learning_rate=0.15,
                momentum=0.0,
                training_seed=123,
            ),
            "train_limit": 2048,
            "eval_limit": 512,
        },
        {
            "name": "cifar10",
            "config": make_train_config(
                dataset_name="cifar10",
                data_dir=str(DATA_DIR),
                model_name="cnn_cifar10",
                input_dim=3072,
                hidden_dim=128,
                num_classes=10,
                learning_rate=0.05,
                momentum=0.9,
                training_seed=123,
            ),
            "train_limit": 1024,
            "eval_limit": 256,
        },
        {
            "name": "adult",
            "config": make_train_config(
                dataset_name="adult",
                data_dir=str(DATA_DIR),
                model_name="tabular_mlp",
                input_dim=adult_input_dim,
                hidden_dim=64,
                num_classes=adult_num_classes,
                learning_rate=0.1,
                momentum=0.0,
                training_seed=123,
            ),
            "train_limit": 4096,
            "eval_limit": 1024,
        },
    ]

    summary: list[dict] = []
    k_max = max(K_VALUES)

    for spec in dataset_specs:
        dataset_name = spec["name"]
        config = spec["config"]
        try:
            logger.info("Loading dataset bundle for %s", dataset_name)
            bundle = load_dataset_bundle(config.dataset, split_seed=config.run.split_seed)
            bundle = subset_bundle(bundle, spec["train_limit"], spec["eval_limit"], seed=777)
            training_case = train_target_case(dataset_name, config, bundle, logger)

            summary.append(
                {
                    "dataset": dataset_name,
                    "attack": "__training__",
                    "status": "ok",
                    "train_size": bundle.train_size,
                    "eval_size": bundle.eval_size,
                    "epsilon_upper_rdp": training_case["record"].epsilon_upper_theory,
                    "epsilon_upper_tighter": training_case["record"].epsilon_upper_pld,
                    "tighter_upper_backend": training_case["record"].pld_accounting_backend,
                    "accuracy": training_case["record"].utility_metrics.get("accuracy"),
                    "loss": training_case["record"].utility_metrics.get("loss"),
                    "training_elapsed_seconds": training_case["elapsed_seconds"],
                    "training_run_id": training_case["record"].training_run_id,
                    "training_result_path": training_case["training_result_path"],
                }
            )

            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            target_model = load_model_for_inference(config.model, training_case["record"].model_artifact_path, device=device)

            query_train_indices, query_eval_indices, per_seed_train, per_seed_eval = sample_query_indices(
                train_size=len(bundle.train_dataset),
                eval_size=len(bundle.eval_dataset),
            )
            target_train_losses = compute_loss_for_indices(
                target_model,
                bundle.train_dataset,
                query_train_indices,
                device=device,
            )
            target_eval_losses = compute_loss_for_indices(
                target_model,
                bundle.eval_dataset,
                query_eval_indices,
                device=device,
            )

            started = time.time()
            shadow_train_losses, shadow_eval_losses, shadow_member_sets = train_shadow_losses(
                training_case=training_case,
                dataset_name=dataset_name,
                query_train_indices=query_train_indices,
                query_eval_indices=query_eval_indices,
                k_max=k_max,
                logger=logger,
            )
            logger.info(
                "Prepared raw LiRA shadow losses for %s in %.1fs",
                dataset_name,
                time.time() - started,
            )

            for k in K_VALUES:
                try:
                    nl_member_scores, nl_nonmember_scores = negative_loss_scores(
                        query_train_indices=query_train_indices,
                        query_eval_indices=query_eval_indices,
                        per_seed_train=per_seed_train,
                        per_seed_eval=per_seed_eval,
                        target_train_losses=target_train_losses,
                        target_eval_losses=target_eval_losses,
                    )
                    row = build_result_row(
                        dataset_name=dataset_name,
                        attack_name="passive_negative_loss_matched",
                        k=k,
                        training_case=training_case,
                        member_scores=nl_member_scores,
                        nonmember_scores=nl_nonmember_scores,
                        score_direction="higher",
                    )
                    summary.append(row)
                    logger.info(
                        "Completed matched negative-loss baseline on %s K=%s eps_lower=%s",
                        dataset_name,
                        k,
                        row["epsilon_lower_conservative"],
                    )
                except Exception as exc:
                    summary.append(
                        {
                            "dataset": dataset_name,
                            "attack": "passive_negative_loss_matched",
                            "k_shadows": k,
                            "status": "error",
                            "error": str(exc),
                            "traceback": traceback.format_exc(),
                        }
                    )
                    logger.exception("Matched negative-loss failed on %s K=%s", dataset_name, k)

                try:
                    raw_member_scores, raw_nonmember_scores = raw_lira_scores(
                        query_train_indices=query_train_indices,
                        query_eval_indices=query_eval_indices,
                        per_seed_train=per_seed_train,
                        per_seed_eval=per_seed_eval,
                        target_train_losses=target_train_losses,
                        target_eval_losses=target_eval_losses,
                        shadow_train_losses=shadow_train_losses,
                        shadow_eval_losses=shadow_eval_losses,
                        shadow_member_sets=shadow_member_sets,
                        k=k,
                    )
                    row = build_result_row(
                        dataset_name=dataset_name,
                        attack_name="passive_raw_lira",
                        k=k,
                        training_case=training_case,
                        member_scores=raw_member_scores,
                        nonmember_scores=raw_nonmember_scores,
                        score_direction="lower",
                    )
                    summary.append(row)
                    logger.info(
                        "Completed raw LiRA on %s K=%s eps_lower=%s",
                        dataset_name,
                        k,
                        row["epsilon_lower_conservative"],
                    )
                except Exception as exc:
                    summary.append(
                        {
                            "dataset": dataset_name,
                            "attack": "passive_raw_lira",
                            "k_shadows": k,
                            "status": "error",
                            "error": str(exc),
                            "traceback": traceback.format_exc(),
                        }
                    )
                    logger.exception("Raw LiRA failed on %s K=%s", dataset_name, k)
        except Exception as exc:
            summary.append(
                {
                    "dataset": dataset_name,
                    "attack": "__dataset__",
                    "status": "error",
                    "error": str(exc),
                    "traceback": traceback.format_exc(),
                }
            )
            logger.exception("Dataset case failed for %s", dataset_name)

    write_summary(summary)
    print(f"Wrote summary JSON to {SUMMARY_JSON}")
    print(f"Wrote summary CSV to {SUMMARY_CSV}")


if __name__ == "__main__":
    main()
""",
    "codex/run_framework_validation.py": """from __future__ import annotations

import csv
import json
import sys
import time
import traceback
from collections import Counter
from pathlib import Path
from typing import Any


ROOT = Path(__file__).resolve().parents[1]
SRC = ROOT / "src"
CODEX_DIR = ROOT / "codex"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if str(CODEX_DIR) not in sys.path:
    sys.path.insert(0, str(CODEX_DIR))

import run_raw_lira_pilot as raw_lira
import run_support_scaled_pilot as support_scaled
from dp_audit_tightness.data.datasets import load_dataset_bundle
from dp_audit_tightness.models.io import load_model_for_inference
from dp_audit_tightness.utils.logging_utils import configure_logging


RESULTS_DIR = CODEX_DIR / "results" / "framework_validation"
SUMMARY_JSON = RESULTS_DIR / "framework_validation_summary.json"
SUMMARY_CSV = RESULTS_DIR / "framework_validation_rows.csv"
CHECKS_JSON = RESULTS_DIR / "framework_validation_checks.json"
REPORT_MD = RESULTS_DIR / "framework_validation_report.md"

PASSIVE_AUDIT_SEEDS = [401, 402, 403, 404, 405]
CANARY_AUDIT_SEEDS = [101, 102, 103, 104, 105]
QUERY_BUDGET = 512
RAW_LIRA_K = 16
NUM_CANARIES = 32


def patch_sidecar_modules() -> None:
    support_scaled.RESULTS_DIR = RESULTS_DIR
    support_scaled.SUMMARY_JSON = RESULTS_DIR / "unused_support_summary.json"
    support_scaled.SUMMARY_CSV = RESULTS_DIR / "unused_support_summary.csv"
    support_scaled.DATA_DIR = CODEX_DIR / "data"

    raw_lira.RESULTS_DIR = RESULTS_DIR
    raw_lira.SUMMARY_JSON = RESULTS_DIR / "unused_raw_lira_summary.json"
    raw_lira.SUMMARY_CSV = RESULTS_DIR / "unused_raw_lira_summary.csv"
    raw_lira.DATA_DIR = CODEX_DIR / "data"
    raw_lira.QUERY_BUDGET = QUERY_BUDGET
    raw_lira.AUDIT_SEEDS = list(PASSIVE_AUDIT_SEEDS)
    raw_lira.K_VALUES = [RAW_LIRA_K]


def dataset_specs() -> list[dict[str, Any]]:
    adult_npz = CODEX_DIR / "data" / "adult.npz"
    adult_input_dim, adult_num_classes, _ = support_scaled.ensure_adult_npz(adult_npz)
    return [
        {
            "name": "mnist",
            "config": support_scaled.make_train_config(
                dataset_name="mnist",
                data_dir=str(ROOT / "data" / "raw"),
                model_name="simple_mlp",
                input_dim=784,
                hidden_dim=64,
                num_classes=10,
                learning_rate=0.15,
                momentum=0.0,
                split_seed=11,
                training_seed=123,
            ),
            "train_limit": 2048,
            "eval_limit": 512,
        },
        {
            "name": "cifar10",
            "config": support_scaled.make_train_config(
                dataset_name="cifar10",
                data_dir=str(CODEX_DIR / "data"),
                model_name="cnn_cifar10",
                input_dim=3072,
                hidden_dim=128,
                num_classes=10,
                learning_rate=0.05,
                momentum=0.9,
                split_seed=11,
                training_seed=123,
            ),
            "train_limit": 1024,
            "eval_limit": 256,
        },
        {
            "name": "adult",
            "config": support_scaled.make_train_config(
                dataset_name="adult",
                data_dir=str(CODEX_DIR / "data"),
                model_name="tabular_mlp",
                input_dim=adult_input_dim,
                hidden_dim=64,
                num_classes=adult_num_classes,
                learning_rate=0.1,
                momentum=0.0,
                split_seed=11,
                training_seed=123,
            ),
            "train_limit": 4096,
            "eval_limit": 1024,
        },
    ]


def safe_float(value: Any) -> float | None:
    if value is None:
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def add_check(
    checks: list[dict[str, Any]],
    *,
    check_id: str,
    status: str,
    description: str,
    scope: str,
    details: str,
) -> None:
    checks.append(
        {
            "check_id": check_id,
            "status": status,
            "description": description,
            "scope": scope,
            "details": details,
        }
    )


def normalize_attack_row(row: dict[str, Any]) -> dict[str, Any]:
    attack_name = str(row.get("attack"))
    status = str(row.get("status", ""))

    direction_map = {
        "passive_negative_loss": "higher_is_member",
        "passive_negative_loss_matched": "higher_is_member",
        "passive_raw_lira": "lower_is_member",
        "canary_random": "higher_is_member",
    }
    score_name_map = {
        "passive_negative_loss": "negative_loss",
        "passive_negative_loss_matched": "negative_loss",
        "passive_raw_lira": "raw_lira_score",
        "canary_random": "target_label_logit_margin",
    }
    attack_family_map = {
        "passive_negative_loss": "passive_threshold",
        "passive_negative_loss_matched": "passive_threshold",
        "passive_raw_lira": "passive_lira",
        "canary_random": "active_canary",
    }

    score_direction = direction_map.get(attack_name)
    tags: list[str] = []
    validation_status = "executed"
    expected_limitation = False

    if status == "not_supported":
        validation_status = "expected_not_supported"
        expected_limitation = True
        tags.append("expected_capability_limit")
    elif status != "ok":
        validation_status = "failed_execution"
        tags.append("execution_failure")

    warning = str(row.get("warning") or "").lower()
    if "sparse tail" in warning:
        tags.append("sparse_tail")

    num_member = int(row.get("num_member_samples") or 0)
    num_nonmember = int(row.get("num_nonmember_samples") or 0)
    min_support = min(num_member, num_nonmember)
    if status == "ok":
        if min_support < 128:
            tags.append("low_support")
        elif min_support < 1000:
            tags.append("medium_support")
        else:
            tags.append("high_support")

    epsilon_upper_tighter = safe_float(row.get("epsilon_upper_tighter"))
    epsilon_lower_conservative = safe_float(row.get("epsilon_lower_conservative"))
    epsilon_lower_point = safe_float(row.get("epsilon_lower_point"))
    if (
        status == "ok"
        and epsilon_upper_tighter is not None
        and epsilon_lower_conservative is not None
        and epsilon_lower_conservative > epsilon_upper_tighter + 0.05
    ):
        tags.extend(["pathological_distribution", "exceeds_theoretical_upper"])

    selected_tpr = safe_float(row.get("selected_tpr"))
    selected_fpr = safe_float(row.get("selected_fpr"))
    if selected_tpr is not None and selected_tpr <= 0.02:
        tags.append("extreme_tail_threshold")
    if selected_fpr is not None and selected_fpr == 0.0:
        tags.append("zero_fpr_threshold")

    if attack_name == "passive_raw_lira":
        tags.append("score_direction_sensitive")
    if attack_name == "canary_random":
        tags.append("active_stress_test")

    if status == "ok" and epsilon_lower_conservative == 0.0 and (epsilon_lower_point or 0.0) > 0.0:
        tags.append("finite_sample_limited_signal")

    if expected_limitation:
        result_trust = "not_applicable"
    elif status != "ok":
        result_trust = "invalidated"
    elif "pathological_distribution" in tags:
        result_trust = "invalidated"
    elif "sparse_tail" in tags and ("zero_fpr_threshold" in tags or "extreme_tail_threshold" in tags):
        result_trust = "exploratory"
    elif epsilon_lower_conservative is not None and epsilon_lower_conservative > 0.0:
        result_trust = "provisional"
    else:
        result_trust = "exploratory"

    return {
        "dataset": row.get("dataset"),
        "attack_name": attack_name,
        "attack_family": attack_family_map.get(attack_name, "other"),
        "status": status,
        "validation_status": validation_status,
        "expected_limitation": expected_limitation,
        "score_name": row.get("score_name") or score_name_map.get(attack_name),
        "score_direction": score_direction,
        "upper_bound_backend": row.get("tighter_upper_backend"),
        "epsilon_upper_rdp": safe_float(row.get("epsilon_upper_rdp")),
        "epsilon_upper_tighter": epsilon_upper_tighter,
        "epsilon_lower_conservative": epsilon_lower_conservative,
        "epsilon_lower_point": epsilon_lower_point,
        "selected_tpr": selected_tpr,
        "selected_fpr": selected_fpr,
        "num_member_samples": num_member,
        "num_nonmember_samples": num_nonmember,
        "query_budget_per_seed": row.get("query_budget_per_seed"),
        "num_audit_seeds": row.get("num_audit_seeds"),
        "audit_seeds": row.get("audit_seeds"),
        "k_shadows": row.get("k_shadows"),
        "num_canaries_per_seed": row.get("num_canaries_per_seed"),
        "mean_inserted_example_count": row.get("mean_inserted_example_count"),
        "warning": row.get("warning"),
        "diagnostic_tags": sorted(set(tags)),
        "result_trust": result_trust,
    }


def support_profile(row: dict[str, Any]) -> str:
    pieces: list[str] = []
    if row.get("query_budget_per_seed"):
        pieces.append(f"budget={row['query_budget_per_seed']}")
    if row.get("num_audit_seeds"):
        pieces.append(f"seeds={row['num_audit_seeds']}")
    if row.get("k_shadows"):
        pieces.append(f"K={row['k_shadows']}")
    if row.get("num_canaries_per_seed"):
        pieces.append(f"canaries={row['num_canaries_per_seed']}")
    return ", ".join(pieces) or "unspecified"


def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    fieldnames: list[str] = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)

    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            out = {
                key: "|".join(value) if isinstance(value, list) else value
                for key, value in row.items()
            }
            writer.writerow(out)


def build_checks(training_rows: list[dict[str, Any]], audit_rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    checks: list[dict[str, Any]] = []

    expected_training = {"mnist", "cifar10", "adult"}
    observed_training_ok = {row["dataset"] for row in training_rows if row.get("status") == "ok"}
    add_check(
        checks,
        check_id="training_matrix_coverage",
        status="pass" if observed_training_ok == expected_training else "fail",
        description="All canonical datasets train successfully.",
        scope="global",
        details=f"observed_ok={sorted(observed_training_ok)} expected={sorted(expected_training)}",
    )

    for row in training_rows:
        dataset = row["dataset"]
        if row.get("status") != "ok":
            add_check(
                checks,
                check_id=f"training_status::{dataset}",
                status="fail",
                description="Training completed successfully.",
                scope=dataset,
                details=f"error={row.get('error')}",
            )
            continue

        backend = row.get("upper_bound_backend")
        upper_rdp = safe_float(row.get("epsilon_upper_rdp"))
        upper_tighter = safe_float(row.get("epsilon_upper_tighter"))
        add_check(
            checks,
            check_id=f"upper_backend::{dataset}",
            status="pass" if backend == "google_dp_accounting" else "fail",
            description="Exact PLD backend is active.",
            scope=dataset,
            details=f"backend={backend}",
        )
        add_check(
            checks,
            check_id=f"upper_bound_order::{dataset}",
            status=(
                "pass"
                if upper_tighter is not None and upper_rdp is not None and upper_tighter > 0.0 and upper_tighter <= upper_rdp
                else "fail"
            ),
            description="Tighter upper bound is positive and no larger than the RDP upper bound.",
            scope=dataset,
            details=f"epsilon_upper_tighter={upper_tighter}, epsilon_upper_rdp={upper_rdp}",
        )

    rows_by_key = {(row["dataset"], row["attack_name"]): row for row in audit_rows}
    expected_rows = {
        ("mnist", "passive_negative_loss"),
        ("cifar10", "passive_negative_loss"),
        ("adult", "passive_negative_loss"),
        ("mnist", "passive_negative_loss_matched"),
        ("cifar10", "passive_negative_loss_matched"),
        ("adult", "passive_negative_loss_matched"),
        ("mnist", "passive_raw_lira"),
        ("cifar10", "passive_raw_lira"),
        ("adult", "passive_raw_lira"),
        ("mnist", "canary_random"),
        ("cifar10", "canary_random"),
        ("adult", "canary_random"),
    }
    observed_rows = set(rows_by_key)
    add_check(
        checks,
        check_id="audit_matrix_coverage",
        status="pass" if observed_rows == expected_rows else "fail",
        description="All canonical audit rows are present.",
        scope="global",
        details=f"observed={len(observed_rows)} expected={len(expected_rows)}",
    )

    for (dataset, attack), row in sorted(rows_by_key.items()):
        scope = f"{dataset}::{attack}"
        if row["validation_status"] == "expected_not_supported":
            add_check(
                checks,
                check_id=f"expected_limitation::{scope}",
                status="pass",
                description="Unsupported capability is explicit rather than silent.",
                scope=scope,
                details="status=not_supported",
            )
            continue

        if row["validation_status"] != "executed":
            add_check(
                checks,
                check_id=f"execution::{scope}",
                status="fail",
                description="Audit row executed successfully.",
                scope=scope,
                details=f"status={row['status']}",
            )
            continue

        add_check(
            checks,
            check_id=f"score_direction::{scope}",
            status="pass" if row.get("score_direction") else "fail",
            description="Score direction is explicit.",
            scope=scope,
            details=f"score_direction={row.get('score_direction')}",
        )
        add_check(
            checks,
            check_id=f"sample_counts::{scope}",
            status=(
                "pass"
                if row["num_member_samples"] > 0 and row["num_nonmember_samples"] > 0
                else "fail"
            ),
            description="Sample counts are positive.",
            scope=scope,
            details=(
                f"num_member_samples={row['num_member_samples']}, "
                f"num_nonmember_samples={row['num_nonmember_samples']}"
            ),
        )

        lower_cons = row.get("epsilon_lower_conservative")
        lower_point = row.get("epsilon_lower_point")
        add_check(
            checks,
            check_id=f"lower_bound_order::{scope}",
            status=(
                "pass"
                if lower_cons is not None
                and lower_point is not None
                and lower_cons <= lower_point + 1e-9
                else "fail"
            ),
            description="Conservative lower bound does not exceed the point estimate.",
            scope=scope,
            details=f"epsilon_lower_conservative={lower_cons}, epsilon_lower_point={lower_point}",
        )

        tpr = row.get("selected_tpr")
        fpr = row.get("selected_fpr")
        rates_ok = (
            (tpr is None or 0.0 <= tpr <= 1.0)
            and (fpr is None or 0.0 <= fpr <= 1.0)
        )
        add_check(
            checks,
            check_id=f"rates_in_range::{scope}",
            status="pass" if rates_ok else "fail",
            description="Selected rates are valid probabilities.",
            scope=scope,
            details=f"selected_tpr={tpr}, selected_fpr={fpr}",
        )

        upper = row.get("epsilon_upper_tighter")
        overshoot = (
            lower_cons is not None
            and upper is not None
            and lower_cons > upper + 0.05
        )
        if overshoot:
            add_check(
                checks,
                check_id=f"pathology_guard::{scope}",
                status="pass" if row["result_trust"] == "invalidated" else "fail",
                description="Empirical overshoot is caught and invalidated rather than accepted.",
                scope=scope,
                details=f"epsilon_lower_conservative={lower_cons}, epsilon_upper_tighter={upper}, trust={row['result_trust']}",
            )
        else:
            add_check(
                checks,
                check_id=f"upper_vs_lower::{scope}",
                status="pass",
                description="Empirical lower bound does not materially exceed the theoretical upper bound.",
                scope=scope,
                details=f"epsilon_lower_conservative={lower_cons}, epsilon_upper_tighter={upper}",
            )

    for dataset in ["mnist", "cifar10", "adult"]:
        baseline = rows_by_key.get((dataset, "passive_negative_loss"))
        matched = rows_by_key.get((dataset, "passive_negative_loss_matched"))
        if not baseline or not matched:
            add_check(
                checks,
                check_id=f"baseline_consistency::{dataset}",
                status="fail",
                description="Matched negative-loss agrees with the canonical passive baseline.",
                scope=dataset,
                details="missing one or both rows",
            )
            continue

        if baseline["validation_status"] != "executed" or matched["validation_status"] != "executed":
            add_check(
                checks,
                check_id=f"baseline_consistency::{dataset}",
                status="fail",
                description="Matched negative-loss agrees with the canonical passive baseline.",
                scope=dataset,
                details=(
                    f"baseline_status={baseline['validation_status']}, "
                    f"matched_status={matched['validation_status']}"
                ),
            )
            continue

        diff = abs(
            float(baseline["epsilon_lower_conservative"] or 0.0)
            - float(matched["epsilon_lower_conservative"] or 0.0)
        )
        add_check(
            checks,
            check_id=f"baseline_consistency::{dataset}",
            status="pass" if diff <= 1e-9 else "warn",
            description="Matched negative-loss agrees with the canonical passive baseline.",
            scope=dataset,
            details=f"absolute_difference={diff}",
        )

    for dataset in ["mnist", "cifar10"]:
        baseline = rows_by_key.get((dataset, "passive_negative_loss"))
        raw_row = rows_by_key.get((dataset, "passive_raw_lira"))
        if baseline and raw_row and baseline["validation_status"] == "executed" and raw_row["validation_status"] == "executed":
            improved = (raw_row["epsilon_lower_conservative"] or 0.0) >= (baseline["epsilon_lower_conservative"] or 0.0)
            add_check(
                checks,
                check_id=f"raw_lira_not_weaker::{dataset}",
                status="pass" if improved else "warn",
                description="Raw LiRA is at least as strong as the passive baseline on the canonical image datasets.",
                scope=dataset,
                details=(
                    f"baseline={baseline['epsilon_lower_conservative']}, "
                    f"raw_lira={raw_row['epsilon_lower_conservative']}"
                ),
            )

    adult_raw = rows_by_key.get(("adult", "passive_raw_lira"))
    if adult_raw:
        add_check(
            checks,
            check_id="adult_raw_lira_flagged",
            status=(
                "pass"
                if adult_raw["result_trust"] == "invalidated"
                and "pathological_distribution" in adult_raw["diagnostic_tags"]
                else "fail"
            ),
            description="Adult Raw LiRA pathology is surfaced explicitly.",
            scope="adult::passive_raw_lira",
            details=(
                f"trust={adult_raw['result_trust']}, "
                f"tags={'|'.join(adult_raw['diagnostic_tags'])}"
            ),
        )

    return checks


def build_report(
    training_rows: list[dict[str, Any]],
    audit_rows: list[dict[str, Any]],
    checks: list[dict[str, Any]],
) -> str:
    check_counts = Counter(check["status"] for check in checks)
    trust_counts = Counter(row["result_trust"] for row in audit_rows)
    executed_rows = [row for row in audit_rows if row["validation_status"] == "executed"]
    best_rows = sorted(
        [row for row in executed_rows if row["result_trust"] in {"provisional", "exploratory"}],
        key=lambda row: -float(row.get("epsilon_lower_conservative") or 0.0),
    )

    if check_counts.get("fail", 0) > 0:
        framework_outcome = "fail"
        scientific_status = "invalid for interpretation"
    elif check_counts.get("warn", 0) > 0:
        framework_outcome = "pass with warnings"
        scientific_status = "still provisional overall"
    else:
        framework_outcome = "pass with findings"
        scientific_status = "still provisional overall"

    lines = [
        "# Framework Validation Report",
        "",
        "## Outcome",
        "",
        f"- framework execution validation: `{framework_outcome}`",
        f"- exact-PLD accounting validation: `{'pass' if check_counts.get('fail', 0) == 0 else 'conditional'}`",
        f"- attack semantics validation: `{'pass' if check_counts.get('fail', 0) == 0 else 'conditional'}`",
        f"- scientific trust level: `{scientific_status}`",
        "",
        "## Counts",
        "",
        f"- training rows: `{len(training_rows)}`",
        f"- audit rows: `{len(audit_rows)}`",
        f"- checks passed: `{check_counts.get('pass', 0)}`",
        f"- checks warned: `{check_counts.get('warn', 0)}`",
        f"- checks failed: `{check_counts.get('fail', 0)}`",
        f"- provisional rows: `{trust_counts.get('provisional', 0)}`",
        f"- exploratory rows: `{trust_counts.get('exploratory', 0)}`",
        f"- invalidated rows: `{trust_counts.get('invalidated', 0)}`",
        f"- not applicable rows: `{trust_counts.get('not_applicable', 0)}`",
        "",
        "## What Passed",
        "",
        "- the canonical matrix executed across all three datasets",
        "- exact Google PLD was active for the training runs",
        "- score direction is now explicit for every supported attack row",
        "- the matched negative-loss baseline agrees with the canonical passive baseline",
        "- the framework catches pathological overshoot instead of accepting it silently",
        "",
        "## Main Findings",
        "",
    ]

    for row in best_rows[:5]:
        lines.append(
            "- "
            f"`{row['dataset']} + {row['attack_name']}` "
            f"({support_profile(row)}): "
            f"`eps_lower_cons={row.get('epsilon_lower_conservative'):.6f}`, "
            f"`eps_upper={row.get('epsilon_upper_tighter'):.6f}`, "
            f"`trust={row['result_trust']}`"
        )

    lines.extend(
        [
            "",
            "## Remaining Problems",
            "",
            "- `adult + passive_raw_lira` still overshoots the theoretical upper bound and remains invalidated",
            "- most non-pathological rows are still provisional or exploratory rather than fully trusted",
            "- canary validation is still only implemented for image datasets in this sidecar setup",
            "",
            "## Interpretation",
            "",
            "- the framework itself now validates as an end-to-end system",
            "- the research story does not validate as final yet, because some rows remain tail-sensitive and one dataset-attack pair is still pathological",
            "- the next phase should scale one canonical trustworthy line rather than expand breadth further",
            "",
        ]
    )
    return "\n".join(lines)


def run_validation() -> dict[str, Any]:
    import torch

    logger = configure_logging()
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    (CODEX_DIR / "data").mkdir(parents=True, exist_ok=True)
    patch_sidecar_modules()

    training_rows: list[dict[str, Any]] = []
    raw_rows: list[dict[str, Any]] = []

    for spec in dataset_specs():
        dataset_name = spec["name"]
        config = spec["config"]
        try:
            logger.info("Validation loading dataset bundle for %s", dataset_name)
            bundle = load_dataset_bundle(config.dataset, split_seed=config.run.split_seed)
            bundle = support_scaled.subset_bundle(
                bundle,
                spec["train_limit"],
                spec["eval_limit"],
                seed=777,
            )

            training_case = support_scaled.train_dataset_case(dataset_name, config, bundle, logger)
            record = training_case["record"]
            training_rows.append(
                {
                    "dataset": dataset_name,
                    "status": "ok",
                    "train_size": bundle.train_size,
                    "eval_size": bundle.eval_size,
                    "epsilon_upper_rdp": record.epsilon_upper_theory,
                    "epsilon_upper_tighter": record.epsilon_upper_pld,
                    "upper_bound_backend": record.pld_accounting_backend,
                    "accuracy": record.utility_metrics.get("accuracy"),
                    "loss": record.utility_metrics.get("loss"),
                    "training_elapsed_seconds": training_case["elapsed_seconds"],
                    "training_result_path": training_case["training_result_path"],
                    "training_run_id": record.training_run_id,
                }
            )

            started = time.time()
            passive_row = support_scaled.run_passive_support_level(
                dataset_name=dataset_name,
                attack_name="passive_negative_loss",
                score_type="negative_loss",
                training_case=training_case,
                support_label="validation_canonical",
                query_budget=QUERY_BUDGET,
                audit_seeds=PASSIVE_AUDIT_SEEDS,
            )
            passive_row["elapsed_seconds"] = round(time.time() - started, 3)
            raw_rows.append(passive_row)

            started = time.time()
            canary_row = support_scaled.run_canary_support_level(
                dataset_name=dataset_name,
                attack_name="canary_random",
                training_case=training_case,
                support_label="validation_canonical",
                num_canaries=NUM_CANARIES,
                audit_seeds=CANARY_AUDIT_SEEDS,
            )
            canary_row["elapsed_seconds"] = round(time.time() - started, 3)
            raw_rows.append(canary_row)

            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            target_model = load_model_for_inference(
                config.model,
                training_case["record"].model_artifact_path,
                device=device,
            )

            query_train_indices, query_eval_indices, per_seed_train, per_seed_eval = raw_lira.sample_query_indices(
                train_size=len(bundle.train_dataset),
                eval_size=len(bundle.eval_dataset),
            )
            target_train_losses = raw_lira.compute_loss_for_indices(
                target_model,
                bundle.train_dataset,
                query_train_indices,
                device=device,
            )
            target_eval_losses = raw_lira.compute_loss_for_indices(
                target_model,
                bundle.eval_dataset,
                query_eval_indices,
                device=device,
            )
            shadow_started = time.time()
            shadow_train_losses, shadow_eval_losses, shadow_member_sets = raw_lira.train_shadow_losses(
                training_case=training_case,
                dataset_name=dataset_name,
                query_train_indices=query_train_indices,
                query_eval_indices=query_eval_indices,
                k_max=RAW_LIRA_K,
                logger=logger,
            )
            logger.info(
                "Validation shadows ready for %s in %.1fs",
                dataset_name,
                time.time() - shadow_started,
            )

            nl_member_scores, nl_nonmember_scores = raw_lira.negative_loss_scores(
                query_train_indices=query_train_indices,
                query_eval_indices=query_eval_indices,
                per_seed_train=per_seed_train,
                per_seed_eval=per_seed_eval,
                target_train_losses=target_train_losses,
                target_eval_losses=target_eval_losses,
            )
            raw_rows.append(
                raw_lira.build_result_row(
                    dataset_name=dataset_name,
                    attack_name="passive_negative_loss_matched",
                    k=RAW_LIRA_K,
                    training_case=training_case,
                    member_scores=nl_member_scores,
                    nonmember_scores=nl_nonmember_scores,
                    score_direction="higher",
                )
            )

            raw_member_scores, raw_nonmember_scores = raw_lira.raw_lira_scores(
                query_train_indices=query_train_indices,
                query_eval_indices=query_eval_indices,
                per_seed_train=per_seed_train,
                per_seed_eval=per_seed_eval,
                target_train_losses=target_train_losses,
                target_eval_losses=target_eval_losses,
                shadow_train_losses=shadow_train_losses,
                shadow_eval_losses=shadow_eval_losses,
                shadow_member_sets=shadow_member_sets,
                k=RAW_LIRA_K,
            )
            raw_rows.append(
                raw_lira.build_result_row(
                    dataset_name=dataset_name,
                    attack_name="passive_raw_lira",
                    k=RAW_LIRA_K,
                    training_case=training_case,
                    member_scores=raw_member_scores,
                    nonmember_scores=raw_nonmember_scores,
                    score_direction="lower",
                )
            )
        except Exception as exc:
            logger.exception("Framework validation failed on dataset %s", dataset_name)
            training_rows.append(
                {
                    "dataset": dataset_name,
                    "status": "error",
                    "error": str(exc),
                    "traceback": traceback.format_exc(),
                }
            )

    normalized_rows = [normalize_attack_row(row) for row in raw_rows]
    checks = build_checks(training_rows, normalized_rows)

    payload = {
        "scope": "canonical full-framework validation pass",
        "training_rows": training_rows,
        "audit_rows": normalized_rows,
        "checks": checks,
    }
    SUMMARY_JSON.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    write_csv(SUMMARY_CSV, normalized_rows)
    CHECKS_JSON.write_text(json.dumps(checks, indent=2), encoding="utf-8")
    REPORT_MD.write_text(build_report(training_rows, normalized_rows, checks), encoding="utf-8")
    return payload


def main() -> None:
    started = time.time()
    payload = run_validation()
    elapsed = round(time.time() - started, 3)
    print(f"Wrote validation summary to {SUMMARY_JSON}")
    print(f"Wrote validation rows CSV to {SUMMARY_CSV}")
    print(f"Wrote validation checks to {CHECKS_JSON}")
    print(f"Wrote validation report to {REPORT_MD}")
    print(f"Validation rows: {len(payload['audit_rows'])}, elapsed_seconds={elapsed}")


if __name__ == "__main__":
    main()
""",
    "codex/run_framework_validation_gpu_scale.py": """from __future__ import annotations

import json
import shutil
import sys
import time
import traceback
from pathlib import Path
from typing import Any


ROOT = Path(__file__).resolve().parents[1]
SRC = ROOT / "src"
CODEX_DIR = ROOT / "codex"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if str(CODEX_DIR) not in sys.path:
    sys.path.insert(0, str(CODEX_DIR))

import torch

import run_framework_validation as framework_validation
import run_raw_lira_pilot as raw_lira
import run_support_scaled_pilot as support_scaled
from dp_audit_tightness.data.datasets import load_dataset_bundle
from dp_audit_tightness.models.io import load_model_for_inference
from dp_audit_tightness.utils.logging_utils import configure_logging


RESULTS_DIR = CODEX_DIR / "results" / "framework_validation_gpu_scale"
SUMMARY_JSON = RESULTS_DIR / "framework_validation_gpu_scale_summary.json"
SUMMARY_CSV = RESULTS_DIR / "framework_validation_gpu_scale_rows.csv"
CHECKS_JSON = RESULTS_DIR / "framework_validation_gpu_scale_checks.json"
REPORT_MD = RESULTS_DIR / "framework_validation_gpu_scale_report.md"
ARTIFACTS_ZIP = RESULTS_DIR / "framework_validation_gpu_scale_artifacts.zip"

PASSIVE_AUDIT_SEEDS = list(range(401, 411))
CANARY_AUDIT_SEEDS = list(range(101, 111))
QUERY_BUDGET = 2048
RAW_LIRA_K = 32
NUM_CANARIES = 128


def require_cuda() -> None:
    if not torch.cuda.is_available():
        raise SystemExit(
            "CUDA is required for codex/run_framework_validation_gpu_scale.py. "
            "This machine does not currently expose a GPU."
        )


def patch_sidecar_modules() -> None:
    support_scaled.RESULTS_DIR = RESULTS_DIR
    support_scaled.SUMMARY_JSON = RESULTS_DIR / "unused_support_summary.json"
    support_scaled.SUMMARY_CSV = RESULTS_DIR / "unused_support_summary.csv"
    support_scaled.DATA_DIR = CODEX_DIR / "data"

    raw_lira.RESULTS_DIR = RESULTS_DIR
    raw_lira.SUMMARY_JSON = RESULTS_DIR / "unused_raw_lira_summary.json"
    raw_lira.SUMMARY_CSV = RESULTS_DIR / "unused_raw_lira_summary.csv"
    raw_lira.DATA_DIR = CODEX_DIR / "data"
    raw_lira.QUERY_BUDGET = QUERY_BUDGET
    raw_lira.AUDIT_SEEDS = list(PASSIVE_AUDIT_SEEDS)
    raw_lira.K_VALUES = [RAW_LIRA_K]
    raw_lira.SHADOW_SUBSET_FRACTION = 0.75


def make_scaled_config(
    *,
    dataset_name: str,
    data_dir: str,
    model_name: str,
    input_dim: int,
    hidden_dim: int,
    num_classes: int,
    learning_rate: float,
    momentum: float,
    split_seed: int,
    training_seed: int,
    batch_size: int,
    eval_batch_size: int,
    epochs: int,
) -> Any:
    config = support_scaled.make_train_config(
        dataset_name=dataset_name,
        data_dir=data_dir,
        model_name=model_name,
        input_dim=input_dim,
        hidden_dim=hidden_dim,
        num_classes=num_classes,
        learning_rate=learning_rate,
        momentum=momentum,
        split_seed=split_seed,
        training_seed=training_seed,
    )
    config.training.batch_size = batch_size
    config.training.eval_batch_size = eval_batch_size
    config.training.epochs = epochs
    config.run.results_root = str(RESULTS_DIR)
    config.run.save_checkpoint = True
    config.run.notes = "Codex GPU-scale framework validation run."
    return config


def dataset_specs() -> list[dict[str, Any]]:
    adult_npz = CODEX_DIR / "data" / "adult.npz"
    adult_input_dim, adult_num_classes, _ = support_scaled.ensure_adult_npz(adult_npz)
    return [
        {
            "name": "mnist",
            "config": make_scaled_config(
                dataset_name="mnist",
                data_dir=str(ROOT / "data" / "raw"),
                model_name="simple_mlp",
                input_dim=784,
                hidden_dim=256,
                num_classes=10,
                learning_rate=0.15,
                momentum=0.0,
                split_seed=11,
                training_seed=123,
                batch_size=256,
                eval_batch_size=512,
                epochs=5,
            ),
            "train_limit": 60000,
            "eval_limit": 10000,
        },
        {
            "name": "cifar10",
            "config": make_scaled_config(
                dataset_name="cifar10",
                data_dir=str(CODEX_DIR / "data"),
                model_name="cnn_cifar10",
                input_dim=3072,
                hidden_dim=256,
                num_classes=10,
                learning_rate=0.05,
                momentum=0.9,
                split_seed=11,
                training_seed=123,
                batch_size=256,
                eval_batch_size=512,
                epochs=8,
            ),
            "train_limit": 50000,
            "eval_limit": 10000,
        },
        {
            "name": "adult",
            "config": make_scaled_config(
                dataset_name="adult",
                data_dir=str(CODEX_DIR / "data"),
                model_name="tabular_mlp",
                input_dim=adult_input_dim,
                hidden_dim=128,
                num_classes=adult_num_classes,
                learning_rate=0.05,
                momentum=0.0,
                split_seed=11,
                training_seed=123,
                batch_size=512,
                eval_batch_size=1024,
                epochs=5,
            ),
            "train_limit": 30000,
            "eval_limit": 10000,
        },
    ]


def run_gpu_scale_validation() -> dict[str, Any]:
    logger = configure_logging()
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    patch_sidecar_modules()

    training_rows: list[dict[str, Any]] = []
    raw_rows: list[dict[str, Any]] = []

    for spec in dataset_specs():
        dataset_name = spec["name"]
        config = spec["config"]
        try:
            logger.info("GPU-scale validation loading dataset bundle for %s", dataset_name)
            bundle = load_dataset_bundle(config.dataset, split_seed=config.run.split_seed)
            bundle = support_scaled.subset_bundle(
                bundle,
                spec["train_limit"],
                spec["eval_limit"],
                seed=777,
            )

            training_case = support_scaled.train_dataset_case(dataset_name, config, bundle, logger)
            record = training_case["record"]
            training_rows.append(
                {
                    "dataset": dataset_name,
                    "status": "ok",
                    "train_size": bundle.train_size,
                    "eval_size": bundle.eval_size,
                    "epsilon_upper_rdp": record.epsilon_upper_theory,
                    "epsilon_upper_tighter": record.epsilon_upper_pld,
                    "upper_bound_backend": record.pld_accounting_backend,
                    "accuracy": record.utility_metrics.get("accuracy"),
                    "loss": record.utility_metrics.get("loss"),
                    "training_elapsed_seconds": training_case["elapsed_seconds"],
                    "training_result_path": training_case["training_result_path"],
                    "training_run_id": record.training_run_id,
                }
            )

            started = time.time()
            passive_row = support_scaled.run_passive_support_level(
                dataset_name=dataset_name,
                attack_name="passive_negative_loss",
                score_type="negative_loss",
                training_case=training_case,
                support_label="gpu_scale",
                query_budget=QUERY_BUDGET,
                audit_seeds=PASSIVE_AUDIT_SEEDS,
            )
            passive_row["elapsed_seconds"] = round(time.time() - started, 3)
            raw_rows.append(passive_row)

            started = time.time()
            canary_row = support_scaled.run_canary_support_level(
                dataset_name=dataset_name,
                attack_name="canary_random",
                training_case=training_case,
                support_label="gpu_scale",
                num_canaries=NUM_CANARIES,
                audit_seeds=CANARY_AUDIT_SEEDS,
            )
            canary_row["elapsed_seconds"] = round(time.time() - started, 3)
            raw_rows.append(canary_row)

            device = torch.device("cuda")
            target_model = load_model_for_inference(
                config.model,
                training_case["record"].model_artifact_path,
                device=device,
            )

            query_train_indices, query_eval_indices, per_seed_train, per_seed_eval = raw_lira.sample_query_indices(
                train_size=len(bundle.train_dataset),
                eval_size=len(bundle.eval_dataset),
            )
            target_train_losses = raw_lira.compute_loss_for_indices(
                target_model,
                bundle.train_dataset,
                query_train_indices,
                device=device,
                batch_size=config.training.eval_batch_size,
            )
            target_eval_losses = raw_lira.compute_loss_for_indices(
                target_model,
                bundle.eval_dataset,
                query_eval_indices,
                device=device,
                batch_size=config.training.eval_batch_size,
            )

            shadow_started = time.time()
            shadow_train_losses, shadow_eval_losses, shadow_member_sets = raw_lira.train_shadow_losses(
                training_case=training_case,
                dataset_name=dataset_name,
                query_train_indices=query_train_indices,
                query_eval_indices=query_eval_indices,
                k_max=RAW_LIRA_K,
                logger=logger,
            )
            logger.info(
                "GPU-scale shadows ready for %s in %.1fs",
                dataset_name,
                time.time() - shadow_started,
            )

            nl_member_scores, nl_nonmember_scores = raw_lira.negative_loss_scores(
                query_train_indices=query_train_indices,
                query_eval_indices=query_eval_indices,
                per_seed_train=per_seed_train,
                per_seed_eval=per_seed_eval,
                target_train_losses=target_train_losses,
                target_eval_losses=target_eval_losses,
            )
            raw_rows.append(
                raw_lira.build_result_row(
                    dataset_name=dataset_name,
                    attack_name="passive_negative_loss_matched",
                    k=RAW_LIRA_K,
                    training_case=training_case,
                    member_scores=nl_member_scores,
                    nonmember_scores=nl_nonmember_scores,
                    score_direction="higher",
                )
            )

            raw_member_scores, raw_nonmember_scores = raw_lira.raw_lira_scores(
                query_train_indices=query_train_indices,
                query_eval_indices=query_eval_indices,
                per_seed_train=per_seed_train,
                per_seed_eval=per_seed_eval,
                target_train_losses=target_train_losses,
                target_eval_losses=target_eval_losses,
                shadow_train_losses=shadow_train_losses,
                shadow_eval_losses=shadow_eval_losses,
                shadow_member_sets=shadow_member_sets,
                k=RAW_LIRA_K,
            )
            raw_rows.append(
                raw_lira.build_result_row(
                    dataset_name=dataset_name,
                    attack_name="passive_raw_lira",
                    k=RAW_LIRA_K,
                    training_case=training_case,
                    member_scores=raw_member_scores,
                    nonmember_scores=raw_nonmember_scores,
                    score_direction="lower",
                )
            )
        except Exception as exc:
            logger.exception("GPU-scale framework validation failed on dataset %s", dataset_name)
            training_rows.append(
                {
                    "dataset": dataset_name,
                    "status": "error",
                    "error": str(exc),
                    "traceback": traceback.format_exc(),
                }
            )

    normalized_rows = [framework_validation.normalize_attack_row(row) for row in raw_rows]
    checks = framework_validation.build_checks(training_rows, normalized_rows)

    payload = {
        "scope": "gpu-scale full-framework validation pass",
        "cuda_device": torch.cuda.get_device_name(0),
        "training_rows": training_rows,
        "audit_rows": normalized_rows,
        "checks": checks,
        "gpu_scale_parameters": {
            "query_budget_per_seed": QUERY_BUDGET,
            "passive_audit_seeds": PASSIVE_AUDIT_SEEDS,
            "canary_audit_seeds": CANARY_AUDIT_SEEDS,
            "raw_lira_k": RAW_LIRA_K,
            "num_canaries": NUM_CANARIES,
        },
    }
    SUMMARY_JSON.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    framework_validation.write_csv(SUMMARY_CSV, normalized_rows)
    CHECKS_JSON.write_text(json.dumps(checks, indent=2), encoding="utf-8")
    REPORT_MD.write_text(
        framework_validation.build_report(training_rows, normalized_rows, checks),
        encoding="utf-8",
    )
    return payload


def main() -> None:
    require_cuda()
    started = time.time()
    payload = run_gpu_scale_validation()
    elapsed = round(time.time() - started, 3)
    if ARTIFACTS_ZIP.exists():
        ARTIFACTS_ZIP.unlink()
    shutil.make_archive(str(ARTIFACTS_ZIP.with_suffix("")), "zip", RESULTS_DIR)
    print(f"Wrote validation summary to {SUMMARY_JSON}")
    print(f"Wrote validation rows CSV to {SUMMARY_CSV}")
    print(f"Wrote validation checks to {CHECKS_JSON}")
    print(f"Wrote validation report to {REPORT_MD}")
    print(f"Wrote bundled artifacts zip to {ARTIFACTS_ZIP}")
    print(f"Validation rows: {len(payload['audit_rows'])}, elapsed_seconds={elapsed}")
    failed_checks = sum(1 for check in payload["checks"] if check["status"] == "fail")
    if failed_checks:
        raise SystemExit(f"GPU-scale validation completed with {failed_checks} failed checks.")


if __name__ == "__main__":
    main()
""",
    "src/dp_audit_tightness/__init__.py": """\"\"\"Experimental pipeline for DP auditing tightness studies.\"\"\"

__all__ = ["__version__"]

__version__ = "0.1.0"

""",
    "src/dp_audit_tightness/auditing/__init__.py": """\"\"\"Auditing interfaces and track-specific implementations.\"\"\"

""",
    "src/dp_audit_tightness/auditing/base.py": """from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any


@dataclass(slots=True)
class AuditObservation:
    audit_regime: str
    auditor_variant: str
    epsilon_candidate: float | None = None
    ci_half_width: float | None = None
    raw_statistics: dict[str, float] = field(default_factory=dict)
    artifact_payload: dict[str, Any] = field(default_factory=dict)
    member_scores: list[float] = field(default_factory=list)
    nonmember_scores: list[float] = field(default_factory=list)
    score_name: str | None = None
    epsilon_upper_theory: float | None = None
    utility_metrics: dict[str, float] = field(default_factory=dict)
    audited_model_run_id: str | None = None
""",
    "src/dp_audit_tightness/auditing/canary/__init__.py": """\"\"\"Evaluator-controlled canary stress-testing auditors.\"\"\"

""",
    "src/dp_audit_tightness/auditing/canary/generation.py": """from __future__ import annotations

from dataclasses import dataclass
import itertools
import random

from dp_audit_tightness.config import CanaryConfig


@dataclass(slots=True)
class CanaryPayload:
    identifier: str
    target_label: int
    inserted_image: object
    reference_image: object
    descriptor: str


@dataclass(slots=True)
class CanaryInsertionResult:
    augmented_train_dataset: object
    inserted_canaries: list[CanaryPayload]
    reference_canaries: list[CanaryPayload]
    inserted_example_count: int
    unique_inserted_count: int


def generate_canaries(
    config: CanaryConfig,
    seed: int,
    *,
    num_classes: int,
    image_shape: tuple[int, ...] = (1, 28, 28),
) -> list[CanaryPayload]:
    \"\"\"Generate canary payloads for evaluator-controlled auditing.

    Parameters
    ----------
    image_shape : tuple
        (C, H, W) of the target dataset.  Defaults to MNIST (1, 28, 28).
        Pass (3, 32, 32) for CIFAR-10.
    \"\"\"
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for canary generation.") from exc

    rng = random.Random(seed)
    canaries: list[CanaryPayload] = []
    for index in range(config.num_canaries):
        target_label = rng.randrange(num_classes)
        inserted_image, inserted_descriptor = _build_canary_image(
            strategy=config.strategy,
            target_label=target_label,
            seed=seed + index,
            inserted=True,
            image_shape=image_shape,
        )
        reference_image, reference_descriptor = _build_canary_image(
            strategy=config.strategy,
            target_label=target_label,
            seed=seed + index + 10_000,
            inserted=False,
            image_shape=image_shape,
        )
        descriptor = f"inserted={inserted_descriptor};reference={reference_descriptor}"
        canaries.append(
            CanaryPayload(
                identifier=f"{config.strategy}_{index}",
                target_label=target_label,
                inserted_image=inserted_image.to(dtype=torch.float32),
                reference_image=reference_image.to(dtype=torch.float32),
                descriptor=descriptor,
            )
        )
    return canaries


def insert_canaries_into_dataset(
    dataset: object,
    canaries: list[CanaryPayload],
    *,
    insertion_rate: float,
    seed: int | None = None,
) -> CanaryInsertionResult:
    try:
        import torch
        from torch.utils.data import ConcatDataset, TensorDataset
    except ImportError as exc:
        raise RuntimeError("torch is required for canary insertion.") from exc

    if not canaries:
        raise ValueError("At least one canary is required for canary insertion.")

    inserted_example_count = max(1, int(round(len(dataset) * insertion_rate)))
    unique_inserted_count = min(len(canaries), inserted_example_count)
    selected_canaries = list(canaries)
    if seed is not None:
        random.Random(seed).shuffle(selected_canaries)
    selected_canaries = selected_canaries[:unique_inserted_count]
    cycle = itertools.cycle(selected_canaries)
    inserted_images = []
    inserted_labels = []
    for _ in range(inserted_example_count):
        payload = next(cycle)
        inserted_images.append(payload.inserted_image.clone())
        inserted_labels.append(payload.target_label)

    tensor_dataset = TensorDataset(
        torch.stack(inserted_images),
        torch.tensor(inserted_labels, dtype=torch.long),
    )
    augmented_train_dataset = ConcatDataset([dataset, tensor_dataset])
    return CanaryInsertionResult(
        augmented_train_dataset=augmented_train_dataset,
        inserted_canaries=selected_canaries,
        reference_canaries=selected_canaries,
        inserted_example_count=inserted_example_count,
        unique_inserted_count=unique_inserted_count,
    )


def _build_canary_image(
    *,
    strategy: str,
    target_label: int,
    seed: int,
    inserted: bool,
    image_shape: tuple[int, ...] = (1, 28, 28),
) -> tuple[object, str]:
    \"\"\"Build a synthetic canary image.

    Supports arbitrary ``image_shape`` (C, H, W) so the same canary
    strategies work for both MNIST (1, 28, 28) and CIFAR-10 (3, 32, 32).
    \"\"\"
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for canary generation.") from exc

    channels, height, width = image_shape
    generator = torch.Generator().manual_seed(seed)
    image = torch.rand((channels, height, width), generator=generator) * 0.04
    patch_size = {"random_canaries": 3, "improved_canaries": 4, "optimized_canaries": 5}.get(
        strategy,
        3,
    )
    intensity = {"random_canaries": 0.8, "improved_canaries": 0.95, "optimized_canaries": 1.0}.get(
        strategy,
        0.8,
    )
    base_row = 2 + ((target_label * 3 + seed) % (height - patch_size - 2))
    base_col = 2 + ((target_label * 5 + seed // 2) % (width - patch_size - 2))
    if not inserted:
        base_row = (base_row + 7) % (height - patch_size - 1)
        base_col = (base_col + 11) % (width - patch_size - 1)
        intensity *= 0.75

    image[:, base_row : base_row + patch_size, base_col : base_col + patch_size] = intensity
    stripe_end = min(height - 2, width - 2)
    if strategy in {"improved_canaries", "optimized_canaries"}:
        stripe_row = min(target_label, height - 3)
        image[:, stripe_row : stripe_row + 2, 2 : stripe_end] = max(intensity - 0.1, 0.55)
    if strategy == "optimized_canaries":
        stripe_col = min(target_label + 2, width - 3)
        image[:, 2 : stripe_end, stripe_col : stripe_col + 2] = max(intensity - 0.05, 0.6)
        mid_r, mid_c = height // 2 - 2, width // 2 - 2
        image[:, mid_r : mid_r + 4, mid_c : mid_c + 4] = 1.0

    return image.clamp(0.0, 1.0), f"label={target_label},row={base_row},col={base_col},inserted={inserted}"
""",
    "src/dp_audit_tightness/auditing/canary/one_run.py": """from __future__ import annotations

from dataclasses import replace
import logging
import statistics

from dp_audit_tightness.auditing.base import AuditObservation
from dp_audit_tightness.auditing.canary.generation import generate_canaries, insert_canaries_into_dataset
from dp_audit_tightness.auditing.canary.seeding import build_canary_seed_plan
from dp_audit_tightness.config import CanaryAuditConfig, TrainExperimentConfig
from dp_audit_tightness.data.datasets import DatasetBundle, load_dataset_bundle
from dp_audit_tightness.models.io import load_model_from_state_dict
from dp_audit_tightness.training.dp_sgd import train_dp_sgd
from dp_audit_tightness.utils.results import TrainingRunRecord
from dp_audit_tightness.utils.seeds import set_global_seed


def run_one_run_canary_audit(
    training_record: TrainingRunRecord,
    training_config: TrainExperimentConfig,
    config: CanaryAuditConfig,
    audit_seed: int,
    *,
    logger: logging.Logger | None = None,
) -> AuditObservation:
    \"\"\"Run one evaluator-controlled canary stress test.

    First-pass implementation:
    - create evaluator-designed canary inputs
    - insert them into the training set
    - retrain a DP-SGD model under the same hyperparameters
    - compare inserted-canary scores to held-out decoy-canary scores
    \"\"\"

    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for canary auditing.") from exc

    active_logger = logger or logging.getLogger("dp_audit_tightness.canary")
    seed_plan = build_canary_seed_plan(
        experiment_seed=training_record.training_seed,
        audit_seed=audit_seed,
        dataset_split_seed=training_record.split_seed,
    )
    base_bundle = load_dataset_bundle(training_config.dataset, split_seed=training_record.split_seed)
    image_shape = _infer_image_shape(training_config.dataset.name)
    canaries = generate_canaries(
        config.canary,
        seed=seed_plan.canary_generation_seed,
        num_classes=training_config.model.num_classes,
        image_shape=image_shape,
    )
    insertion_result = insert_canaries_into_dataset(
        base_bundle.train_dataset,
        canaries,
        insertion_rate=config.canary.insertion_rate,
        seed=seed_plan.canary_insertion_seed,
    )
    augmented_bundle = DatasetBundle(
        train_dataset=insertion_result.augmented_train_dataset,
        eval_dataset=base_bundle.eval_dataset,
        input_dim=base_bundle.input_dim,
        num_classes=base_bundle.num_classes,
        train_size=base_bundle.train_size + insertion_result.inserted_example_count,
        eval_size=base_bundle.eval_size,
    )

    audited_training_config = replace(
        training_config,
        experiment_name=f"{training_config.experiment_name}_{config.auditor_variant}_canary",
        run=replace(
            training_config.run,
            training_seed=seed_plan.retrain_seed,
            save_checkpoint=False,
            notes=(
                "Evaluator-controlled canary stress-test retraining. "
                "This model is used only for controlled leakage evaluation."
            ),
        ),
    )
    set_global_seed(seed_plan.retrain_seed)
    training_outcome = train_dp_sgd(
        audited_training_config,
        active_logger,
        dataset_bundle=augmented_bundle,
        save_checkpoint=False,
        run_descriptor=(
            f"{audited_training_config.experiment_name}"
            f"_exp{seed_plan.experiment_seed}_audit{seed_plan.audit_seed}"
        ),
        return_model_state=True,
    )
    if training_outcome.model_state_dict is None:
        raise RuntimeError("Canary stress-test training did not return model weights.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = load_model_from_state_dict(
        training_config.model,
        training_outcome.model_state_dict,
        device=device,
    )
    member_scores = _score_canaries(model, insertion_result.inserted_canaries, device=device)
    nonmember_scores = _score_reference_canaries(model, insertion_result.reference_canaries, device=device)
    mean_member_score = statistics.fmean(member_scores)
    mean_nonmember_score = statistics.fmean(nonmember_scores)
    return AuditObservation(
        audit_regime=config.audit_regime,
        auditor_variant=config.auditor_variant,
        raw_statistics={
            "experiment_seed": float(seed_plan.experiment_seed),
            "audit_seed": float(seed_plan.audit_seed),
            "canary_generation_seed": float(seed_plan.canary_generation_seed),
            "canary_insertion_seed": float(seed_plan.canary_insertion_seed),
            "retrain_seed": float(seed_plan.retrain_seed),
            "inserted_example_count": float(insertion_result.inserted_example_count),
            "unique_canary_count": float(insertion_result.unique_inserted_count),
            "member_score_mean": float(mean_member_score),
            "nonmember_score_mean": float(mean_nonmember_score),
            "score_mean_gap": float(mean_member_score - mean_nonmember_score),
        },
        artifact_payload={
            "audit_mode": "one_run",
            "canary_strategy": config.canary.strategy,
            "optimize_steps": config.canary.optimize_steps,
            "seed_plan": seed_plan.to_dict(),
            "canary_training_run_id": training_outcome.record.training_run_id,
            "canary_training_epsilon_upper_theory": training_outcome.record.epsilon_upper_theory,
            "canary_training_utility_metrics": training_outcome.record.utility_metrics,
            "inserted_canaries": [
                {
                    "identifier": payload.identifier,
                    "target_label": payload.target_label,
                    "descriptor": payload.descriptor,
                }
                for payload in insertion_result.inserted_canaries
            ],
        },
        member_scores=member_scores,
        nonmember_scores=nonmember_scores,
        score_name="target_label_logit_margin",
        epsilon_upper_theory=training_outcome.record.epsilon_upper_theory,
        utility_metrics=training_outcome.record.utility_metrics,
        audited_model_run_id=training_outcome.record.training_run_id,
    )


def _score_canaries(model, canaries, *, device) -> list[float]:
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for canary scoring.") from exc

    with torch.no_grad():
        images = torch.stack([payload.inserted_image for payload in canaries]).to(device)
        labels = torch.tensor([payload.target_label for payload in canaries], dtype=torch.long, device=device)
        logits = model(images)
        return _target_margin_scores(logits, labels)


def _score_reference_canaries(model, canaries, *, device) -> list[float]:
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for canary scoring.") from exc

    with torch.no_grad():
        images = torch.stack([payload.reference_image for payload in canaries]).to(device)
        labels = torch.tensor([payload.target_label for payload in canaries], dtype=torch.long, device=device)
        logits = model(images)
        return _target_margin_scores(logits, labels)


def _target_margin_scores(logits, labels) -> list[float]:
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for canary scoring.") from exc

    target_logits = logits.gather(1, labels.unsqueeze(1)).squeeze(1)
    mask = torch.ones_like(logits, dtype=torch.bool)
    mask.scatter_(1, labels.unsqueeze(1), False)
    other_logits = logits.masked_fill(~mask, float("-inf"))
    strongest_other = other_logits.max(dim=1).values
    return (target_logits - strongest_other).detach().cpu().tolist()


# Dataset name -> (C, H, W) for canary image generation.
_IMAGE_SHAPES: dict[str, tuple[int, int, int]] = {
    "mnist": (1, 28, 28),
    "cifar10": (3, 32, 32),
}


def _infer_image_shape(dataset_name: str) -> tuple[int, int, int]:
    name = dataset_name.lower()
    if name in _IMAGE_SHAPES:
        return _IMAGE_SHAPES[name]
    # Tabular datasets don't have a natural image shape; use a reasonable
    # 1-channel square for canary construction.
    return (1, 28, 28)
""",
    "src/dp_audit_tightness/auditing/canary/repeated_run.py": """from __future__ import annotations

import logging
import statistics

from dp_audit_tightness.auditing.base import AuditObservation
from dp_audit_tightness.auditing.canary.one_run import run_one_run_canary_audit
from dp_audit_tightness.auditing.canary.seeding import build_canary_seed_plan
from dp_audit_tightness.config import CanaryAuditConfig, TrainExperimentConfig
from dp_audit_tightness.utils.results import TrainingRunRecord


def run_repeated_canary_audit(
    training_record: TrainingRunRecord,
    training_config: TrainExperimentConfig,
    config: CanaryAuditConfig,
    *,
    logger: logging.Logger | None = None,
) -> AuditObservation:
    observations = [
        run_one_run_canary_audit(
            training_record=training_record,
            training_config=training_config,
            config=config,
            audit_seed=seed,
            logger=logger,
        )
        for seed in config.repeated_seeds
    ]
    member_scores = [
        score
        for observation in observations
        for score in observation.member_scores
    ]
    nonmember_scores = [
        score
        for observation in observations
        for score in observation.nonmember_scores
    ]
    epsilon_values = [
        observation.epsilon_upper_theory
        for observation in observations
        if observation.epsilon_upper_theory is not None
    ]
    utility_keys = {
        key
        for observation in observations
        for key in observation.utility_metrics.keys()
    }
    averaged_utility_metrics = {
        key: statistics.fmean(
            observation.utility_metrics[key]
            for observation in observations
            if key in observation.utility_metrics
        )
        for key in utility_keys
    }
    return AuditObservation(
        audit_regime=config.audit_regime,
        auditor_variant=config.auditor_variant,
        raw_statistics={
            "experiment_seed": float(training_record.training_seed),
            "num_runs": float(len(observations)),
            "member_score_mean": float(statistics.fmean(member_scores)),
            "nonmember_score_mean": float(statistics.fmean(nonmember_scores)),
            "score_mean_gap": float(statistics.fmean(member_scores) - statistics.fmean(nonmember_scores)),
        },
        artifact_payload={
            "audit_mode": "repeated_run",
            "experiment_seed": training_record.training_seed,
            "repeated_audit_seeds": list(config.repeated_seeds),
            "derived_seed_plans": [
                build_canary_seed_plan(
                    experiment_seed=training_record.training_seed,
                    audit_seed=seed,
                    dataset_split_seed=training_record.split_seed,
                ).to_dict()
                for seed in config.repeated_seeds
            ],
            "per_seed_runs": [
                {
                    "audit_seed": seed,
                    "seed_plan": build_canary_seed_plan(
                        experiment_seed=training_record.training_seed,
                        audit_seed=seed,
                        dataset_split_seed=training_record.split_seed,
                    ).to_dict(),
                    "audited_model_run_id": observation.audited_model_run_id,
                    "epsilon_upper_theory": observation.epsilon_upper_theory,
                    "utility_metrics": observation.utility_metrics,
                    "member_score_mean": statistics.fmean(observation.member_scores),
                    "nonmember_score_mean": statistics.fmean(observation.nonmember_scores),
                }
                for seed, observation in zip(config.repeated_seeds, observations, strict=False)
            ],
        },
        member_scores=member_scores,
        nonmember_scores=nonmember_scores,
        score_name=observations[0].score_name if observations else None,
        epsilon_upper_theory=statistics.fmean(epsilon_values) if epsilon_values else None,
        utility_metrics=averaged_utility_metrics,
    )
""",
    "src/dp_audit_tightness/auditing/canary/seeding.py": """from __future__ import annotations

from dataclasses import asdict, dataclass


_SEED_MODULUS = 2_147_483_647


@dataclass(slots=True)
class CanarySeedPlan:
    experiment_seed: int
    audit_seed: int
    dataset_split_seed: int
    canary_generation_seed: int
    canary_insertion_seed: int
    retrain_seed: int

    def to_dict(self) -> dict[str, int]:
        return asdict(self)


def build_canary_seed_plan(
    *,
    experiment_seed: int,
    audit_seed: int,
    dataset_split_seed: int,
) -> CanarySeedPlan:
    \"\"\"Derive deterministic canary seeds from the experiment seed and audit seed.

    This ensures that repeated-run canary audits remain reproducible while still
    changing materially across different experiment seeds.
    \"\"\"

    return CanarySeedPlan(
        experiment_seed=experiment_seed,
        audit_seed=audit_seed,
        dataset_split_seed=dataset_split_seed,
        canary_generation_seed=_derive_seed(experiment_seed, audit_seed, stream=11),
        canary_insertion_seed=_derive_seed(experiment_seed, audit_seed, stream=23),
        retrain_seed=_derive_seed(experiment_seed, audit_seed, stream=37),
    )


def _derive_seed(experiment_seed: int, audit_seed: int, *, stream: int) -> int:
    seed = (
        abs(experiment_seed) * 1_000_003
        + abs(audit_seed) * 9_176
        + stream * 104_729
        + 17
    ) % _SEED_MODULUS
    return seed if seed != 0 else stream + 1
""",
    "src/dp_audit_tightness/auditing/passive/__init__.py": """\"\"\"Passive final-model-only auditors.\"\"\"

""",
    "src/dp_audit_tightness/auditing/passive/auditors.py": """from __future__ import annotations

import random
import statistics

from dp_audit_tightness.auditing.base import AuditObservation
from dp_audit_tightness.auditing.passive.calibration import (
    apply_temperature,
    fit_temperature,
    percentile_rank_transform,
    zscore_with_reference,
)
from dp_audit_tightness.config import PassiveAuditConfig, TrainExperimentConfig
from dp_audit_tightness.data.datasets import load_dataset_bundle
from dp_audit_tightness.models.io import load_model_for_inference
from dp_audit_tightness.utils.results import TrainingRunRecord


def run_passive_audit_once(
    training_record: TrainingRunRecord,
    training_config: TrainExperimentConfig,
    config: PassiveAuditConfig,
    seed: int,
    *,
    dataset_bundle=None,
    model=None,
) -> AuditObservation:
    \"\"\"Run one passive final-model-only audit using model outputs only.\"\"\"

    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for passive auditing.") from exc

    active_bundle = dataset_bundle or load_dataset_bundle(
        training_config.dataset,
        split_seed=training_record.split_seed,
    )
    if training_record.model_artifact_path is None:
        raise ValueError("Passive auditing requires a saved model artifact path.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    active_model = model or load_model_for_inference(
        training_config.model,
        training_record.model_artifact_path,
        device=device,
    )
    rng = random.Random(seed)

    calibration_budget = int(config.passive.query_budget * config.passive.calibration_fraction)
    member_budget = max(1, config.passive.query_budget // 2)
    nonmember_budget = max(1, config.passive.query_budget - member_budget)

    member_indices = _sample_indices(len(active_bundle.train_dataset), member_budget, rng)
    nonmember_indices = _sample_indices(len(active_bundle.eval_dataset), nonmember_budget, rng)
    nonmember_index_set = set(nonmember_indices)
    remaining_eval_indices = [
        index for index in range(len(active_bundle.eval_dataset)) if index not in nonmember_index_set
    ]
    calibration_indices = (
        _sample_indices(len(remaining_eval_indices), calibration_budget, rng, values=remaining_eval_indices)
        if calibration_budget > 0
        else []
    )

    member_logits, member_labels = _collect_logits_and_labels(
        active_model,
        active_bundle.train_dataset,
        member_indices,
        device=device,
        batch_size=training_record.batch_size,
    )
    nonmember_logits, nonmember_labels = _collect_logits_and_labels(
        active_model,
        active_bundle.eval_dataset,
        nonmember_indices,
        device=device,
        batch_size=training_record.batch_size,
    )
    calibration_logits = calibration_labels = None
    if calibration_indices:
        calibration_logits, calibration_labels = _collect_logits_and_labels(
            active_model,
            active_bundle.eval_dataset,
            calibration_indices,
            device=device,
            batch_size=training_record.batch_size,
        )

    temperature = 1.0
    if config.passive.calibration_method == "temperature_scaling" and calibration_logits is not None:
        temperature = fit_temperature(calibration_logits, calibration_labels)
        member_logits = apply_temperature(member_logits, temperature)
        nonmember_logits = apply_temperature(nonmember_logits, temperature)
        calibration_logits = apply_temperature(calibration_logits, temperature)

    member_scores, nonmember_scores, calibration_reference = _build_membership_scores(
        member_logits=member_logits,
        member_labels=member_labels,
        nonmember_logits=nonmember_logits,
        nonmember_labels=nonmember_labels,
        calibration_logits=calibration_logits,
        calibration_labels=calibration_labels,
        score_type=config.passive.score_type,
        calibration_method=config.passive.calibration_method,
    )
    return AuditObservation(
        audit_regime=config.audit_regime,
        auditor_variant=config.auditor_variant,
        raw_statistics={
            "seed": float(seed),
            "query_budget": float(config.passive.query_budget),
            "calibration_budget": float(len(calibration_indices)),
            "member_score_mean": float(statistics.fmean(member_scores)),
            "nonmember_score_mean": float(statistics.fmean(nonmember_scores)),
            "score_mean_gap": float(statistics.fmean(member_scores) - statistics.fmean(nonmember_scores)),
            "temperature": float(temperature),
        },
        artifact_payload={
            "score_type": config.passive.score_type,
            "calibration_method": config.passive.calibration_method,
            "calibration_fraction": config.passive.calibration_fraction,
            "calibration_reference_size": len(calibration_reference),
        },
        member_scores=member_scores,
        nonmember_scores=nonmember_scores,
        score_name=config.passive.score_type,
        epsilon_upper_theory=training_record.epsilon_upper_theory,
        utility_metrics=training_record.utility_metrics,
        audited_model_run_id=training_record.training_run_id,
    )


def run_repeated_passive_audit(
    training_record: TrainingRunRecord,
    training_config: TrainExperimentConfig,
    config: PassiveAuditConfig,
) -> AuditObservation:
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for passive auditing.") from exc

    dataset_bundle = load_dataset_bundle(training_config.dataset, split_seed=training_record.split_seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = load_model_for_inference(
        training_config.model,
        training_record.model_artifact_path,
        device=device,
    )
    observations = [
        run_passive_audit_once(
            training_record=training_record,
            training_config=training_config,
            config=config,
            seed=seed,
            dataset_bundle=dataset_bundle,
            model=model,
        )
        for seed in config.repeated_seeds
    ]
    member_scores = [score for observation in observations for score in observation.member_scores]
    nonmember_scores = [score for observation in observations for score in observation.nonmember_scores]
    return AuditObservation(
        audit_regime=config.audit_regime,
        auditor_variant=config.auditor_variant,
        raw_statistics={
            "num_runs": float(len(observations)),
            "member_score_mean": float(statistics.fmean(member_scores)),
            "nonmember_score_mean": float(statistics.fmean(nonmember_scores)),
            "score_mean_gap": float(statistics.fmean(member_scores) - statistics.fmean(nonmember_scores)),
        },
        artifact_payload={
            "score_type": config.passive.score_type,
            "calibration_method": config.passive.calibration_method,
            "per_seed_runs": [
                {
                    "seed": seed,
                    "member_score_mean": statistics.fmean(observation.member_scores),
                    "nonmember_score_mean": statistics.fmean(observation.nonmember_scores),
                    "score_mean_gap": statistics.fmean(observation.member_scores)
                    - statistics.fmean(observation.nonmember_scores),
                }
                for seed, observation in zip(config.repeated_seeds, observations, strict=False)
            ],
        },
        member_scores=member_scores,
        nonmember_scores=nonmember_scores,
        score_name=observations[0].score_name if observations else None,
        epsilon_upper_theory=training_record.epsilon_upper_theory,
        utility_metrics=training_record.utility_metrics,
        audited_model_run_id=training_record.training_run_id,
    )


def _sample_indices(total_size: int, sample_size: int, rng: random.Random, *, values=None) -> list[int]:
    population = list(range(total_size)) if values is None else list(values)
    sample_size = min(sample_size, len(population))
    return rng.sample(population, sample_size)


def _collect_logits_and_labels(model, dataset, indices, *, device, batch_size: int):
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for passive auditing.") from exc

    if not indices:
        raise ValueError("At least one index is required to collect model outputs.")

    logits_batches = []
    label_batches = []
    with torch.no_grad():
        for start in range(0, len(indices), max(1, batch_size)):
            batch_indices = indices[start : start + max(1, batch_size)]
            images = []
            labels = []
            for index in batch_indices:
                image, label = dataset[index]
                images.append(image)
                labels.append(label)
            image_tensor = torch.stack(images).to(device)
            label_tensor = torch.tensor(labels, dtype=torch.long, device=device)
            logits_batches.append(model(image_tensor))
            label_batches.append(label_tensor)

    return torch.cat(logits_batches, dim=0), torch.cat(label_batches, dim=0)


def _build_membership_scores(
    *,
    member_logits,
    member_labels,
    nonmember_logits,
    nonmember_labels,
    calibration_logits,
    calibration_labels,
    score_type: str,
    calibration_method: str,
) -> tuple[list[float], list[float], list[float]]:
    member_components = _score_components(member_logits, member_labels)
    nonmember_components = _score_components(nonmember_logits, nonmember_labels)
    if calibration_logits is not None and calibration_labels is not None:
        calibration_components = _score_components(calibration_logits, calibration_labels)
    else:
        calibration_components = nonmember_components

    if score_type == "negative_loss":
        member_scores = member_components["negative_loss"]
        nonmember_scores = nonmember_components["negative_loss"]
        calibration_reference = calibration_components["negative_loss"]
    elif score_type == "max_probability":
        member_scores = member_components["max_probability"]
        nonmember_scores = nonmember_components["max_probability"]
        calibration_reference = calibration_components["max_probability"]
    elif score_type == "logit_margin":
        member_scores = member_components["logit_margin"]
        nonmember_scores = nonmember_components["logit_margin"]
        calibration_reference = calibration_components["logit_margin"]
    elif score_type == "score_fusion":
        member_scores, nonmember_scores, calibration_reference = _score_fusion(
            member_components,
            nonmember_components,
            calibration_components,
            calibration_method=calibration_method,
        )
    else:
        raise NotImplementedError(f"Unsupported passive score_type: {score_type}")

    return (
        member_scores.detach().cpu().tolist(),
        nonmember_scores.detach().cpu().tolist(),
        calibration_reference.detach().cpu().tolist(),
    )


def _score_components(logits, labels):
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for passive auditing.") from exc

    probabilities = torch.softmax(logits, dim=1)
    top_two = probabilities.topk(k=2, dim=1).values
    max_probability = top_two[:, 0]
    logit_margin = top_two[:, 0] - top_two[:, 1]
    negative_loss = -torch.nn.functional.cross_entropy(logits, labels, reduction="none")
    return {
        "negative_loss": negative_loss,
        "max_probability": max_probability,
        "logit_margin": logit_margin,
    }


def _score_fusion(member_components, nonmember_components, calibration_components, *, calibration_method: str):
    if calibration_method == "isotonic_like":
        member_scores = _average_tensors(
            percentile_rank_transform(member_components["negative_loss"], calibration_components["negative_loss"]),
            percentile_rank_transform(member_components["max_probability"], calibration_components["max_probability"]),
            percentile_rank_transform(member_components["logit_margin"], calibration_components["logit_margin"]),
        )
        nonmember_scores = _average_tensors(
            percentile_rank_transform(nonmember_components["negative_loss"], calibration_components["negative_loss"]),
            percentile_rank_transform(nonmember_components["max_probability"], calibration_components["max_probability"]),
            percentile_rank_transform(nonmember_components["logit_margin"], calibration_components["logit_margin"]),
        )
        calibration_reference = _average_tensors(
            percentile_rank_transform(calibration_components["negative_loss"], calibration_components["negative_loss"]),
            percentile_rank_transform(calibration_components["max_probability"], calibration_components["max_probability"]),
            percentile_rank_transform(calibration_components["logit_margin"], calibration_components["logit_margin"]),
        )
        return member_scores, nonmember_scores, calibration_reference

    member_scores = _average_tensors(
        zscore_with_reference(member_components["negative_loss"], calibration_components["negative_loss"]),
        zscore_with_reference(member_components["max_probability"], calibration_components["max_probability"]),
        zscore_with_reference(member_components["logit_margin"], calibration_components["logit_margin"]),
    )
    nonmember_scores = _average_tensors(
        zscore_with_reference(nonmember_components["negative_loss"], calibration_components["negative_loss"]),
        zscore_with_reference(nonmember_components["max_probability"], calibration_components["max_probability"]),
        zscore_with_reference(nonmember_components["logit_margin"], calibration_components["logit_margin"]),
    )
    calibration_reference = _average_tensors(
        zscore_with_reference(calibration_components["negative_loss"], calibration_components["negative_loss"]),
        zscore_with_reference(calibration_components["max_probability"], calibration_components["max_probability"]),
        zscore_with_reference(calibration_components["logit_margin"], calibration_components["logit_margin"]),
    )
    return member_scores, nonmember_scores, calibration_reference


def _average_tensors(*tensors):
    return sum(tensors) / len(tensors)
""",
    "src/dp_audit_tightness/auditing/passive/calibration.py": """from __future__ import annotations


def fit_temperature(logits, labels, max_iter: int = 50) -> float:
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for temperature scaling.") from exc

    if logits.numel() == 0:
        return 1.0

    temperature = torch.nn.Parameter(torch.ones((), device=logits.device))
    optimizer = torch.optim.LBFGS([temperature], lr=0.1, max_iter=max_iter, line_search_fn="strong_wolfe")
    criterion = torch.nn.CrossEntropyLoss()

    def closure():
        optimizer.zero_grad()
        loss = criterion(logits / temperature.clamp(min=0.05), labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(temperature.detach().clamp(min=0.05, max=10.0).item())


def apply_temperature(logits, temperature: float):
    scale = max(float(temperature), 1e-6)
    return logits / scale


def zscore_with_reference(scores, reference_scores):
    mean = reference_scores.mean()
    std = reference_scores.std(unbiased=False).clamp(min=1e-6)
    return (scores - mean) / std


def percentile_rank_transform(scores, reference_scores):
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for score calibration.") from exc

    reference = reference_scores.detach().cpu()
    transformed = []
    for score in scores.detach().cpu():
        transformed.append(float((reference <= score).float().mean().item()))
    return torch.tensor(transformed, dtype=scores.dtype, device=scores.device)
""",
    "src/dp_audit_tightness/config.py": """from __future__ import annotations

from dataclasses import asdict, dataclass
import json
from pathlib import Path
from typing import Any, Mapping

import yaml


@dataclass(slots=True)
class DatasetConfig:
    name: str
    data_dir: str = "data/raw"
    validation_fraction: float = 0.1
    num_workers: int = 0


@dataclass(slots=True)
class ModelConfig:
    name: str
    input_dim: int
    hidden_dim: int
    num_classes: int


@dataclass(slots=True)
class OptimizerConfig:
    name: str
    learning_rate: float
    weight_decay: float = 0.0
    momentum: float = 0.0


@dataclass(slots=True)
class TrainingConfig:
    batch_size: int
    eval_batch_size: int
    epochs: int
    clipping_norm: float
    noise_multiplier: float
    optimizer: OptimizerConfig
    sampling_rate: float | None = None


@dataclass(slots=True)
class PrivacyConfig:
    delta: float
    accountant: str = "rdp"


@dataclass(slots=True)
class RunConfig:
    split_seed: int = 0
    training_seed: int = 0
    split_seeds: list[int] | None = None
    training_seeds: list[int] | None = None
    results_root: str = "results"
    save_checkpoint: bool = True
    save_debug_artifacts: bool = True
    notes: str | None = None


@dataclass(slots=True)
class TrainExperimentConfig:
    experiment_name: str
    dataset: DatasetConfig
    model: ModelConfig
    training: TrainingConfig
    privacy: PrivacyConfig
    run: RunConfig

    @classmethod
    def from_dict(cls, payload: Mapping[str, Any]) -> "TrainExperimentConfig":
        run_payload = dict(payload["run"])
        run_payload["split_seeds"] = _optional_int_list(run_payload.get("split_seeds"))
        run_payload["training_seeds"] = _optional_int_list(run_payload.get("training_seeds"))
        return cls(
            experiment_name=str(payload["experiment_name"]),
            dataset=DatasetConfig(**payload["dataset"]),
            model=ModelConfig(**payload["model"]),
            training=TrainingConfig(
                batch_size=int(payload["training"]["batch_size"]),
                eval_batch_size=int(payload["training"]["eval_batch_size"]),
                epochs=int(payload["training"]["epochs"]),
                clipping_norm=float(payload["training"]["clipping_norm"]),
                noise_multiplier=float(payload["training"]["noise_multiplier"]),
                sampling_rate=_optional_float(payload["training"].get("sampling_rate")),
                optimizer=OptimizerConfig(**payload["training"]["optimizer"]),
            ),
            privacy=PrivacyConfig(**payload["privacy"]),
            run=RunConfig(**run_payload),
        )


@dataclass(slots=True)
class CanaryConfig:
    strategy: str
    num_canaries: int
    insertion_rate: float
    optimize_steps: int = 0


@dataclass(slots=True)
class PassiveAuditorConfig:
    query_budget: int
    score_type: str
    calibration_method: str
    calibration_fraction: float = 0.0


@dataclass(slots=True)
class SaturationConfig:
    min_absolute_improvement: float = 0.05
    min_relative_improvement: float = 0.02
    tightness_ratio_tolerance: float = 0.01
    use_confidence_interval_overlap: bool = True


@dataclass(slots=True)
class AuditRunConfig:
    results_root: str = "results"
    save_debug_artifacts: bool = True
    training_result_paths: list[str] | None = None


@dataclass(slots=True)
class CanaryAuditConfig:
    training_result_path: str
    audit_regime: str
    auditor_variant: str
    auditor_strength_rank: int
    audit_mode: str
    delta: float
    repeated_seeds: list[int]
    canary: CanaryConfig
    saturation: SaturationConfig
    run: AuditRunConfig

    @classmethod
    def from_dict(cls, payload: Mapping[str, Any]) -> "CanaryAuditConfig":
        run_payload = dict(payload["run"])
        run_payload["training_result_paths"] = _optional_string_list(run_payload.get("training_result_paths"))
        return cls(
            training_result_path=str(payload["training_result_path"]),
            audit_regime=str(payload["audit_regime"]),
            auditor_variant=str(payload["auditor_variant"]),
            auditor_strength_rank=int(payload["auditor_strength_rank"]),
            audit_mode=str(payload["audit_mode"]),
            delta=float(payload["delta"]),
            repeated_seeds=[int(seed) for seed in payload.get("repeated_seeds", [])],
            canary=CanaryConfig(**payload["canary"]),
            saturation=SaturationConfig(**payload["saturation"]),
            run=AuditRunConfig(**run_payload),
        )


@dataclass(slots=True)
class PassiveAuditConfig:
    training_result_path: str
    audit_regime: str
    auditor_variant: str
    auditor_strength_rank: int
    delta: float
    repeated_seeds: list[int]
    passive: PassiveAuditorConfig
    saturation: SaturationConfig
    run: AuditRunConfig

    @classmethod
    def from_dict(cls, payload: Mapping[str, Any]) -> "PassiveAuditConfig":
        run_payload = dict(payload["run"])
        run_payload["training_result_paths"] = _optional_string_list(run_payload.get("training_result_paths"))
        return cls(
            training_result_path=str(payload["training_result_path"]),
            audit_regime=str(payload["audit_regime"]),
            auditor_variant=str(payload["auditor_variant"]),
            auditor_strength_rank=int(payload["auditor_strength_rank"]),
            delta=float(payload["delta"]),
            repeated_seeds=[int(seed) for seed in payload.get("repeated_seeds", [])],
            passive=PassiveAuditorConfig(**payload["passive"]),
            saturation=SaturationConfig(**payload["saturation"]),
            run=AuditRunConfig(**run_payload),
        )


def load_train_config(path: str | Path) -> TrainExperimentConfig:
    return TrainExperimentConfig.from_dict(_load_yaml(path))


def load_train_config_snapshot(path: str | Path) -> TrainExperimentConfig:
    return TrainExperimentConfig.from_dict(_load_json(path))


def load_canary_audit_config(path: str | Path) -> CanaryAuditConfig:
    return CanaryAuditConfig.from_dict(_load_yaml(path))


def load_passive_audit_config(path: str | Path) -> PassiveAuditConfig:
    return PassiveAuditConfig.from_dict(_load_yaml(path))


def config_to_dict(config: Any) -> dict[str, Any]:
    return asdict(config)


def _load_yaml(path: str | Path) -> dict[str, Any]:
    with Path(path).open("r", encoding="utf-8") as handle:
        payload = yaml.safe_load(handle)
    if not isinstance(payload, dict):
        raise ValueError(f"Expected mapping in config file: {path}")
    return payload


def _optional_float(value: Any) -> float | None:
    if value is None:
        return None
    return float(value)


def _optional_int_list(value: Any) -> list[int] | None:
    if value is None:
        return None
    return [int(item) for item in value]


def _optional_string_list(value: Any) -> list[str] | None:
    if value is None:
        return None
    return [str(item) for item in value]


def _load_json(path: str | Path) -> dict[str, Any]:
    with Path(path).open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    if not isinstance(payload, dict):
        raise ValueError(f"Expected mapping in config snapshot: {path}")
    return payload
""",
    "src/dp_audit_tightness/data/__init__.py": """\"\"\"Data loading and preprocessing modules.\"\"\"

""",
    "src/dp_audit_tightness/data/datasets.py": """from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

from dp_audit_tightness.config import DatasetConfig
from dp_audit_tightness.data.preprocessing import build_transform


@dataclass(slots=True)
class DatasetBundle:
    train_dataset: object
    eval_dataset: object
    input_dim: int
    num_classes: int
    train_size: int
    eval_size: int


def load_dataset_bundle(config: DatasetConfig, split_seed: int) -> DatasetBundle:
    name = config.name.lower()
    if name == "mnist":
        return _load_mnist(config, split_seed)
    if name == "cifar10":
        return _load_cifar10(config, split_seed)
    if name == "purchase100":
        return _load_purchase100(config, split_seed)
    if name == "adult":
        return _load_adult(config, split_seed)
    raise NotImplementedError(f"Dataset not supported: {config.name}")


# ---------------------------------------------------------------------------
# MNIST
# ---------------------------------------------------------------------------

def _load_mnist(config: DatasetConfig, split_seed: int) -> DatasetBundle:
    import torch
    from torch.utils.data import random_split
    from torchvision.datasets import MNIST

    data_root = Path(config.data_dir)
    transform = build_transform("mnist")
    full_dataset = MNIST(root=data_root, train=True, download=True, transform=transform)
    eval_size = int(len(full_dataset) * config.validation_fraction)
    train_size = len(full_dataset) - eval_size
    generator = torch.Generator().manual_seed(split_seed)
    train_dataset, eval_dataset = random_split(full_dataset, [train_size, eval_size], generator=generator)
    return DatasetBundle(
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        input_dim=28 * 28,
        num_classes=10,
        train_size=train_size,
        eval_size=eval_size,
    )


# ---------------------------------------------------------------------------
# CIFAR-10
# ---------------------------------------------------------------------------

def _load_cifar10(config: DatasetConfig, split_seed: int) -> DatasetBundle:
    import torch
    from torch.utils.data import random_split
    from torchvision.datasets import CIFAR10

    data_root = Path(config.data_dir)
    transform = build_transform("cifar10")
    full_dataset = CIFAR10(root=data_root, train=True, download=True, transform=transform)
    eval_size = int(len(full_dataset) * config.validation_fraction)
    train_size = len(full_dataset) - eval_size
    generator = torch.Generator().manual_seed(split_seed)
    train_dataset, eval_dataset = random_split(full_dataset, [train_size, eval_size], generator=generator)
    return DatasetBundle(
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        input_dim=3 * 32 * 32,      # 3072 (used by tabular fallbacks; CNN ignores this)
        num_classes=10,
        train_size=train_size,
        eval_size=eval_size,
    )


# ---------------------------------------------------------------------------
# Purchase-100  (Shokri et al. 2017 — classic MIA benchmark)
# ---------------------------------------------------------------------------

def _load_purchase100(config: DatasetConfig, split_seed: int) -> DatasetBundle:
    import torch
    from torch.utils.data import TensorDataset, random_split

    data_root = Path(config.data_dir)
    data_path = data_root / "purchase100.npz"
    if not data_path.exists():
        raise FileNotFoundError(
            f"Purchase-100 data not found at {data_path}. "
            "Download the preprocessed dataset and save it as an .npz file with "
            "keys 'features' (float32, shape [N, 600]) and 'labels' (int64, shape [N,])."
        )
    import numpy as np

    data = np.load(data_path)
    features = torch.tensor(data["features"], dtype=torch.float32)
    labels = torch.tensor(data["labels"], dtype=torch.long)
    full_dataset = TensorDataset(features, labels)
    eval_size = int(len(full_dataset) * config.validation_fraction)
    train_size = len(full_dataset) - eval_size
    generator = torch.Generator().manual_seed(split_seed)
    train_dataset, eval_dataset = random_split(full_dataset, [train_size, eval_size], generator=generator)
    num_classes = int(labels.max().item()) + 1
    return DatasetBundle(
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        input_dim=600,
        num_classes=num_classes,
        train_size=train_size,
        eval_size=eval_size,
    )


# ---------------------------------------------------------------------------
# Adult Census Income  (UCI — policy-relevant tabular benchmark)
# ---------------------------------------------------------------------------

def _load_adult(config: DatasetConfig, split_seed: int) -> DatasetBundle:
    import torch
    from torch.utils.data import TensorDataset, random_split

    data_root = Path(config.data_dir)
    data_path = data_root / "adult.npz"
    if not data_path.exists():
        raise FileNotFoundError(
            f"Adult Census data not found at {data_path}. "
            "Download from UCI ML Repository, preprocess with one-hot encoding, "
            "and save as .npz with keys 'features' (float32) and 'labels' (int64)."
        )
    import numpy as np

    data = np.load(data_path)
    features = torch.tensor(data["features"], dtype=torch.float32)
    labels = torch.tensor(data["labels"], dtype=torch.long)
    full_dataset = TensorDataset(features, labels)
    eval_size = int(len(full_dataset) * config.validation_fraction)
    train_size = len(full_dataset) - eval_size
    generator = torch.Generator().manual_seed(split_seed)
    train_dataset, eval_dataset = random_split(full_dataset, [train_size, eval_size], generator=generator)
    input_dim = int(features.shape[1])
    num_classes = int(labels.max().item()) + 1
    return DatasetBundle(
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        input_dim=input_dim,
        num_classes=num_classes,
        train_size=train_size,
        eval_size=eval_size,
    )

""",
    "src/dp_audit_tightness/data/preprocessing.py": """from __future__ import annotations


def build_transform(dataset_name: str):
    try:
        from torchvision import transforms
    except ImportError as exc:
        raise RuntimeError(
            "torchvision is required for dataset preprocessing. Install project dependencies first."
        ) from exc

    name = dataset_name.lower()

    if name == "mnist":
        return transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        )

    if name == "cifar10":
        # Standard CIFAR-10 normalization (per-channel mean and std).
        # No data augmentation — consistent with DP-SGD auditing literature
        # (Nasr et al. 2023, Lu & Groth 2024).
        return transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Normalize(
                    (0.4914, 0.4822, 0.4465),
                    (0.2470, 0.2435, 0.2616),
                ),
            ]
        )

    if name == "purchase100":
        # Purchase-100 is already numeric; no image transforms needed.
        # The dataset loader handles tensor conversion directly.
        return None

    if name == "adult":
        # Adult Census Income is tabular; no image transforms needed.
        # The dataset loader handles preprocessing directly.
        return None

    raise NotImplementedError(f"No transform defined for dataset={dataset_name}")

""",
    "src/dp_audit_tightness/evaluation/__init__.py": """\"\"\"Evaluation metrics and saturation analysis.\"\"\"

""",
    "src/dp_audit_tightness/evaluation/gap_decomposition.py": """\"\"\"Gap decomposition: attribute the tightness gap to its sources.

The total gap between epsilon_upper_theory and epsilon_lower_empirical
can be decomposed into attributable components:

  total_gap = accounting_gap + threat_model_gap + attack_gap + residual

Where:
    accounting_gap   = eps_upper_rdp - eps_upper_pld
        How much of the gap comes from using a looser privacy accountant.

    threat_model_gap = eps_lower_canary - eps_lower_passive
        How much the evaluator-controlled threat model recovers beyond
        the passive threat model.

    attack_gap       = eps_lower_best_possible - eps_lower_best_achieved
        How much stronger attacks could close the remaining gap.
        (Only computable when a stronger attack is available.)

    residual         = eps_upper_pld - eps_lower_canary
        The gap that cannot be attributed to accounting or threat model.
        Captures: worst-case vs. actual data, last-iterate advantage,
        subsampling mismatch, and fundamental limits of the auditor.

This module computes the decomposition from available data and flags
which components are available vs. estimated.
\"\"\"

from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Any


@dataclass(slots=True)
class GapDecomposition:
    \"\"\"Structured decomposition of the tightness gap for one experiment.\"\"\"

    # Identifiers
    training_run_id: str
    dataset: str
    model: str

    # Upper bounds
    epsilon_upper_rdp: float
    epsilon_upper_pld: float | None  # None if PLD accounting not available

    # Lower bounds
    epsilon_lower_canary: float | None
    epsilon_lower_passive: float | None

    # Decomposition components
    total_gap: float                  # eps_upper_rdp - best_lower
    accounting_gap: float | None      # eps_upper_rdp - eps_upper_pld
    threat_model_gap: float | None    # eps_lower_canary - eps_lower_passive
    residual_gap: float | None        # tightest_upper - eps_lower_canary

    # Ratios (for cross-experiment comparison)
    tightness_ratio_canary: float | None
    tightness_ratio_passive: float | None

    # Metadata
    components_available: list[str]   # Which decomposition pieces are computed

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


def decompose_gap(
    *,
    training_run_id: str,
    dataset: str,
    model: str,
    epsilon_upper_rdp: float,
    epsilon_upper_pld: float | None = None,
    epsilon_lower_canary: float | None = None,
    epsilon_lower_passive: float | None = None,
) -> GapDecomposition:
    \"\"\"Compute the gap decomposition from available measurements.

    Parameters
    ----------
    epsilon_upper_rdp : float
        Theoretical upper bound from RDP accountant (always available).
    epsilon_upper_pld : float or None
        Theoretical upper bound from PLD accountant (tighter, optional).
    epsilon_lower_canary : float or None
        Best empirical lower bound from canary auditing.
    epsilon_lower_passive : float or None
        Best empirical lower bound from passive auditing.

    Returns
    -------
    GapDecomposition
        Structured decomposition with all computable components.
    \"\"\"
    components: list[str] = []

    # Accounting gap: how much looser is RDP vs PLD?
    accounting_gap = None
    if epsilon_upper_pld is not None:
        accounting_gap = round(epsilon_upper_rdp - epsilon_upper_pld, 6)
        components.append("accounting_gap")

    # Threat model gap: how much more does canary recover vs passive?
    threat_model_gap = None
    if epsilon_lower_canary is not None and epsilon_lower_passive is not None:
        threat_model_gap = round(epsilon_lower_canary - epsilon_lower_passive, 6)
        components.append("threat_model_gap")

    # Best available lower bound
    lowers = [v for v in (epsilon_lower_canary, epsilon_lower_passive) if v is not None]
    best_lower = max(lowers) if lowers else 0.0

    # Total gap (always computable)
    total_gap = round(epsilon_upper_rdp - best_lower, 6)
    components.append("total_gap")

    # Residual: tightest upper - best canary lower
    residual_gap = None
    tightest_upper = epsilon_upper_pld if epsilon_upper_pld is not None else epsilon_upper_rdp
    if epsilon_lower_canary is not None:
        residual_gap = round(tightest_upper - epsilon_lower_canary, 6)
        components.append("residual_gap")

    # Tightness ratios
    tightness_canary = None
    tightness_passive = None
    if epsilon_upper_rdp > 0:
        if epsilon_lower_canary is not None:
            tightness_canary = round(epsilon_lower_canary / epsilon_upper_rdp, 6)
        if epsilon_lower_passive is not None:
            tightness_passive = round(epsilon_lower_passive / epsilon_upper_rdp, 6)

    return GapDecomposition(
        training_run_id=training_run_id,
        dataset=dataset,
        model=model,
        epsilon_upper_rdp=epsilon_upper_rdp,
        epsilon_upper_pld=epsilon_upper_pld,
        epsilon_lower_canary=epsilon_lower_canary,
        epsilon_lower_passive=epsilon_lower_passive,
        total_gap=total_gap,
        accounting_gap=accounting_gap,
        threat_model_gap=threat_model_gap,
        residual_gap=residual_gap,
        tightness_ratio_canary=tightness_canary,
        tightness_ratio_passive=tightness_passive,
        components_available=components,
    )
""",
    "src/dp_audit_tightness/evaluation/metrics.py": """from __future__ import annotations

from dataclasses import dataclass


@dataclass(slots=True)
class PrivacyTightnessMetrics:
    privacy_loss_gap: float
    tightness_ratio: float | None


def compute_privacy_loss_gap(epsilon_upper_theory: float, epsilon_lower_empirical: float) -> float:
    return epsilon_upper_theory - epsilon_lower_empirical


def compute_tightness_ratio(
    epsilon_upper_theory: float,
    epsilon_lower_empirical: float,
) -> float | None:
    if epsilon_upper_theory <= 0.0:
        return None
    return epsilon_lower_empirical / epsilon_upper_theory


def compute_privacy_tightness_metrics(
    epsilon_upper_theory: float,
    epsilon_lower_empirical: float,
) -> PrivacyTightnessMetrics:
    return PrivacyTightnessMetrics(
        privacy_loss_gap=compute_privacy_loss_gap(epsilon_upper_theory, epsilon_lower_empirical),
        tightness_ratio=compute_tightness_ratio(epsilon_upper_theory, epsilon_lower_empirical),
    )

""",
    "src/dp_audit_tightness/evaluation/saturation.py": """from __future__ import annotations

from dataclasses import dataclass
from typing import Mapping, Sequence

from dp_audit_tightness.config import SaturationConfig


@dataclass(slots=True)
class SaturationDecision:
    saturation_detected: bool
    reason: str | None
    absolute_improvement: float | None
    relative_improvement: float | None
    tightness_ratio_change: float | None
    confidence_interval_overlap: bool | None


def detect_saturation(
    history: Sequence[Mapping[str, float | None]],
    config: SaturationConfig,
) -> SaturationDecision:
    \"\"\"Detect whether stronger auditors have stopped materially improving the lower bound.

    The intended use is to compare successive auditor variants ordered by strength.
    Saturation is treated as an interpretable result, not as an execution failure.
    \"\"\"

    if len(history) < 2:
        return SaturationDecision(
            saturation_detected=False,
            reason=None,
            absolute_improvement=None,
            relative_improvement=None,
            tightness_ratio_change=None,
            confidence_interval_overlap=None,
        )

    previous = history[-2]
    current = history[-1]
    previous_lower = float(previous["epsilon_lower_empirical"] or 0.0)
    current_lower = float(current["epsilon_lower_empirical"] or 0.0)
    absolute_improvement = current_lower - previous_lower
    relative_improvement = None
    if previous_lower > 0.0:
        relative_improvement = absolute_improvement / previous_lower

    previous_ratio = previous.get("tightness_ratio")
    current_ratio = current.get("tightness_ratio")
    tightness_ratio_change = None
    if previous_ratio is not None and current_ratio is not None:
        tightness_ratio_change = float(current_ratio) - float(previous_ratio)

    confidence_interval_overlap = None
    if config.use_confidence_interval_overlap:
        confidence_interval_overlap = _intervals_overlap(
            previous.get("empirical_ci_lower"),
            previous.get("empirical_ci_upper"),
            current.get("empirical_ci_lower"),
            current.get("empirical_ci_upper"),
        )

    absolute_small = absolute_improvement < config.min_absolute_improvement
    relative_small = (
        relative_improvement is None or relative_improvement < config.min_relative_improvement
    )
    tightness_stable = (
        tightness_ratio_change is None
        or abs(tightness_ratio_change) < config.tightness_ratio_tolerance
    )

    improvement_small = absolute_small and relative_small
    ci_supports_saturation = True
    if config.use_confidence_interval_overlap and confidence_interval_overlap is not None:
        ci_supports_saturation = confidence_interval_overlap
    saturation_detected = improvement_small and tightness_stable and ci_supports_saturation

    reasons: list[str] = []
    if absolute_small:
        reasons.append("absolute improvement below threshold")
    if relative_small:
        reasons.append("relative improvement below threshold")
    if tightness_stable:
        reasons.append("tightness ratio stable")
    if confidence_interval_overlap:
        reasons.append("successive confidence intervals overlap")

    return SaturationDecision(
        saturation_detected=saturation_detected,
        reason="; ".join(reasons) if reasons else None,
        absolute_improvement=absolute_improvement,
        relative_improvement=relative_improvement,
        tightness_ratio_change=tightness_ratio_change,
        confidence_interval_overlap=confidence_interval_overlap,
    )


def _intervals_overlap(
    lower_a: float | None,
    upper_a: float | None,
    lower_b: float | None,
    upper_b: float | None,
) -> bool | None:
    values = [lower_a, upper_a, lower_b, upper_b]
    if any(value is None for value in values):
        return None
    return not (float(upper_a) < float(lower_b) or float(upper_b) < float(lower_a))
""",
    "src/dp_audit_tightness/models/__init__.py": """\"\"\"Model definitions.\"\"\"

""",
    "src/dp_audit_tightness/models/cnn.py": """\"\"\"Simple CNN for CIFAR-10 DP-SGD experiments.

Architecture follows the standard used in the DP auditing literature
(Nasr et al. 2023, Lu & Groth NeurIPS 2024): two convolutional layers
with max-pooling followed by two fully-connected layers.  Kept
deliberately simple so that Opacus can wrap it without modification.
\"\"\"

from __future__ import annotations


def build_cnn_cifar10(*, num_classes: int = 10):
    \"\"\"Return a small CNN compatible with Opacus DP-SGD.

    The architecture avoids batch normalization (incompatible with
    per-sample gradient clipping) and uses only operations that Opacus
    supports out of the box.
    \"\"\"
    try:
        import torch
        import torch.nn as nn
    except ImportError as exc:
        raise RuntimeError("torch is required for CNN construction.") from exc

    class SimpleCNN(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            # Conv block 1: 3 -> 32 channels
            self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
            self.pool1 = nn.MaxPool2d(2, 2)       # 32x32 -> 16x16

            # Conv block 2: 32 -> 64 channels
            self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
            self.pool2 = nn.MaxPool2d(2, 2)       # 16x16 -> 8x8

            # Classifier: 64*8*8 = 4096 -> 128 -> num_classes
            self.flatten = nn.Flatten()
            self.fc1 = nn.Linear(64 * 8 * 8, 128)
            self.fc2 = nn.Linear(128, num_classes)
            self.relu = nn.ReLU()

        def forward(self, x):
            x = self.relu(self.conv1(x))
            x = self.pool1(x)
            x = self.relu(self.conv2(x))
            x = self.pool2(x)
            x = self.flatten(x)
            x = self.relu(self.fc1(x))
            x = self.fc2(x)
            return x

    return SimpleCNN()
""",
    "src/dp_audit_tightness/models/io.py": """from __future__ import annotations

from typing import Any, Mapping

from dp_audit_tightness.config import ModelConfig
from dp_audit_tightness.models.simple_mlp import build_model


def load_model_for_inference(
    model_config: ModelConfig,
    checkpoint_path: str,
    device=None,
):
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for model loading.") from exc

    resolved_device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(model_config).to(resolved_device)
    state_dict = torch.load(checkpoint_path, map_location=resolved_device)
    model.load_state_dict(normalize_state_dict_keys(state_dict))
    model.eval()
    return model


def load_model_from_state_dict(
    model_config: ModelConfig,
    state_dict: Mapping[str, Any],
    device=None,
):
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for model loading.") from exc

    resolved_device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(model_config).to(resolved_device)
    model.load_state_dict(normalize_state_dict_keys(state_dict))
    model.eval()
    return model


def export_inference_state_dict(model) -> dict[str, Any]:
    return normalize_state_dict_keys(model.state_dict())


def normalize_state_dict_keys(state_dict: Mapping[str, Any]) -> dict[str, Any]:
    normalized: dict[str, Any] = {}
    for key, value in state_dict.items():
        clean_key = key.removeprefix("_module.")
        normalized[clean_key] = value
    return normalized

""",
    "src/dp_audit_tightness/models/simple_mlp.py": """from __future__ import annotations

from dp_audit_tightness.config import ModelConfig


def build_model(config: ModelConfig):
    \"\"\"Central model factory.  Dispatches on ``config.name``.\"\"\"
    name = config.name.lower()

    if name == "simple_mlp":
        return _build_simple_mlp(config)

    if name == "cnn_cifar10":
        from dp_audit_tightness.models.cnn import build_cnn_cifar10
        return build_cnn_cifar10(num_classes=config.num_classes)

    if name == "tabular_mlp":
        from dp_audit_tightness.models.tabular_mlp import build_tabular_mlp
        return build_tabular_mlp(
            input_dim=config.input_dim,
            hidden_dim=config.hidden_dim,
            num_classes=config.num_classes,
        )

    raise NotImplementedError(f"No model factory registered for model={config.name}")


def _build_simple_mlp(config: ModelConfig):
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for model construction. Install project dependencies first.") from exc

    class SimpleMLP(torch.nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.network = torch.nn.Sequential(
                torch.nn.Flatten(),
                torch.nn.Linear(config.input_dim, config.hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Linear(config.hidden_dim, config.num_classes),
            )

        def forward(self, inputs):
            return self.network(inputs)

    return SimpleMLP()

""",
    "src/dp_audit_tightness/models/tabular_mlp.py": """\"\"\"Tabular MLP for Purchase-100 and Adult Census experiments.

A simple multi-layer perceptron for tabular (non-image) data.
Fully compatible with Opacus DP-SGD — no batch norm, no operations
that break per-sample gradient clipping.
\"\"\"

from __future__ import annotations


def build_tabular_mlp(*, input_dim: int, hidden_dim: int, num_classes: int):
    \"\"\"Return a 2-hidden-layer MLP for tabular classification.

    Architecture: input_dim -> hidden_dim -> hidden_dim // 2 -> num_classes
    \"\"\"
    try:
        import torch
        import torch.nn as nn
    except ImportError as exc:
        raise RuntimeError("torch is required for TabularMLP construction.") from exc

    mid_dim = max(16, hidden_dim // 2)

    class TabularMLP(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.network = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, mid_dim),
                nn.ReLU(),
                nn.Linear(mid_dim, num_classes),
            )

        def forward(self, x):
            return self.network(x)

    return TabularMLP()
""",
    "src/dp_audit_tightness/privacy/__init__.py": """\"\"\"Privacy accounting and empirical lower-bound estimation.\"\"\"

""",
    "src/dp_audit_tightness/privacy/accounting.py": """from __future__ import annotations

from typing import Any


def compute_theoretical_upper_bound(*, privacy_engine: Any, delta: float) -> float:
    \"\"\"Return the theoretical upper bound on privacy loss from the configured accountant.

    This function should only return a quantity with the semantics of a guaranteed upper
    bound under the accountant and mechanism actually used in training.
    \"\"\"

    accountant = getattr(privacy_engine, "accountant", privacy_engine)
    if not hasattr(accountant, "get_epsilon"):
        raise TypeError("privacy_engine/accountant must expose get_epsilon(delta).")
    epsilon_upper_theory = float(accountant.get_epsilon(delta))
    if epsilon_upper_theory < 0.0:
        raise ValueError("Theoretical upper bound on privacy loss must be non-negative.")
    return epsilon_upper_theory

""",
    "src/dp_audit_tightness/privacy/empirical.py": """from __future__ import annotations

from dataclasses import dataclass
import math
from typing import Mapping, Sequence


@dataclass(slots=True)
class EmpiricalEpsilonEstimate:
    epsilon_lower_empirical: float
    epsilon_lower_empirical_point_estimate: float
    epsilon_lower_empirical_conservative: float
    empirical_ci_lower: float | None
    empirical_ci_upper: float | None
    estimation_method: str
    delta: float
    selected_threshold: float | None = None
    selected_event: str | None = None
    selected_event_direction: str | None = None
    selected_tpr: float | None = None
    selected_fpr: float | None = None
    member_event_fraction: float | None = None
    nonmember_event_fraction: float | None = None
    member_favoring: bool | None = None
    no_member_favoring_event_found: bool = False
    selected_event_is_tiny_tail: bool | None = None
    selected_member_event_count: int | None = None
    selected_nonmember_event_count: int | None = None
    warning_message: str | None = None
    num_member_samples: int = 0
    num_nonmember_samples: int = 0


def estimate_empirical_lower_bound(
    *,
    member_scores: Sequence[float] | None = None,
    nonmember_scores: Sequence[float] | None = None,
    audit_statistics: Mapping[str, float] | None = None,
    delta: float,
    score_direction: str = "higher",
    align_event_to_score_direction: bool = False,
    require_member_favoring: bool = False,
    report_confidence_supported_lower_bound: bool = False,
    tiny_tail_fraction_threshold: float = 0.125,
    tiny_tail_min_event_count: int = 5,
) -> EmpiricalEpsilonEstimate:
    \"\"\"Estimate an empirical lower bound on privacy loss from auditor outputs.

    For first-pass implementations, this uses a threshold sweep over member and
    non-member score distributions and converts the best observed hypothesis-testing
    event into a demonstrated lower bound on privacy loss.
    \"\"\"

    if member_scores is not None and nonmember_scores is not None:
        return _estimate_from_score_distributions(
            member_scores=member_scores,
            nonmember_scores=nonmember_scores,
            delta=delta,
            score_direction=score_direction,
            align_event_to_score_direction=align_event_to_score_direction,
            require_member_favoring=require_member_favoring,
            report_confidence_supported_lower_bound=report_confidence_supported_lower_bound,
            tiny_tail_fraction_threshold=tiny_tail_fraction_threshold,
            tiny_tail_min_event_count=tiny_tail_min_event_count,
        )
    if audit_statistics is None:
        raise ValueError("Provide either score distributions or audit_statistics.")

    epsilon_candidate = max(0.0, float(audit_statistics.get("epsilon_candidate", 0.0)))
    ci_half_width = max(0.0, float(audit_statistics.get("ci_half_width", 0.0)))
    return EmpiricalEpsilonEstimate(
        epsilon_lower_empirical=epsilon_candidate,
        epsilon_lower_empirical_point_estimate=epsilon_candidate,
        epsilon_lower_empirical_conservative=max(0.0, epsilon_candidate - ci_half_width),
        empirical_ci_lower=max(0.0, epsilon_candidate - ci_half_width),
        empirical_ci_upper=epsilon_candidate + ci_half_width,
        estimation_method="direct_candidate_passthrough",
        delta=delta,
    )


def _estimate_from_score_distributions(
    *,
    member_scores: Sequence[float],
    nonmember_scores: Sequence[float],
    delta: float,
    score_direction: str,
    align_event_to_score_direction: bool,
    require_member_favoring: bool,
    report_confidence_supported_lower_bound: bool,
    tiny_tail_fraction_threshold: float,
    tiny_tail_min_event_count: int,
) -> EmpiricalEpsilonEstimate:
    member = [float(score) for score in member_scores]
    nonmember = [float(score) for score in nonmember_scores]
    if not member or not nonmember:
        raise ValueError("Member and non-member score lists must both be non-empty.")

    if score_direction == "lower":
        transformed_member = [-score for score in member]
        transformed_nonmember = [-score for score in nonmember]
        selected_event_direction = "score<threshold"
        threshold_transform = lambda value: -value
    elif score_direction != "higher":
        raise ValueError(f"Unsupported score_direction: {score_direction}")
    else:
        transformed_member = member
        transformed_nonmember = nonmember
        selected_event_direction = "score>=threshold"
        threshold_transform = lambda value: value

    thresholds = sorted(set(transformed_member + transformed_nonmember))
    if not thresholds:
        raise ValueError("At least one score threshold is required.")

    if align_event_to_score_direction:
        return _estimate_member_aligned_threshold_sweep(
            member_scores=transformed_member,
            nonmember_scores=transformed_nonmember,
            delta=delta,
            thresholds=thresholds,
            selected_event_direction=selected_event_direction,
            threshold_transform=threshold_transform,
            require_member_favoring=require_member_favoring,
            report_confidence_supported_lower_bound=report_confidence_supported_lower_bound,
            tiny_tail_fraction_threshold=tiny_tail_fraction_threshold,
            tiny_tail_min_event_count=tiny_tail_min_event_count,
        )
    return _estimate_legacy_threshold_sweep(
        member_scores=transformed_member,
        nonmember_scores=transformed_nonmember,
        delta=delta,
        thresholds=thresholds,
        report_confidence_supported_lower_bound=report_confidence_supported_lower_bound,
        tiny_tail_fraction_threshold=tiny_tail_fraction_threshold,
        tiny_tail_min_event_count=tiny_tail_min_event_count,
    )


def _estimate_member_aligned_threshold_sweep(
    *,
    member_scores: Sequence[float],
    nonmember_scores: Sequence[float],
    delta: float,
    thresholds: Sequence[float],
    selected_event_direction: str,
    threshold_transform,
    require_member_favoring: bool,
    report_confidence_supported_lower_bound: bool,
    tiny_tail_fraction_threshold: float,
    tiny_tail_min_event_count: int,
) -> EmpiricalEpsilonEstimate:
    best_point = -math.inf
    best_lower = 0.0
    best_upper = 0.0
    best_threshold: float | None = None
    best_tpr = 0.0
    best_fpr = 0.0
    best_member_favoring = False
    best_member_count = 0
    best_nonmember_count = 0

    for threshold in thresholds:
        member_successes = sum(score >= threshold for score in member_scores)
        nonmember_successes = sum(score >= threshold for score in nonmember_scores)
        candidate = _evaluate_member_aligned_threshold(
            member_successes=member_successes,
            member_trials=len(member_scores),
            nonmember_successes=nonmember_successes,
            nonmember_trials=len(nonmember_scores),
            delta=delta,
        )
        if require_member_favoring and not candidate["member_favoring"]:
            continue
        if candidate["point"] > best_point:
            best_point = candidate["point"]
            best_lower = candidate["lower"]
            best_upper = candidate["upper"]
            best_threshold = threshold
            best_tpr = candidate["tpr"]
            best_fpr = candidate["fpr"]
            best_member_favoring = bool(candidate["member_favoring"])
            best_member_count = member_successes
            best_nonmember_count = nonmember_successes

    if best_threshold is None:
        warning_message = (
            "No member-favoring event found on the scanned threshold grid; "
            "returning a conservative zero empirical lower bound."
        )
        return EmpiricalEpsilonEstimate(
            epsilon_lower_empirical=0.0,
            epsilon_lower_empirical_point_estimate=0.0,
            epsilon_lower_empirical_conservative=0.0,
            empirical_ci_lower=0.0,
            empirical_ci_upper=0.0,
            estimation_method="threshold_sweep_member_aligned_no_member_favoring_event",
            delta=delta,
            selected_threshold=None,
            selected_event=None,
            selected_event_direction=selected_event_direction,
            selected_tpr=None,
            selected_fpr=None,
            member_event_fraction=None,
            nonmember_event_fraction=None,
            member_favoring=False,
            no_member_favoring_event_found=True,
            selected_event_is_tiny_tail=None,
            selected_member_event_count=None,
            selected_nonmember_event_count=None,
            warning_message=warning_message,
            num_member_samples=len(member_scores),
            num_nonmember_samples=len(nonmember_scores),
        )

    point_estimate = max(0.0, best_point)
    conservative_estimate = max(0.0, best_lower)
    selected_event_is_tiny_tail = _is_tiny_tail_event(
        member_event_fraction=best_tpr,
        nonmember_event_fraction=best_fpr,
        member_event_count=best_member_count,
        nonmember_event_count=best_nonmember_count,
        member_trials=len(member_scores),
        nonmember_trials=len(nonmember_scores),
        fraction_threshold=tiny_tail_fraction_threshold,
        min_event_count=tiny_tail_min_event_count,
    )
    warning_message = _compose_warning_message(
        report_confidence_supported_lower_bound=report_confidence_supported_lower_bound,
        point_estimate=point_estimate,
        conservative_estimate=conservative_estimate,
        selected_event_is_tiny_tail=selected_event_is_tiny_tail,
        existing_warning=None,
    )

    return EmpiricalEpsilonEstimate(
        epsilon_lower_empirical=conservative_estimate
        if report_confidence_supported_lower_bound
        else point_estimate,
        epsilon_lower_empirical_point_estimate=point_estimate,
        epsilon_lower_empirical_conservative=conservative_estimate,
        empirical_ci_lower=max(0.0, best_lower),
        empirical_ci_upper=max(0.0, best_upper),
        estimation_method="threshold_sweep_member_aligned_member_favoring"
        if require_member_favoring
        else "threshold_sweep_member_aligned",
        delta=delta,
        selected_threshold=threshold_transform(best_threshold),
        selected_event=selected_event_direction,
        selected_event_direction=selected_event_direction,
        selected_tpr=best_tpr,
        selected_fpr=best_fpr,
        member_event_fraction=best_tpr,
        nonmember_event_fraction=best_fpr,
        member_favoring=best_member_favoring,
        no_member_favoring_event_found=False,
        selected_event_is_tiny_tail=selected_event_is_tiny_tail,
        selected_member_event_count=best_member_count,
        selected_nonmember_event_count=best_nonmember_count,
        warning_message=warning_message,
        num_member_samples=len(member_scores),
        num_nonmember_samples=len(nonmember_scores),
    )


def _estimate_legacy_threshold_sweep(
    *,
    member_scores: Sequence[float],
    nonmember_scores: Sequence[float],
    delta: float,
    thresholds: Sequence[float],
    report_confidence_supported_lower_bound: bool,
    tiny_tail_fraction_threshold: float,
    tiny_tail_min_event_count: int,
) -> EmpiricalEpsilonEstimate:
    best_point = -math.inf
    best_lower = 0.0
    best_upper = 0.0
    best_threshold = thresholds[0]
    best_event = "score>=threshold"
    best_tpr = 0.0
    best_fpr = 0.0
    best_member_count = 0
    best_nonmember_count = 0

    for threshold in thresholds:
        member_successes = sum(score >= threshold for score in member_scores)
        nonmember_successes = sum(score >= threshold for score in nonmember_scores)
        candidate = _evaluate_legacy_threshold(
            member_successes=member_successes,
            member_trials=len(member_scores),
            nonmember_successes=nonmember_successes,
            nonmember_trials=len(nonmember_scores),
            delta=delta,
        )
        if candidate["point"] > best_point:
            best_point = candidate["point"]
            best_lower = candidate["lower"]
            best_upper = candidate["upper"]
            best_threshold = threshold
            best_event = candidate["event"]
            best_tpr = candidate["tpr"]
            best_fpr = candidate["fpr"]
            best_member_count = member_successes
            best_nonmember_count = nonmember_successes

    member_event_fraction = best_tpr if best_event == "score>=threshold" else 1.0 - best_tpr
    nonmember_event_fraction = best_fpr if best_event == "score>=threshold" else 1.0 - best_fpr
    selected_member_event_count = best_member_count if best_event == "score>=threshold" else len(member_scores) - best_member_count
    selected_nonmember_event_count = best_nonmember_count if best_event == "score>=threshold" else len(nonmember_scores) - best_nonmember_count
    point_estimate = max(0.0, best_point)
    conservative_estimate = max(0.0, best_lower)
    selected_event_is_tiny_tail = _is_tiny_tail_event(
        member_event_fraction=member_event_fraction,
        nonmember_event_fraction=nonmember_event_fraction,
        member_event_count=selected_member_event_count,
        nonmember_event_count=selected_nonmember_event_count,
        member_trials=len(member_scores),
        nonmember_trials=len(nonmember_scores),
        fraction_threshold=tiny_tail_fraction_threshold,
        min_event_count=tiny_tail_min_event_count,
    )
    warning_message = _compose_warning_message(
        report_confidence_supported_lower_bound=report_confidence_supported_lower_bound,
        point_estimate=point_estimate,
        conservative_estimate=conservative_estimate,
        selected_event_is_tiny_tail=selected_event_is_tiny_tail,
        existing_warning=None,
    )

    return EmpiricalEpsilonEstimate(
        epsilon_lower_empirical=conservative_estimate
        if report_confidence_supported_lower_bound
        else point_estimate,
        epsilon_lower_empirical_point_estimate=point_estimate,
        epsilon_lower_empirical_conservative=conservative_estimate,
        empirical_ci_lower=max(0.0, best_lower),
        empirical_ci_upper=max(0.0, best_upper),
        estimation_method="threshold_sweep_binary_hypothesis_test",
        delta=delta,
        selected_threshold=best_threshold,
        selected_event=best_event,
        selected_event_direction=best_event,
        selected_tpr=best_tpr,
        selected_fpr=best_fpr,
        member_event_fraction=member_event_fraction,
        nonmember_event_fraction=nonmember_event_fraction,
        member_favoring=(member_event_fraction > nonmember_event_fraction),
        no_member_favoring_event_found=False,
        selected_event_is_tiny_tail=selected_event_is_tiny_tail,
        selected_member_event_count=selected_member_event_count,
        selected_nonmember_event_count=selected_nonmember_event_count,
        warning_message=warning_message,
        num_member_samples=len(member_scores),
        num_nonmember_samples=len(nonmember_scores),
    )


def _evaluate_member_aligned_threshold(
    *,
    member_successes: int,
    member_trials: int,
    nonmember_successes: int,
    nonmember_trials: int,
    delta: float,
) -> dict[str, float | str]:
    tpr = member_successes / max(1, member_trials)
    fpr = nonmember_successes / max(1, nonmember_trials)
    tpr_lower, tpr_upper = _wilson_interval(member_successes, member_trials)
    fpr_lower, fpr_upper = _wilson_interval(nonmember_successes, nonmember_trials)

    event_positive_point = _epsilon_candidate(
        numerator_rate=tpr,
        denominator_rate=fpr,
        delta=delta,
        denominator_floor=0.5 / max(1, nonmember_trials),
    )
    event_positive_lower = _epsilon_candidate(
        numerator_rate=tpr_lower,
        denominator_rate=fpr_upper,
        delta=delta,
        denominator_floor=0.5 / max(1, nonmember_trials),
    )
    event_positive_upper = _epsilon_candidate(
        numerator_rate=tpr_upper,
        denominator_rate=fpr_lower,
        delta=delta,
        denominator_floor=0.5 / max(1, nonmember_trials),
    )

    return {
        "point": event_positive_point,
        "lower": event_positive_lower,
        "upper": event_positive_upper,
        "tpr": tpr,
        "fpr": fpr,
        "member_favoring": tpr > fpr,
    }


def _evaluate_legacy_threshold(
    *,
    member_successes: int,
    member_trials: int,
    nonmember_successes: int,
    nonmember_trials: int,
    delta: float,
) -> dict[str, float | str]:
    tpr = member_successes / max(1, member_trials)
    fpr = nonmember_successes / max(1, nonmember_trials)
    tpr_lower, tpr_upper = _wilson_interval(member_successes, member_trials)
    fpr_lower, fpr_upper = _wilson_interval(nonmember_successes, nonmember_trials)

    event_positive_point = _epsilon_candidate(
        numerator_rate=tpr,
        denominator_rate=fpr,
        delta=delta,
        denominator_floor=0.5 / max(1, nonmember_trials),
    )
    event_positive_lower = _epsilon_candidate(
        numerator_rate=tpr_lower,
        denominator_rate=fpr_upper,
        delta=delta,
        denominator_floor=0.5 / max(1, nonmember_trials),
    )
    event_positive_upper = _epsilon_candidate(
        numerator_rate=tpr_upper,
        denominator_rate=fpr_lower,
        delta=delta,
        denominator_floor=0.5 / max(1, nonmember_trials),
    )

    event_negative_point = _epsilon_candidate(
        numerator_rate=1.0 - fpr,
        denominator_rate=1.0 - tpr,
        delta=delta,
        denominator_floor=0.5 / max(1, member_trials),
    )
    event_negative_lower = _epsilon_candidate(
        numerator_rate=1.0 - fpr_upper,
        denominator_rate=1.0 - tpr_lower,
        delta=delta,
        denominator_floor=0.5 / max(1, member_trials),
    )
    event_negative_upper = _epsilon_candidate(
        numerator_rate=1.0 - fpr_lower,
        denominator_rate=1.0 - tpr_upper,
        delta=delta,
        denominator_floor=0.5 / max(1, member_trials),
    )

    if event_positive_point >= event_negative_point:
        return {
            "point": event_positive_point,
            "lower": event_positive_lower,
            "upper": event_positive_upper,
            "event": "score>=threshold",
            "tpr": tpr,
            "fpr": fpr,
        }
    return {
        "point": event_negative_point,
        "lower": event_negative_lower,
        "upper": event_negative_upper,
        "event": "score<threshold",
        "tpr": tpr,
        "fpr": fpr,
    }


def _epsilon_candidate(
    *,
    numerator_rate: float,
    denominator_rate: float,
    delta: float,
    denominator_floor: float,
) -> float:
    numerator = numerator_rate - delta
    if numerator <= 0.0:
        return 0.0
    denominator = max(denominator_rate, denominator_floor)
    if denominator <= 0.0:
        return 0.0
    return max(0.0, math.log(numerator / denominator))


def _wilson_interval(successes: int, trials: int, z: float = 1.96) -> tuple[float, float]:
    if trials <= 0:
        return 0.0, 0.0
    phat = successes / trials
    denom = 1.0 + (z**2 / trials)
    center = (phat + (z**2 / (2.0 * trials))) / denom
    margin = (
        z
        * math.sqrt((phat * (1.0 - phat) / trials) + (z**2 / (4.0 * trials**2)))
        / denom
    )
    return max(0.0, center - margin), min(1.0, center + margin)


def _is_tiny_tail_event(
    *,
    member_event_fraction: float,
    nonmember_event_fraction: float,
    member_event_count: int,
    nonmember_event_count: int,
    member_trials: int,
    nonmember_trials: int,
    fraction_threshold: float,
    min_event_count: int,
) -> bool:
    max_fraction = max(member_event_fraction, nonmember_event_fraction)
    return (
        max_fraction <= fraction_threshold
        or member_event_count < min_event_count
        or nonmember_event_count < min_event_count
        or member_event_count == member_trials
        or nonmember_event_count == nonmember_trials
    )


def _compose_warning_message(
    *,
    report_confidence_supported_lower_bound: bool,
    point_estimate: float,
    conservative_estimate: float,
    selected_event_is_tiny_tail: bool,
    existing_warning: str | None,
) -> str | None:
    warnings: list[str] = []
    if existing_warning:
        warnings.append(existing_warning)
    if selected_event_is_tiny_tail:
        warnings.append("Selected threshold lies in a sparse tail region.")
    if report_confidence_supported_lower_bound and conservative_estimate < point_estimate:
        warnings.append(
            "Reported empirical lower bound uses the confidence-supported lower side rather than the optimistic point estimate."
        )
    if not warnings:
        return None
    return " ".join(warnings)
""",
    "src/dp_audit_tightness/privacy/gdp_estimation.py": """\"\"\"GDP (Gaussian Differential Privacy) estimation for tighter lower bounds.

Instead of the threshold sweep + Wilson CI approach in empirical.py, this module:
1. Computes the empirical AUC between member and non-member scores
2. Estimates the GDP mu parameter from AUC: mu = sqrt(2) * Phi^{-1}(AUC)
3. Converts mu to (epsilon, delta)-DP via the standard GDP-to-DP conversion
4. Uses bootstrap confidence intervals on mu for conservative bounds

The AUC-based estimator avoids threshold selection bias (picking the "best"
threshold inflates mu by cherry-picking noise). Under GDP(mu), the ROC curve
is fully determined by mu, and AUC = Phi(mu / sqrt(2)), giving a clean
single-parameter estimate from the entire score distribution.

Reference: Dong, Roth & Su "Gaussian Differential Privacy" (2019)
Reference: Balle et al. "Hypothesis Testing Interpretations..." (2020)
\"\"\"
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Sequence

import numpy as np


@dataclass
class GDPEstimate:
    \"\"\"Result of GDP-based privacy estimation.\"\"\"
    epsilon_lower: float
    mu_lower: float
    mu_point: float
    mu_ci_lower: float
    delta: float
    method: str
    auc: float = 0.0
    n_member: int = 0
    n_nonmember: int = 0
    warning: str | None = None


def estimate_epsilon_gdp(
    member_scores: Sequence[float],
    nonmember_scores: Sequence[float],
    delta: float,
    n_bootstrap: int = 1000,
    ci_level: float = 0.95,
) -> GDPEstimate:
    \"\"\"Estimate epsilon lower bound using GDP framework.

    Computes the empirical AUC between member and non-member scores,
    estimates mu via mu = sqrt(2) * Phi^{-1}(AUC), then converts to epsilon.
    Uses bootstrap for conservative confidence intervals.

    Parameters
    ----------
    member_scores : array-like
        Scores for members (higher = more likely member).
    nonmember_scores : array-like
        Scores for non-members.
    delta : float
        Privacy parameter delta.
    n_bootstrap : int
        Number of bootstrap resamples for CI.
    ci_level : float
        Confidence level for bootstrap CI.
    \"\"\"
    m = np.array(member_scores, dtype=np.float64)
    n = np.array(nonmember_scores, dtype=np.float64)

    if len(m) < 5 or len(n) < 5:
        return GDPEstimate(
            epsilon_lower=0.0, mu_lower=0.0, mu_point=0.0,
            mu_ci_lower=0.0, delta=delta, method="gdp_auc",
            n_member=len(m), n_nonmember=len(n),
            warning="Too few samples")

    # Point estimate of mu from AUC
    auc = _compute_auc(m, n)
    mu_point = _auc_to_mu(auc)

    # Bootstrap CI on mu
    rng = np.random.RandomState(42)
    mu_boots = []
    for _ in range(n_bootstrap):
        m_boot = rng.choice(m, size=len(m), replace=True)
        n_boot = rng.choice(n, size=len(n), replace=True)
        auc_b = _compute_auc(m_boot, n_boot)
        mu_boots.append(_auc_to_mu(auc_b))

    alpha = 1.0 - ci_level
    mu_ci_lower = float(np.percentile(mu_boots, 100 * alpha / 2))
    mu_ci_lower = max(0.0, mu_ci_lower)

    # Convert mu to epsilon (use conservative CI lower bound)
    eps_conservative = _mu_to_epsilon(mu_ci_lower, delta) if mu_ci_lower > 0 else 0.0

    return GDPEstimate(
        epsilon_lower=max(0.0, eps_conservative),
        mu_lower=mu_ci_lower,
        mu_point=mu_point,
        mu_ci_lower=mu_ci_lower,
        delta=delta,
        method="gdp_auc_bootstrap",
        auc=auc,
        n_member=len(m),
        n_nonmember=len(n),
    )


def _compute_auc(member: np.ndarray, nonmember: np.ndarray) -> float:
    \"\"\"Compute AUC via the Mann-Whitney U statistic.

    AUC = P(member_score > nonmember_score) + 0.5 * P(member_score == nonmember_score)
    This is equivalent to U / (n_m * n_n) where U is the Mann-Whitney U statistic.
    \"\"\"
    n_m = len(member)
    n_n = len(nonmember)
    # Vectorized: count how many nonmember scores each member score beats
    # For large arrays, use sorted-rank approach for O(n log n)
    all_scores = np.concatenate([member, nonmember])
    labels = np.concatenate([np.ones(n_m), np.zeros(n_n)])

    # Sort by score descending
    order = np.argsort(-all_scores)
    sorted_labels = labels[order]

    # AUC = (sum of ranks of positives - n_m*(n_m+1)/2) / (n_m * n_n)
    ranks = np.arange(1, len(all_scores) + 1, dtype=np.float64)
    # Handle ties: assign average rank
    sorted_scores = all_scores[order]
    i = 0
    while i < len(sorted_scores):
        j = i + 1
        while j < len(sorted_scores) and sorted_scores[j] == sorted_scores[i]:
            j += 1
        if j > i + 1:
            avg_rank = np.mean(ranks[i:j])
            ranks[i:j] = avg_rank
        i = j

    pos_rank_sum = ranks[sorted_labels == 1].sum()
    auc = (pos_rank_sum - n_m * (n_m + 1) / 2) / (n_m * n_n)
    return float(auc)


def _auc_to_mu(auc: float) -> float:
    \"\"\"Convert AUC to GDP mu parameter.

    Under GDP(mu): AUC = Phi(mu / sqrt(2))
    So mu = sqrt(2) * Phi^{-1}(AUC)

    Returns 0.0 if AUC <= 0.5 (no discrimination).
    \"\"\"
    from scipy.stats import norm

    if auc <= 0.5:
        return 0.0
    # Clip to avoid Phi^{-1}(1) = inf
    auc_clipped = min(auc, 1.0 - 1e-10)
    mu = math.sqrt(2) * norm.ppf(auc_clipped)
    return max(0.0, mu)


def _mu_to_epsilon(mu: float, delta: float) -> float:
    \"\"\"Convert GDP mu to (epsilon, delta)-DP via binary search.\"\"\"
    if mu <= 0:
        return 0.0

    lo, hi = 0.0, mu * mu + 10.0
    for _ in range(100):
        mid = (lo + hi) / 2
        d = _gdp_delta(mid, mu)
        if d > delta:
            lo = mid
        else:
            hi = mid
        if hi - lo < 1e-8:
            break
    return lo


def _gdp_delta(epsilon: float, mu: float) -> float:
    \"\"\"Compute delta for given epsilon under GDP(mu).\"\"\"
    from scipy.stats import norm
    if mu <= 0:
        return 0.0
    term1 = norm.cdf(-epsilon / mu + mu / 2)
    term2 = math.exp(epsilon) * norm.cdf(-epsilon / mu - mu / 2)
    return max(0.0, term1 - term2)
""",
    "src/dp_audit_tightness/privacy/pld_accounting.py": """\"\"\"Privacy Loss Distribution (PLD) accounting for tighter epsilon upper bounds.

The RDP accountant (used by Opacus) produces valid but *loose* upper bounds on
epsilon.  PLD-based accounting computes the privacy loss numerically via
discretised distributions and yields a *tighter* upper bound for the same
(noise_multiplier, sampling_rate, num_steps, delta) tuple.

This module provides two back-ends:

1. **dp_accounting** (Google's library) — preferred; uses the analytical
   Gaussian mechanism PLD with Poisson subsampling.
2. **Fallback analytical** — a closed-form Gaussian-mechanism bound via the
   inverse-CDF approach (Balle et al., 2020).  Less tight than PLD but still
   tighter than RDP for most regimes, and has zero extra dependencies.

The public entry point is :func:`compute_epsilon_pld`.
\"\"\"

from __future__ import annotations

import math
from typing import Any

_VALUE_DISCRETIZATION_INTERVAL = 1e-4


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def compute_epsilon_pld(
    *,
    noise_multiplier: float,
    sampling_rate: float,
    num_steps: int,
    delta: float,
    backend: str = "auto",
) -> dict[str, Any]:
    \"\"\"Compute a tight epsilon upper bound using PLD-based accounting.

    Parameters
    ----------
    noise_multiplier : float
        Ratio of noise standard deviation to the clipping norm (sigma / C).
    sampling_rate : float
        Probability that each example is included in a batch (batch_size / n).
    num_steps : int
        Total number of gradient steps (epochs * steps_per_epoch).
    delta : float
        Target delta for (epsilon, delta)-DP.
    backend : str
        ``"google"`` forces the dp_accounting library, ``"analytical"`` uses
        the closed-form bound, ``"auto"`` (default) tries google first.

    Returns
    -------
    dict with keys:
        epsilon_pld : float   — the computed epsilon upper bound
        backend_used : str    — which backend produced the result
        num_steps : int       — echoed back for record-keeping
    \"\"\"
    if noise_multiplier <= 0:
        raise ValueError("noise_multiplier must be positive.")
    if not (0 < sampling_rate <= 1):
        raise ValueError("sampling_rate must be in (0, 1].")
    if num_steps <= 0:
        raise ValueError("num_steps must be positive.")
    if delta <= 0:
        raise ValueError("delta must be positive.")

    if backend in ("auto", "google"):
        try:
            eps = _compute_with_dp_accounting(
                noise_multiplier=noise_multiplier,
                sampling_rate=sampling_rate,
                num_steps=num_steps,
                delta=delta,
            )
            return {"epsilon_pld": round(eps, 6), "backend_used": "google_dp_accounting", "num_steps": num_steps}
        except (ImportError, AttributeError, Exception) as e:
            if backend == "google":
                raise
            # Fall through to analytical

    # Analytical fallback (always available)
    eps = _compute_analytical_gaussian(
        noise_multiplier=noise_multiplier,
        sampling_rate=sampling_rate,
        num_steps=num_steps,
        delta=delta,
    )
    return {"epsilon_pld": round(eps, 6), "backend_used": "analytical_gaussian", "num_steps": num_steps}


# ---------------------------------------------------------------------------
# Backend 1: Google dp_accounting (PLD-based, tightest)
# ---------------------------------------------------------------------------

def _compute_with_dp_accounting(
    *,
    noise_multiplier: float,
    sampling_rate: float,
    num_steps: int,
    delta: float,
) -> float:
    \"\"\"Use google/dp-accounting's PLD accountant for tight composition.\"\"\"
    from dp_accounting.pld import privacy_loss_distribution as pld_lib

    # Build the PLD for a single step of subsampled Gaussian mechanism.
    kwargs = {
        "standard_deviation": noise_multiplier,
        "pessimistic_estimate": True,
        "sampling_prob": sampling_rate,
        "use_connect_dots": True,
        "value_discretization_interval": _VALUE_DISCRETIZATION_INTERVAL,
    }
    try:
        pld_single = pld_lib.from_gaussian_mechanism(
            sensitivity=1.0,
            **kwargs,
        )
    except TypeError:
        # Older dp-accounting builds used a plural sensitivities argument.
        pld_single = pld_lib.from_gaussian_mechanism(
            sensitivities=[1.0],
            **kwargs,
        )

    # Self-compose for num_steps applications.
    pld_composed = pld_single.self_compose(num_steps)

    # Extract epsilon at the target delta.
    epsilon = pld_composed.get_epsilon_for_delta(delta)
    return float(epsilon)


# ---------------------------------------------------------------------------
# Backend 2: GDP (Gaussian Differential Privacy) — Dong, Roth & Su (2019)
# ---------------------------------------------------------------------------

def _compute_analytical_gaussian(
    *,
    noise_multiplier: float,
    sampling_rate: float,
    num_steps: int,
    delta: float,
) -> float:
    \"\"\"Tight closed-form bound via the GDP (Gaussian DP) CLT framework.

    The Central Limit Theorem for DP (Dong, Roth & Su, 2019; Balle et al.,
    2020) shows that T-fold composition of the subsampled Gaussian mechanism
    converges to mu-GDP with:

        mu = q * sqrt(T) / sigma

    where q is the sampling rate and sigma is the noise multiplier.  We then
    convert mu-GDP to (epsilon, delta)-DP by inverting the trade-off function.

    This gives bounds that are *much* tighter than advanced composition and
    comparable to (though slightly looser than) exact PLD numerical accounting.
    \"\"\"
    sigma = noise_multiplier

    # GDP parameter: mu = q * sqrt(T) / sigma
    mu = sampling_rate * math.sqrt(num_steps) / sigma

    # Convert mu-GDP to (epsilon, delta)-DP by inverting:
    #   delta(eps) = Phi(-eps/mu + mu/2) - exp(eps) * Phi(-eps/mu - mu/2)
    epsilon = _gdp_to_eps_delta(mu, delta)
    return float(epsilon)


def _norm_cdf(x: float) -> float:
    \"\"\"Standard normal CDF.\"\"\"
    return 0.5 * math.erfc(-x / math.sqrt(2.0))


def _gdp_to_eps_delta(mu: float, target_delta: float) -> float:
    \"\"\"Convert mu-GDP to epsilon given a target delta.

    Inverts: delta(eps) = Phi(-eps/mu + mu/2) - exp(eps) * Phi(-eps/mu - mu/2)
    via binary search.
    \"\"\"
    if mu <= 0:
        return 0.0

    def _delta_of_eps(eps: float) -> float:
        t1 = _norm_cdf(-eps / mu + mu / 2.0)
        t2 = math.exp(eps) * _norm_cdf(-eps / mu - mu / 2.0)
        return t1 - t2

    # Binary search for the smallest epsilon where delta(eps) <= target_delta
    lo, hi = 0.0, mu * mu + 2.0 * mu * math.sqrt(math.log(1.0 / target_delta))

    # Safety check: if delta(0) is already <= target, epsilon = 0
    if _delta_of_eps(0.0) <= target_delta:
        return 0.0

    for _ in range(200):
        mid = (lo + hi) / 2.0
        if _delta_of_eps(mid) > target_delta:
            lo = mid
        else:
            hi = mid
        if hi - lo < 1e-10:
            break

    return hi
""",
    "src/dp_audit_tightness/reporting/__init__.py": """\"\"\"Aggregation and reporting helpers.\"\"\"

""",
    "src/dp_audit_tightness/reporting/canary_estimator_diagnostics.py": """from __future__ import annotations

import csv
from dataclasses import dataclass
import math
from pathlib import Path
import statistics
from typing import Any

from dp_audit_tightness.evaluation.metrics import compute_privacy_tightness_metrics
from dp_audit_tightness.privacy.empirical import estimate_empirical_lower_bound
from dp_audit_tightness.utils.paths import ensure_directory
from dp_audit_tightness.utils.results import load_audit_result, load_json, save_json


OBJECTIVE_ORDER = ["saved_current", "recomputed_unconstrained", "recomputed_canary_favoring"]


@dataclass(slots=True)
class ScoreSummary:
    count: int
    mean: float
    std: float
    min: float
    p05: float
    p25: float
    median: float
    p75: float
    p95: float
    max: float


def generate_canary_estimator_diagnostics(
    *,
    canary_results_dir: str | Path,
    canary_artifacts_dir: str | Path,
    output_dir: str | Path,
) -> dict[str, Path]:
    runs = _load_canary_runs(
        canary_results_dir=Path(canary_results_dir),
        canary_artifacts_dir=Path(canary_artifacts_dir),
    )
    comparison_rows: list[dict[str, Any]] = []
    score_summary_rows: list[dict[str, Any]] = []

    for run in runs:
        score_summary_rows.extend(_build_score_summary_rows(run))
        comparison_rows.extend(_build_comparison_rows(run))

    comparison_rows.sort(key=lambda row: (row["seed"], _objective_rank(row["objective_mode"])))
    score_summary_rows.sort(key=lambda row: (row["seed"], row["group"]))

    output_root = ensure_directory(output_dir)
    comparison_csv = _write_csv(output_root / "comparison_table.csv", comparison_rows)
    comparison_json = save_json(output_root / "comparison_table.json", {"rows": comparison_rows})
    score_csv = _write_csv(output_root / "score_summary_table.csv", score_summary_rows)
    score_json = save_json(output_root / "score_summary_table.json", {"rows": score_summary_rows})
    summary_path = output_root / "summary.md"
    summary_path.write_text(
        _build_summary(comparison_rows=comparison_rows, score_summary_rows=score_summary_rows),
        encoding="utf-8",
    )

    return {
        "comparison_csv": comparison_csv,
        "comparison_json": comparison_json,
        "score_summary_csv": score_csv,
        "score_summary_json": score_json,
        "summary_markdown": summary_path,
    }


def _load_canary_runs(*, canary_results_dir: Path, canary_artifacts_dir: Path) -> list[dict[str, Any]]:
    runs: list[dict[str, Any]] = []
    for audit_path in sorted(canary_results_dir.glob("canary_audit_seed*.json")):
        record = load_audit_result(audit_path)
        debug_artifact_path = canary_artifacts_dir / f"{record.audit_run_id}_canary_debug.json"
        debug_payload = load_json(debug_artifact_path)
        runs.append(
            {
                "audit_path": audit_path,
                "record": record,
                "debug_artifact_path": debug_artifact_path,
                "debug_payload": debug_payload,
                "member_scores": [float(score) for score in debug_payload["member_scores"]],
                "nonmember_scores": [float(score) for score in debug_payload["nonmember_scores"]],
            }
        )
    return sorted(runs, key=lambda run: run["record"].training_seed)


def _build_comparison_rows(run: dict[str, Any]) -> list[dict[str, Any]]:
    record = run["record"]
    member_scores = run["member_scores"]
    nonmember_scores = run["nonmember_scores"]
    saved_row = _saved_current_row(run)
    unconstrained = estimate_empirical_lower_bound(
        member_scores=member_scores,
        nonmember_scores=nonmember_scores,
        delta=record.delta,
    )
    constrained = estimate_empirical_lower_bound(
        member_scores=member_scores,
        nonmember_scores=nonmember_scores,
        delta=record.delta,
        align_event_to_score_direction=True,
        require_member_favoring=True,
    )
    return [
        saved_row,
        _estimate_row(run, "recomputed_unconstrained", unconstrained),
        _estimate_row(run, "recomputed_canary_favoring", constrained),
    ]


def _saved_current_row(run: dict[str, Any]) -> dict[str, Any]:
    record = run["record"]
    metadata = record.audit_metadata
    selected_event = metadata.get("selected_event")
    selected_tpr = _optional_float(metadata.get("selected_tpr"))
    selected_fpr = _optional_float(metadata.get("selected_fpr"))
    canary_event_fraction, control_event_fraction = _event_fractions_from_selection(
        selected_event=selected_event,
        selected_tpr=selected_tpr,
        selected_fpr=selected_fpr,
    )
    canary_favoring = _is_canary_favoring(
        canary_event_fraction=canary_event_fraction,
        control_event_fraction=control_event_fraction,
    )
    warning_flags = _warning_flags(
        epsilon_upper_theory=record.epsilon_upper_theory,
        epsilon_lower_empirical=record.epsilon_lower_empirical,
        canary_favoring=canary_favoring,
        canary_event_fraction=canary_event_fraction,
        control_event_fraction=control_event_fraction,
        ci_lower=record.empirical_ci_lower,
    )
    return {
        "seed": record.training_seed,
        "objective_mode": "saved_current",
        "epsilon_upper_theory": record.epsilon_upper_theory,
        "epsilon_lower_empirical": record.epsilon_lower_empirical,
        "privacy_loss_gap": record.privacy_loss_gap,
        "tightness_ratio": record.tightness_ratio,
        "selected_threshold": metadata.get("selected_threshold"),
        "selected_event": selected_event,
        "canary_event_fraction": canary_event_fraction,
        "control_event_fraction": control_event_fraction,
        "canary_favoring": canary_favoring,
        "ci_lower": record.empirical_ci_lower,
        "ci_upper": record.empirical_ci_upper,
        "saturation_detected": record.saturation_detected,
        "tiny_tail_event": _is_tiny_tail_event(
            canary_event_fraction=canary_event_fraction,
            control_event_fraction=control_event_fraction,
        ),
        "objective_numerator_group": "canary"
        if selected_event == "score>=threshold"
        else "control",
        "objective_denominator_group": "control"
        if selected_event == "score>=threshold"
        else "canary",
        "warning_flags": warning_flags,
        "matches_saved_selection": True,
        "same_as_canary_favoring_selection": None,
        "audit_path": str(run["audit_path"]),
        "debug_artifact_path": str(run["debug_artifact_path"]),
    }


def _estimate_row(run: dict[str, Any], objective_mode: str, estimate) -> dict[str, Any]:
    record = run["record"]
    tightness = compute_privacy_tightness_metrics(
        epsilon_upper_theory=record.epsilon_upper_theory,
        epsilon_lower_empirical=estimate.epsilon_lower_empirical,
    )
    warning_flags = _warning_flags(
        epsilon_upper_theory=record.epsilon_upper_theory,
        epsilon_lower_empirical=estimate.epsilon_lower_empirical,
        canary_favoring=estimate.member_favoring,
        canary_event_fraction=estimate.member_event_fraction,
        control_event_fraction=estimate.nonmember_event_fraction,
        ci_lower=estimate.empirical_ci_lower,
        explicit_warning=estimate.warning_message,
    )
    saved_event = record.audit_metadata.get("selected_event")
    saved_threshold = _optional_float(record.audit_metadata.get("selected_threshold"))
    matches_saved_selection = (
        saved_event == estimate.selected_event
        and saved_threshold is not None
        and estimate.selected_threshold is not None
        and abs(saved_threshold - estimate.selected_threshold) <= 1e-12
    )
    return {
        "seed": record.training_seed,
        "objective_mode": objective_mode,
        "epsilon_upper_theory": record.epsilon_upper_theory,
        "epsilon_lower_empirical": estimate.epsilon_lower_empirical,
        "privacy_loss_gap": tightness.privacy_loss_gap,
        "tightness_ratio": tightness.tightness_ratio,
        "selected_threshold": estimate.selected_threshold,
        "selected_event": estimate.selected_event,
        "canary_event_fraction": estimate.member_event_fraction,
        "control_event_fraction": estimate.nonmember_event_fraction,
        "canary_favoring": estimate.member_favoring,
        "ci_lower": estimate.empirical_ci_lower,
        "ci_upper": estimate.empirical_ci_upper,
        "saturation_detected": record.saturation_detected,
        "tiny_tail_event": _is_tiny_tail_event(
            canary_event_fraction=estimate.member_event_fraction,
            control_event_fraction=estimate.nonmember_event_fraction,
        ),
        "objective_numerator_group": "canary"
        if estimate.selected_event == "score>=threshold"
        else "control",
        "objective_denominator_group": "control"
        if estimate.selected_event == "score>=threshold"
        else "canary",
        "warning_flags": warning_flags,
        "matches_saved_selection": matches_saved_selection,
        "same_as_canary_favoring_selection": None,
        "audit_path": str(run["audit_path"]),
        "debug_artifact_path": str(run["debug_artifact_path"]),
    }


def _build_score_summary_rows(run: dict[str, Any]) -> list[dict[str, Any]]:
    record = run["record"]
    canary_summary = _summarize_scores(run["member_scores"])
    control_summary = _summarize_scores(run["nonmember_scores"])
    return [
        _score_summary_row(record.training_seed, "canary", canary_summary),
        _score_summary_row(record.training_seed, "control", control_summary),
    ]


def _score_summary_row(seed: int, group: str, summary: ScoreSummary) -> dict[str, Any]:
    return {
        "seed": seed,
        "group": group,
        "count": summary.count,
        "mean": summary.mean,
        "std": summary.std,
        "min": summary.min,
        "p05": summary.p05,
        "p25": summary.p25,
        "median": summary.median,
        "p75": summary.p75,
        "p95": summary.p95,
        "max": summary.max,
    }


def _summarize_scores(values: list[float]) -> ScoreSummary:
    ordered = sorted(values)
    if not ordered:
        raise ValueError("Cannot summarize an empty score list.")
    return ScoreSummary(
        count=len(ordered),
        mean=float(statistics.fmean(ordered)),
        std=float(statistics.stdev(ordered)) if len(ordered) > 1 else 0.0,
        min=float(ordered[0]),
        p05=_percentile(ordered, 0.05),
        p25=_percentile(ordered, 0.25),
        median=_percentile(ordered, 0.50),
        p75=_percentile(ordered, 0.75),
        p95=_percentile(ordered, 0.95),
        max=float(ordered[-1]),
    )


def _percentile(values: list[float], q: float) -> float:
    if len(values) == 1:
        return float(values[0])
    position = (len(values) - 1) * q
    lower_index = math.floor(position)
    upper_index = math.ceil(position)
    if lower_index == upper_index:
        return float(values[lower_index])
    weight = position - lower_index
    return float(values[lower_index] * (1.0 - weight) + values[upper_index] * weight)


def _event_fractions_from_selection(
    *,
    selected_event: str | None,
    selected_tpr: float | None,
    selected_fpr: float | None,
) -> tuple[float | None, float | None]:
    if selected_event is None or selected_tpr is None or selected_fpr is None:
        return None, None
    if selected_event == "score>=threshold":
        return selected_tpr, selected_fpr
    if selected_event == "score<threshold":
        return 1.0 - selected_tpr, 1.0 - selected_fpr
    return None, None


def _is_canary_favoring(
    *,
    canary_event_fraction: float | None,
    control_event_fraction: float | None,
) -> bool | None:
    if canary_event_fraction is None or control_event_fraction is None:
        return None
    return canary_event_fraction > control_event_fraction


def _is_tiny_tail_event(
    *,
    canary_event_fraction: float | None,
    control_event_fraction: float | None,
) -> bool | None:
    if canary_event_fraction is None or control_event_fraction is None:
        return None
    return max(canary_event_fraction, control_event_fraction) <= 0.125


def _warning_flags(
    *,
    epsilon_upper_theory: float,
    epsilon_lower_empirical: float,
    canary_favoring: bool | None,
    canary_event_fraction: float | None,
    control_event_fraction: float | None,
    ci_lower: float | None,
    explicit_warning: str | None = None,
) -> str | None:
    warnings: list[str] = []
    if epsilon_lower_empirical > epsilon_upper_theory:
        warnings.append("empirical_lower_exceeds_theoretical_upper")
    if canary_favoring is False:
        warnings.append("selected_event_not_canary_favoring")
    if _is_tiny_tail_event(
        canary_event_fraction=canary_event_fraction,
        control_event_fraction=control_event_fraction,
    ):
        warnings.append("tiny_tail_event")
    if ci_lower is not None and ci_lower <= 0.0:
        warnings.append("ci_lower_is_zero")
    if explicit_warning:
        warnings.append(explicit_warning)
    return "; ".join(warnings) if warnings else None


def _build_summary(
    *,
    comparison_rows: list[dict[str, Any]],
    score_summary_rows: list[dict[str, Any]],
) -> str:
    saved_rows = [row for row in comparison_rows if row["objective_mode"] == "saved_current"]
    unconstrained_rows = [
        row for row in comparison_rows if row["objective_mode"] == "recomputed_unconstrained"
    ]
    constrained_rows = [
        row for row in comparison_rows if row["objective_mode"] == "recomputed_canary_favoring"
    ]
    saved_pathologies = sum(
        row["epsilon_lower_empirical"] > row["epsilon_upper_theory"] for row in saved_rows
    )
    constrained_pathologies = sum(
        row["epsilon_lower_empirical"] > row["epsilon_upper_theory"] for row in constrained_rows
    )
    non_favoring_saved = sum(row["canary_favoring"] is False for row in saved_rows)
    tiny_tail_saved = sum(bool(row["tiny_tail_event"]) for row in saved_rows)
    same_after_constraint = sum(
        (
            saved["selected_event"] == constrained["selected_event"]
            and saved["selected_threshold"] is not None
            and constrained["selected_threshold"] is not None
            and abs(float(saved["selected_threshold"]) - float(constrained["selected_threshold"])) <= 1e-12
        )
        for saved, constrained in zip(saved_rows, constrained_rows, strict=False)
    )
    shrinkages = [
        unconstrained["epsilon_lower_empirical"] - constrained["epsilon_lower_empirical"]
        for unconstrained, constrained in zip(unconstrained_rows, constrained_rows, strict=False)
    ]
    canary_rows = [row for row in score_summary_rows if row["group"] == "canary"]
    control_rows = [row for row in score_summary_rows if row["group"] == "control"]
    mean_gap = statistics.fmean(
        canary_row["mean"] - control_row["mean"]
        for canary_row, control_row in zip(canary_rows, control_rows, strict=False)
    )

    return "\n".join(
        [
            "# Canary Estimator Diagnostic Summary",
            "",
            "## Trace",
            "",
            "- Candidate thresholds are the pooled unique canary/control scores from the saved canary debug artifact.",
            "- `run_audit_canary.py` currently calls `estimate_empirical_lower_bound(...)` with the legacy unconstrained objective.",
            "- In repeated-run canary mode, per-seed canary and control scores are concatenated across the repeated audit seeds before threshold selection.",
            "- The empirical estimator converts event frequencies into an empirical lower-bound point estimate using `log((p_canary - delta) / max(p_control, floor))` for `score>=threshold`, and it also has a complement-event branch for `score<threshold`.",
            "- Confidence intervals are Wilson intervals on the event frequencies, but the saved `epsilon_lower_empirical` is the point estimate, not the conservative lower confidence bound.",
            "- Saturation is inherited from the saved canary audit records and is not recomputed here.",
            "",
            "## Direct Answers",
            "",
            "- Is the canary pathology caused by wrong event-direction or complement handling?",
            "  Not in these post-seedfix runs. All three saved canary runs selected `score>=threshold`, so the current pathological values are not coming from the complement branch in the selected operating point. The complement logic still exists in the estimator, but it is not the active cause here.",
            "- Is the estimator selecting non-canary-favoring events?",
            f"  No. {non_favoring_saved}/3 saved runs selected non-canary-favoring events.",
            "- Are the pathological lower bounds driven by tiny tails / instability?",
            f"  Yes. {tiny_tail_saved}/3 saved runs use tiny-tail operating points under the 12.5% heuristic, and all 3/3 have `ci_lower = 0.0`, indicating that the apparent signal is not stable enough to support a confident lower bound.",
            "- Does constraining to canary-favoring events remove the `epsilon_lower_empirical > epsilon_upper_theory` cases?",
            f"  No. Saved current has {saved_pathologies}/3 warning cases and the canary-favoring-only recomputation still has {constrained_pathologies}/3. The constrained sweep matches the unconstrained optimum in {same_after_constraint}/3 runs.",
            "- Is the remaining canary signal still strong after such a correction, or does it shrink sharply like passive did?",
            f"  It does not shrink here. Mean shrinkage from unconstrained to canary-favoring-only is {statistics.fmean(shrinkages):.6f}, which is effectively zero on this benchmark.",
            "",
            "## Interpretation",
            "",
            f"- The raw score means do not show a strong stable separation: mean(canary_mean - control_mean) = {mean_gap:.6f}.",
            "- The problem is that the selected operating points are in very sparse tails, often with only one or a few control-side hits.",
            "- Because the saved `epsilon_lower_empirical` uses the optimistic point estimate rather than a conservative one-sided bound, the estimator can report a large empirical lower bound even when the confidence interval lower bound is zero.",
            "- These `epsilon_lower_empirical > epsilon_upper_theory` cases are diagnostic warnings about the estimator path, not evidence that the theoretical upper bound is violated.",
            "",
            "## Recommendation",
            "",
            "The next production fix should prioritize revising the canary empirical lower-bound calibration so that the returned `epsilon_lower_empirical` is conservative, CI-aware, and robust to tiny-tail thresholds. Constraining to canary-favoring events is still a reasonable defensive hardening step, but it will not resolve the current pathology by itself on this benchmark.",
            "",
        ]
    )


def _write_csv(path: Path, rows: list[dict[str, Any]]) -> Path:
    ensure_directory(path.parent)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    return path


def _optional_float(value: Any) -> float | None:
    if value is None:
        return None
    return float(value)


def _objective_rank(mode: str) -> int:
    try:
        return OBJECTIVE_ORDER.index(mode)
    except ValueError:
        return len(OBJECTIVE_ORDER)
""",
    "src/dp_audit_tightness/reporting/passive_direction_diagnostics.py": """from __future__ import annotations

from dataclasses import dataclass
import csv
import math
from pathlib import Path
import statistics
from typing import Any

from dp_audit_tightness.evaluation.metrics import compute_privacy_tightness_metrics
from dp_audit_tightness.privacy.empirical import _epsilon_candidate, _wilson_interval
from dp_audit_tightness.utils.paths import ensure_directory
from dp_audit_tightness.utils.results import PASSIVE_AUDIT_REGIME, load_audit_result, load_json, save_json


VARIANT_ORDER = [
    "max_probability_passive",
    "negative_loss_passive",
    "logit_margin_passive",
    "score_fusion_passive",
]

DIRECTION_ORDER = ["score>=threshold", "score<threshold"]
SELECTION_MODE_ORDER = ["unconstrained", "member_favoring_only"]


@dataclass(slots=True)
class PassiveRunData:
    audit_path: Path
    debug_artifact_path: Path
    audit_run_id: str
    training_run_id: str
    auditor_variant: str
    score_type: str
    training_seed: int
    epsilon_upper_theory: float
    epsilon_lower_empirical: float
    empirical_ci_lower: float | None
    empirical_ci_upper: float | None
    privacy_loss_gap: float
    tightness_ratio: float | None
    saturation_detected: bool
    selected_threshold: float | None
    selected_event: str | None
    member_scores: list[float]
    nonmember_scores: list[float]


@dataclass(slots=True)
class DirectionDiagnosticCandidate:
    direction: str
    selection_mode: str
    threshold: float | None
    member_event_fraction: float | None
    nonmember_event_fraction: float | None
    tpr: float | None
    fpr: float | None
    epsilon_lower_empirical: float | None
    empirical_ci_lower: float | None
    empirical_ci_upper: float | None
    member_favoring: bool
    objective_numerator_group: str | None
    objective_denominator_group: str | None
    note: str | None = None


@dataclass(slots=True)
class ScoreSummary:
    count: int
    mean: float
    std: float
    min: float
    p05: float
    p25: float
    median: float
    p75: float
    p95: float
    max: float


def generate_passive_direction_diagnostics(
    *,
    passive_results_dir: str | Path,
    passive_artifacts_dir: str | Path,
    output_dir: str | Path,
) -> dict[str, Path]:
    \"\"\"Diagnose passive threshold-direction behavior without changing production code.\"\"\"

    runs = load_clean_passive_runs(
        passive_results_dir=passive_results_dir,
        passive_artifacts_dir=passive_artifacts_dir,
    )
    comparison_rows: list[dict[str, Any]] = []
    score_summary_rows: list[dict[str, Any]] = []
    per_run_best_rows: list[dict[str, Any]] = []

    for run in runs:
        score_summary_rows.append(_build_score_summary_row(run))
        thresholds = sorted(set(run.member_scores + run.nonmember_scores))
        run_candidates: list[DirectionDiagnosticCandidate] = []

        for direction in DIRECTION_ORDER:
            for selection_mode in SELECTION_MODE_ORDER:
                candidate = _find_best_candidate(
                    member_scores=run.member_scores,
                    nonmember_scores=run.nonmember_scores,
                    thresholds=thresholds,
                    delta=_delta_from_run(run),
                    direction=direction,
                    require_member_favoring=selection_mode == "member_favoring_only",
                )
                if candidate is None:
                    candidate = DirectionDiagnosticCandidate(
                        direction=direction,
                        selection_mode=selection_mode,
                        threshold=None,
                        member_event_fraction=None,
                        nonmember_event_fraction=None,
                        tpr=None,
                        fpr=None,
                        epsilon_lower_empirical=None,
                        empirical_ci_lower=None,
                        empirical_ci_upper=None,
                        member_favoring=False,
                        objective_numerator_group=None,
                        objective_denominator_group=None,
                        note="No threshold satisfied the member-favoring constraint."
                        if selection_mode == "member_favoring_only"
                        else "No threshold evaluated.",
                    )
                run_candidates.append(candidate)
                comparison_rows.append(_build_comparison_row(run, candidate, len(thresholds)))

        best_member_favoring = _best_across_directions(
            candidates=run_candidates,
            selection_mode="member_favoring_only",
        )
        if best_member_favoring is not None:
            per_run_best_rows.append(
                _build_comparison_row(
                    run=run,
                    candidate=best_member_favoring,
                    threshold_grid_size=len(thresholds),
                )
            )

    output_root = ensure_directory(output_dir)
    comparison_json_path = save_json(output_root / "comparison_table.json", {"rows": comparison_rows})
    score_summary_json_path = save_json(
        output_root / "score_summary_table.json",
        {"rows": score_summary_rows},
    )
    comparison_csv_path = _write_csv(output_root / "comparison_table.csv", comparison_rows)
    score_summary_csv_path = _write_csv(output_root / "score_summary_table.csv", score_summary_rows)
    markdown_path = output_root / "summary.md"
    markdown_path.write_text(
        _build_markdown_summary(
            runs=runs,
            comparison_rows=comparison_rows,
            score_summary_rows=score_summary_rows,
            per_run_best_rows=per_run_best_rows,
            comparison_csv_path=comparison_csv_path,
            score_summary_csv_path=score_summary_csv_path,
        ),
        encoding="utf-8",
    )

    return {
        "comparison_csv": comparison_csv_path,
        "comparison_json": comparison_json_path,
        "score_summary_csv": score_summary_csv_path,
        "score_summary_json": score_summary_json_path,
        "summary_markdown": markdown_path,
    }


def load_clean_passive_runs(
    *,
    passive_results_dir: str | Path,
    passive_artifacts_dir: str | Path,
) -> list[PassiveRunData]:
    runs: list[PassiveRunData] = []
    results_dir = Path(passive_results_dir)
    artifacts_dir = Path(passive_artifacts_dir)

    for audit_path in sorted(results_dir.glob("passive_audit_seed*.json")):
        record = load_audit_result(audit_path)
        if record.audit_regime != PASSIVE_AUDIT_REGIME:
            continue
        debug_artifact_path = artifacts_dir / f"{record.audit_run_id}_passive_debug.json"
        if not debug_artifact_path.exists():
            raise FileNotFoundError(f"Missing paired passive debug artifact: {debug_artifact_path}")
        debug_payload = load_json(debug_artifact_path)
        runs.append(
            PassiveRunData(
                audit_path=audit_path,
                debug_artifact_path=debug_artifact_path,
                audit_run_id=record.audit_run_id,
                training_run_id=record.training_run_id,
                auditor_variant=record.auditor_variant,
                score_type=str(debug_payload["score_name"]),
                training_seed=record.training_seed,
                epsilon_upper_theory=record.epsilon_upper_theory,
                epsilon_lower_empirical=record.epsilon_lower_empirical,
                empirical_ci_lower=record.empirical_ci_lower,
                empirical_ci_upper=record.empirical_ci_upper,
                privacy_loss_gap=record.privacy_loss_gap,
                tightness_ratio=record.tightness_ratio,
                saturation_detected=record.saturation_detected,
                selected_threshold=_optional_float(record.audit_metadata.get("selected_threshold")),
                selected_event=_optional_string(record.audit_metadata.get("selected_event")),
                member_scores=[float(score) for score in debug_payload["member_scores"]],
                nonmember_scores=[float(score) for score in debug_payload["nonmember_scores"]],
            )
        )

    return sorted(runs, key=lambda run: (_variant_rank(run.auditor_variant), run.training_seed))


def _build_score_summary_row(run: PassiveRunData) -> dict[str, Any]:
    member_summary = _summarize_scores(run.member_scores)
    nonmember_summary = _summarize_scores(run.nonmember_scores)
    thresholds = sorted(set(run.member_scores + run.nonmember_scores))
    return {
        "variant": run.auditor_variant,
        "seed": run.training_seed,
        "score_type": run.score_type,
        "lower_scores_more_member_like_by_construction": _lower_scores_more_member_like(run.score_type),
        "member_count": member_summary.count,
        "member_mean": member_summary.mean,
        "member_std": member_summary.std,
        "member_min": member_summary.min,
        "member_p05": member_summary.p05,
        "member_p25": member_summary.p25,
        "member_median": member_summary.median,
        "member_p75": member_summary.p75,
        "member_p95": member_summary.p95,
        "member_max": member_summary.max,
        "nonmember_count": nonmember_summary.count,
        "nonmember_mean": nonmember_summary.mean,
        "nonmember_std": nonmember_summary.std,
        "nonmember_min": nonmember_summary.min,
        "nonmember_p05": nonmember_summary.p05,
        "nonmember_p25": nonmember_summary.p25,
        "nonmember_median": nonmember_summary.median,
        "nonmember_p75": nonmember_summary.p75,
        "nonmember_p95": nonmember_summary.p95,
        "nonmember_max": nonmember_summary.max,
        "threshold_search_min": thresholds[0],
        "threshold_search_max": thresholds[-1],
        "threshold_grid_size": len(thresholds),
        "audit_path": str(run.audit_path),
        "debug_artifact_path": str(run.debug_artifact_path),
    }


def _find_best_candidate(
    *,
    member_scores: list[float],
    nonmember_scores: list[float],
    thresholds: list[float],
    delta: float,
    direction: str,
    require_member_favoring: bool,
) -> DirectionDiagnosticCandidate | None:
    best: DirectionDiagnosticCandidate | None = None
    best_point = -math.inf

    for threshold in thresholds:
        candidate = _evaluate_direction(
            member_scores=member_scores,
            nonmember_scores=nonmember_scores,
            threshold=threshold,
            delta=delta,
            direction=direction,
        )
        if require_member_favoring and not candidate.member_favoring:
            continue
        point = candidate.epsilon_lower_empirical if candidate.epsilon_lower_empirical is not None else -math.inf
        if point > best_point:
            best = candidate
            best_point = point
    if best is None:
        return None
    best.selection_mode = "member_favoring_only" if require_member_favoring else "unconstrained"
    return best


def _evaluate_direction(
    *,
    member_scores: list[float],
    nonmember_scores: list[float],
    threshold: float,
    delta: float,
    direction: str,
) -> DirectionDiagnosticCandidate:
    member_trials = len(member_scores)
    nonmember_trials = len(nonmember_scores)
    member_ge = sum(score >= threshold for score in member_scores)
    nonmember_ge = sum(score >= threshold for score in nonmember_scores)
    tpr_ge = member_ge / member_trials
    fpr_ge = nonmember_ge / nonmember_trials
    tpr_lower, tpr_upper = _wilson_interval(member_ge, member_trials)
    fpr_lower, fpr_upper = _wilson_interval(nonmember_ge, nonmember_trials)

    if direction == "score>=threshold":
        member_event_fraction = tpr_ge
        nonmember_event_fraction = fpr_ge
        point = _epsilon_candidate(
            numerator_rate=tpr_ge,
            denominator_rate=fpr_ge,
            delta=delta,
            denominator_floor=0.5 / max(1, nonmember_trials),
        )
        lower = _epsilon_candidate(
            numerator_rate=tpr_lower,
            denominator_rate=fpr_upper,
            delta=delta,
            denominator_floor=0.5 / max(1, nonmember_trials),
        )
        upper = _epsilon_candidate(
            numerator_rate=tpr_upper,
            denominator_rate=fpr_lower,
            delta=delta,
            denominator_floor=0.5 / max(1, nonmember_trials),
        )
        objective_numerator_group = "member"
        objective_denominator_group = "nonmember"
    elif direction == "score<threshold":
        member_event_fraction = 1.0 - tpr_ge
        nonmember_event_fraction = 1.0 - fpr_ge
        point = _epsilon_candidate(
            numerator_rate=1.0 - fpr_ge,
            denominator_rate=1.0 - tpr_ge,
            delta=delta,
            denominator_floor=0.5 / max(1, member_trials),
        )
        lower = _epsilon_candidate(
            numerator_rate=1.0 - fpr_upper,
            denominator_rate=1.0 - tpr_lower,
            delta=delta,
            denominator_floor=0.5 / max(1, member_trials),
        )
        upper = _epsilon_candidate(
            numerator_rate=1.0 - fpr_lower,
            denominator_rate=1.0 - tpr_upper,
            delta=delta,
            denominator_floor=0.5 / max(1, member_trials),
        )
        objective_numerator_group = "nonmember"
        objective_denominator_group = "member"
    else:
        raise ValueError(f"Unsupported direction: {direction}")

    return DirectionDiagnosticCandidate(
        direction=direction,
        selection_mode="unconstrained",
        threshold=threshold,
        member_event_fraction=member_event_fraction,
        nonmember_event_fraction=nonmember_event_fraction,
        tpr=member_event_fraction,
        fpr=nonmember_event_fraction,
        epsilon_lower_empirical=point,
        empirical_ci_lower=lower,
        empirical_ci_upper=upper,
        member_favoring=member_event_fraction > nonmember_event_fraction,
        objective_numerator_group=objective_numerator_group,
        objective_denominator_group=objective_denominator_group,
    )


def _build_comparison_row(
    run: PassiveRunData,
    candidate: DirectionDiagnosticCandidate,
    threshold_grid_size: int,
) -> dict[str, Any]:
    epsilon_lower = candidate.epsilon_lower_empirical
    if epsilon_lower is None:
        gap = None
        tightness_ratio = None
    else:
        tightness = compute_privacy_tightness_metrics(
            epsilon_upper_theory=run.epsilon_upper_theory,
            epsilon_lower_empirical=epsilon_lower,
        )
        gap = tightness.privacy_loss_gap
        tightness_ratio = tightness.tightness_ratio

    return {
        "variant": run.auditor_variant,
        "seed": run.training_seed,
        "direction": candidate.direction,
        "selection_mode": candidate.selection_mode,
        "threshold": candidate.threshold,
        "member_event_fraction": candidate.member_event_fraction,
        "nonmember_event_fraction": candidate.nonmember_event_fraction,
        "tpr": candidate.tpr,
        "fpr": candidate.fpr,
        "epsilon_upper_theory": run.epsilon_upper_theory,
        "epsilon_lower_empirical": candidate.epsilon_lower_empirical,
        "privacy_loss_gap": gap,
        "tightness_ratio": tightness_ratio,
        "ci_lower": candidate.empirical_ci_lower,
        "ci_upper": candidate.empirical_ci_upper,
        "saturation_detected": run.saturation_detected,
        "member_favoring": candidate.member_favoring,
        "lower_scores_more_member_like_by_construction": _lower_scores_more_member_like(run.score_type),
        "objective_numerator_group": candidate.objective_numerator_group,
        "objective_denominator_group": candidate.objective_denominator_group,
        "current_selected_threshold": run.selected_threshold,
        "current_selected_event": run.selected_event,
        "current_selected_epsilon_lower_empirical": run.epsilon_lower_empirical,
        "current_selected_ci_lower": run.empirical_ci_lower,
        "current_selected_ci_upper": run.empirical_ci_upper,
        "current_selection_relation": _selection_relation(run, candidate),
        "exact_complement_of_current_selection": _is_exact_complement_of_current_selection(run, candidate),
        "threshold_grid_size": threshold_grid_size,
        "audit_run_id": run.audit_run_id,
        "training_run_id": run.training_run_id,
        "score_type": run.score_type,
        "audit_path": str(run.audit_path),
        "debug_artifact_path": str(run.debug_artifact_path),
        "note": candidate.note,
    }


def _best_across_directions(
    *,
    candidates: list[DirectionDiagnosticCandidate],
    selection_mode: str,
) -> DirectionDiagnosticCandidate | None:
    filtered = [
        candidate
        for candidate in candidates
        if candidate.selection_mode == selection_mode and candidate.epsilon_lower_empirical is not None
    ]
    if not filtered:
        return None
    return max(filtered, key=lambda candidate: candidate.epsilon_lower_empirical or 0.0)


def _build_markdown_summary(
    *,
    runs: list[PassiveRunData],
    comparison_rows: list[dict[str, Any]],
    score_summary_rows: list[dict[str, Any]],
    per_run_best_rows: list[dict[str, Any]],
    comparison_csv_path: Path,
    score_summary_csv_path: Path,
) -> str:
    current_pathologies = sum(
        1 for run in runs if run.epsilon_lower_empirical > run.epsilon_upper_theory
    )
    unconstrained_rows = [row for row in comparison_rows if row["selection_mode"] == "unconstrained"]
    member_favoring_rows = [
        row for row in comparison_rows if row["selection_mode"] == "member_favoring_only"
    ]

    ge_unconstrained = [row for row in unconstrained_rows if row["direction"] == "score>=threshold"]
    lt_unconstrained = [row for row in unconstrained_rows if row["direction"] == "score<threshold"]
    ge_wins_unconstrained, lt_wins_unconstrained, ties_unconstrained = _direction_win_counts(
        ge_rows=ge_unconstrained,
        lt_rows=lt_unconstrained,
    )

    ge_member_favoring = [
        row
        for row in member_favoring_rows
        if row["direction"] == "score>=threshold" and row["epsilon_lower_empirical"] is not None
    ]
    lt_member_favoring = [
        row
        for row in member_favoring_rows
        if row["direction"] == "score<threshold" and row["epsilon_lower_empirical"] is not None
    ]
    ge_wins_member_favoring, lt_wins_member_favoring, ties_member_favoring = _direction_win_counts(
        ge_rows=ge_member_favoring,
        lt_rows=lt_member_favoring,
    )

    unconstrained_pathologies = sum(
        1
        for row in unconstrained_rows
        if row["epsilon_lower_empirical"] is not None
        and row["epsilon_lower_empirical"] > row["epsilon_upper_theory"]
    )
    member_favoring_pathologies = sum(
        1
        for row in member_favoring_rows
        if row["epsilon_lower_empirical"] is not None
        and row["epsilon_lower_empirical"] > row["epsilon_upper_theory"]
    )
    best_member_favoring_pathologies = sum(
        1
        for row in per_run_best_rows
        if row["epsilon_lower_empirical"] is not None
        and row["epsilon_lower_empirical"] > row["epsilon_upper_theory"]
    )

    weak_tail_rows = [
        row
        for row in ge_member_favoring
        if row["member_event_fraction"] is not None
        and row["nonmember_event_fraction"] is not None
        and (
            max(row["member_event_fraction"], row["nonmember_event_fraction"]) <= 0.1
            or abs(row["member_event_fraction"] - row["nonmember_event_fraction"]) <= 0.05
        )
    ]

    lines = [
        "# Passive Direction Diagnostic Summary",
        "",
        "## Scope",
        "",
        f"- Clean passive runs analyzed: {len(runs)}",
        f"- Comparison table: `{comparison_csv_path}`",
        f"- Score summary table: `{score_summary_csv_path}`",
        "- Production passive auditing code was not modified by this diagnostic.",
        "- `saturation_detected` is carried through unchanged from the saved production audit record because saturation is defined across auditor-strength history, not across threshold directions within one run.",
        "",
        "## Direct Answers",
        "",
        f"- Does `score>=threshold` consistently outperform `score<threshold`?",
        f"  No under the current objective. Unconstrained, `score>=threshold` wins {ge_wins_unconstrained}/12 runs, `score<threshold` wins {lt_wins_unconstrained}/12, with {ties_unconstrained} ties. Under member-favoring-only filtering, `score>=threshold` wins {ge_wins_member_favoring}/12 runs, `score<threshold` wins {lt_wins_member_favoring}/12, with {ties_member_favoring} ties.",
        f"- When constrained to member-favoring events only, do the pathological `epsilon_lower_empirical > epsilon_upper_theory` cases mostly disappear?",
        f"  Yes, relative to the current selected production results. The current selected passive runs have {current_pathologies}/12 warnings. Across all 24 unconstrained direction rows there are {unconstrained_pathologies} warnings, across all 24 member-favoring-only rows there are {member_favoring_pathologies} warnings, and among the single best member-favoring event per run there are {best_member_favoring_pathologies}/12 warnings.",
        "- Are the best `score>=threshold` events genuinely discriminative, or still based on weak overlap / tiny tails?",
        f"  They still look weak. {len(weak_tail_rows)}/{len(ge_member_favoring) or 1} member-favoring `score>=threshold` optima are either tiny-tail events or have member/non-member event fractions within 0.05 of each other.",
        "- Is the current instability more consistent with (a) wrong event-direction/complement selection, (b) weak score separability, or (c) both?",
        "  Both. The current `score<threshold` objective reuses the complement event in a way that places the numerator on the non-member side, but the raw passive scores also show heavy overlap, so fixing direction alone is unlikely to make the passive regime scientifically reliable.",
        "",
        "## Key Observations",
        "",
        "- All four passive score definitions are higher-is-more-confident scores by construction, so lower scores are not naturally more member-like.",
        "- The current selected production events all use `score<threshold`.",
        "- The `score<threshold` direction, under the current objective, uses a non-member numerator and a member denominator. That is diagnostic evidence that complement selection is part of the instability.",
        "- The raw score summaries still show substantial overlap between member and non-member distributions, especially in medians and interquartile ranges.",
        "- Cases where the empirical lower bound exceeds the theoretical upper bound are diagnostic warnings only. They should not be interpreted as substantive claims.",
        "",
        "## Recommendation",
        "",
        "The next code change should start with restricting threshold search to member-favoring events and then explicitly repairing the event-direction logic so the lower-bound objective always uses a member-side numerator for the event being evaluated. After that, re-evaluate whether the passive scores remain too overlapped to support stable empirical lower bounds.",
        "",
    ]

    return "\n".join(lines)


def _direction_win_counts(
    *,
    ge_rows: list[dict[str, Any]],
    lt_rows: list[dict[str, Any]],
) -> tuple[int, int, int]:
    ge_by_key = {(row["variant"], row["seed"]): row for row in ge_rows}
    lt_by_key = {(row["variant"], row["seed"]): row for row in lt_rows}
    ge_wins = 0
    lt_wins = 0
    ties = 0
    for key in sorted(set(ge_by_key) | set(lt_by_key)):
        ge_value = ge_by_key.get(key, {}).get("epsilon_lower_empirical")
        lt_value = lt_by_key.get(key, {}).get("epsilon_lower_empirical")
        if ge_value is None or lt_value is None:
            continue
        if ge_value > lt_value:
            ge_wins += 1
        elif lt_value > ge_value:
            lt_wins += 1
        else:
            ties += 1
    return ge_wins, lt_wins, ties


def _selection_relation(run: PassiveRunData, candidate: DirectionDiagnosticCandidate) -> str:
    if run.selected_threshold is None or run.selected_event is None or candidate.threshold is None:
        return "unknown"
    if _same_threshold(candidate.threshold, run.selected_threshold) and candidate.direction == run.selected_event:
        return "matches_current_selection"
    if _is_exact_complement_of_current_selection(run, candidate):
        return "exact_complement_of_current_selection"
    return "genuinely_different_optimum"


def _is_exact_complement_of_current_selection(
    run: PassiveRunData,
    candidate: DirectionDiagnosticCandidate,
) -> bool:
    if run.selected_threshold is None or run.selected_event is None or candidate.threshold is None:
        return False
    return _same_threshold(candidate.threshold, run.selected_threshold) and {
        run.selected_event,
        candidate.direction,
    } == {"score<threshold", "score>=threshold"}


def _same_threshold(left: float, right: float, tolerance: float = 1e-12) -> bool:
    return abs(left - right) <= tolerance


def _summarize_scores(values: list[float]) -> ScoreSummary:
    ordered = sorted(values)
    if not ordered:
        raise ValueError("Cannot summarize an empty score list.")
    return ScoreSummary(
        count=len(ordered),
        mean=float(statistics.fmean(ordered)),
        std=float(statistics.stdev(ordered)) if len(ordered) > 1 else 0.0,
        min=float(ordered[0]),
        p05=_percentile(ordered, 0.05),
        p25=_percentile(ordered, 0.25),
        median=_percentile(ordered, 0.50),
        p75=_percentile(ordered, 0.75),
        p95=_percentile(ordered, 0.95),
        max=float(ordered[-1]),
    )


def _percentile(values: list[float], q: float) -> float:
    if len(values) == 1:
        return float(values[0])
    position = (len(values) - 1) * q
    lower_index = math.floor(position)
    upper_index = math.ceil(position)
    if lower_index == upper_index:
        return float(values[lower_index])
    weight = position - lower_index
    return float(values[lower_index] * (1.0 - weight) + values[upper_index] * weight)


def _lower_scores_more_member_like(score_type: str) -> bool:
    direction_by_score = {
        "max_probability": False,
        "negative_loss": False,
        "logit_margin": False,
        "score_fusion": False,
    }
    return direction_by_score.get(score_type, False)


def _write_csv(path: Path, rows: list[dict[str, Any]]) -> Path:
    ensure_directory(path.parent)
    if not rows:
        raise ValueError(f"No rows available to write CSV: {path}")
    fieldnames = list(rows[0].keys())
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path


def _delta_from_run(run: PassiveRunData) -> float:
    payload = load_json(run.audit_path)
    return float(payload["delta"])


def _variant_rank(variant: str) -> int:
    try:
        return VARIANT_ORDER.index(variant)
    except ValueError:
        return len(VARIANT_ORDER)


def _optional_float(value: Any) -> float | None:
    if value is None:
        return None
    return float(value)


def _optional_string(value: Any) -> str | None:
    if value is None:
        return None
    return str(value)
""",
    "src/dp_audit_tightness/reporting/plotting.py": """from __future__ import annotations

from pathlib import Path


def plot_tightness_curve(
    auditor_labels: list[str],
    epsilon_upper_series: list[float],
    epsilon_lower_series: list[float],
    output_path: str | Path,
) -> Path:
    \"\"\"Plot theoretical upper bounds against empirical lower bounds across auditors.\"\"\"

    try:
        import matplotlib.pyplot as plt
    except ImportError as exc:
        raise RuntimeError("matplotlib is required for plotting. Install project dependencies first.") from exc

    target = Path(output_path)
    target.parent.mkdir(parents=True, exist_ok=True)
    figure, axis = plt.subplots(figsize=(8, 4.5))
    axis.plot(auditor_labels, epsilon_upper_series, marker="o", label="epsilon_upper_theory")
    axis.plot(auditor_labels, epsilon_lower_series, marker="s", label="epsilon_lower_empirical")
    axis.set_ylabel("privacy loss epsilon")
    axis.set_title("Theoretical upper bounds vs empirical lower bounds")
    axis.legend()
    axis.grid(alpha=0.3)
    figure.tight_layout()
    figure.savefig(target, dpi=200)
    plt.close(figure)
    return target

""",
    "src/dp_audit_tightness/reporting/summary.py": """from __future__ import annotations

from collections import defaultdict
import math
import statistics
from typing import Any

from dp_audit_tightness.utils.paths import build_run_id
from dp_audit_tightness.utils.results import AggregatedSummaryRecord, AuditRunRecord, TrainingRunRecord


def aggregate_audit_records(records: list[AuditRunRecord]) -> list[AggregatedSummaryRecord]:
    grouped: dict[tuple[str, str, str, str], list[AuditRunRecord]] = defaultdict(list)
    for record in records:
        key = (record.dataset, record.model_name, record.audit_regime, record.auditor_variant)
        grouped[key].append(record)

    summaries: list[AggregatedSummaryRecord] = []
    for (dataset, model_name, audit_regime, auditor_variant), group in grouped.items():
        lower_values = [record.epsilon_lower_empirical for record in group]
        upper_values = [record.epsilon_upper_theory for record in group]
        gap_values = [record.privacy_loss_gap for record in group]
        ratio_values = [record.tightness_ratio for record in group if record.tightness_ratio is not None]
        saturation_rate = sum(record.saturation_detected for record in group) / len(group)
        summaries.append(
            AggregatedSummaryRecord(
                summary_id=build_run_id("summary", f"{audit_regime}_{auditor_variant}"),
                dataset=dataset,
                model_name=model_name,
                audit_regime=audit_regime,
                auditor_variant=auditor_variant,
                num_runs=len(group),
                mean_epsilon_upper_theory=statistics.fmean(upper_values),
                mean_epsilon_lower_empirical=statistics.fmean(lower_values),
                std_epsilon_lower_empirical=statistics.pstdev(lower_values) if len(group) > 1 else 0.0,
                mean_privacy_loss_gap=statistics.fmean(gap_values),
                mean_tightness_ratio=statistics.fmean(ratio_values) if ratio_values else None,
                saturation_rate=saturation_rate,
            )
        )
    return summaries


def summarize_saturation(records: list[AuditRunRecord]) -> dict[str, float]:
    if not records:
        return {"saturation_rate": math.nan}
    return {"saturation_rate": sum(record.saturation_detected for record in records) / len(records)}


def build_audit_run_summary_rows(
    training_records: dict[str, TrainingRunRecord],
    audit_records: list[AuditRunRecord],
) -> list[dict[str, Any]]:
    utility_keys = sorted(
        {
            key
            for record in audit_records
            for key in record.utility_metrics.keys()
        }
    )
    rows: list[dict[str, Any]] = []
    for record in audit_records:
        if record.training_run_id not in training_records:
            raise ValueError(
                f"Missing training record for audit run {record.audit_run_id}: {record.training_run_id}"
            )
        row: dict[str, Any] = {
            "run_id": record.audit_run_id,
            "seed": record.training_seed,
            "audit_regime": record.audit_regime,
            "auditor_variant": record.auditor_variant,
            "epsilon_upper_theory": record.epsilon_upper_theory,
            "epsilon_lower_empirical": record.epsilon_lower_empirical,
            "privacy_loss_gap": record.privacy_loss_gap,
            "tightness_ratio": record.tightness_ratio,
            "saturation_detected": record.saturation_detected,
        }
        for key in utility_keys:
            row[f"utility_{key}"] = record.utility_metrics.get(key)
        rows.append(row)
    return rows
""",
    "src/dp_audit_tightness/training/__init__.py": """\"\"\"DP-SGD training utilities.\"\"\"

""",
    "src/dp_audit_tightness/training/dp_sgd.py": """from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import logging
from typing import Any

from dp_audit_tightness.config import TrainExperimentConfig
from dp_audit_tightness.data.datasets import DatasetBundle, load_dataset_bundle
from dp_audit_tightness.models.io import export_inference_state_dict
from dp_audit_tightness.models.simple_mlp import build_model
from dp_audit_tightness.privacy.accounting import compute_theoretical_upper_bound
from dp_audit_tightness.privacy.pld_accounting import compute_epsilon_pld
from dp_audit_tightness.utils.paths import build_run_id, ensure_directory
from dp_audit_tightness.utils.results import TrainingRunRecord


@dataclass(slots=True)
class TrainingOutcome:
    record: TrainingRunRecord
    checkpoint_path: Path | None
    model_state_dict: dict[str, Any] | None = None


def train_dp_sgd(
    config: TrainExperimentConfig,
    logger: logging.Logger,
    *,
    dataset_bundle: DatasetBundle | None = None,
    save_checkpoint: bool | None = None,
    run_descriptor: str | None = None,
    return_model_state: bool = False,
) -> TrainingOutcome:
    try:
        import torch
        from opacus import PrivacyEngine
        from torch.utils.data import DataLoader
    except ImportError as exc:
        raise RuntimeError(
            "torch, torchvision, and opacus are required for DP-SGD training. Install project dependencies first."
        ) from exc

    active_bundle = dataset_bundle or load_dataset_bundle(config.dataset, split_seed=config.run.split_seed)
    train_loader = DataLoader(
        active_bundle.train_dataset,
        batch_size=config.training.batch_size,
        shuffle=True,
        num_workers=config.dataset.num_workers,
    )
    eval_loader = DataLoader(
        active_bundle.eval_dataset,
        batch_size=config.training.eval_batch_size,
        shuffle=False,
        num_workers=config.dataset.num_workers,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(config.model).to(device)
    optimizer = _build_optimizer(model, config)
    criterion = torch.nn.CrossEntropyLoss()

    privacy_engine = PrivacyEngine(accountant=config.privacy.accountant)
    model, optimizer, train_loader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=train_loader,
        noise_multiplier=config.training.noise_multiplier,
        max_grad_norm=config.training.clipping_norm,
    )

    logger.info("Starting DP-SGD training.")
    for epoch_index in range(config.training.epochs):
        model.train()
        running_loss = 0.0
        for inputs, targets in train_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            optimizer.zero_grad()
            logits = model(inputs)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()
            running_loss += float(loss.item())
        logger.info(
            "Completed epoch %s/%s with average_loss=%.4f",
            epoch_index + 1,
            config.training.epochs,
            running_loss / max(1, len(train_loader)),
        )

    utility_metrics = _evaluate_classifier(model=model, data_loader=eval_loader, device=device)
    sampling_rate = config.training.sampling_rate or (
        config.training.batch_size / active_bundle.train_size
    )
    epsilon_upper_theory = compute_theoretical_upper_bound(
        privacy_engine=privacy_engine,
        delta=config.privacy.delta,
    )

    # PLD-based accounting for a tighter upper bound.
    num_steps = config.training.epochs * len(train_loader)
    try:
        pld_result = compute_epsilon_pld(
            noise_multiplier=config.training.noise_multiplier,
            sampling_rate=sampling_rate,
            num_steps=num_steps,
            delta=config.privacy.delta,
        )
        epsilon_upper_pld = pld_result["epsilon_pld"]
        pld_backend = pld_result["backend_used"]
        logger.info(
            "PLD accounting: epsilon_pld=%.4f (backend=%s) vs RDP epsilon=%.4f",
            epsilon_upper_pld, pld_backend, epsilon_upper_theory,
        )
    except Exception as exc:
        logger.warning("PLD accounting unavailable: %s", exc)
        epsilon_upper_pld = None
        pld_backend = None

    training_run_id = build_run_id(
        prefix="train",
        descriptor=run_descriptor or config.experiment_name,
        seed=config.run.training_seed,
    )
    save_checkpoint = config.run.save_checkpoint if save_checkpoint is None else save_checkpoint
    inference_state_dict = export_inference_state_dict(model)
    checkpoint_path = None
    if save_checkpoint:
        checkpoint_dir = ensure_directory(Path(config.run.results_root) / "training" / "checkpoints")
        checkpoint_path = checkpoint_dir / f"{training_run_id}.pt"
        torch.save(inference_state_dict, checkpoint_path)

    record = TrainingRunRecord(
        training_run_id=training_run_id,
        experiment_name=config.experiment_name,
        dataset=config.dataset.name,
        split_seed=config.run.split_seed,
        training_seed=config.run.training_seed,
        model_name=config.model.name,
        optimizer_name=config.training.optimizer.name,
        learning_rate=config.training.optimizer.learning_rate,
        weight_decay=config.training.optimizer.weight_decay,
        clipping_norm=config.training.clipping_norm,
        noise_multiplier=config.training.noise_multiplier,
        batch_size=config.training.batch_size,
        epochs=config.training.epochs,
        sampling_rate=sampling_rate,
        delta=config.privacy.delta,
        epsilon_upper_theory=epsilon_upper_theory,
        utility_metrics=utility_metrics,
        model_artifact_path=str(checkpoint_path) if checkpoint_path else None,
        epsilon_upper_pld=epsilon_upper_pld,
        pld_accounting_backend=pld_backend,
        notes=config.run.notes,
    )
    return TrainingOutcome(
        record=record,
        checkpoint_path=checkpoint_path,
        model_state_dict=inference_state_dict if return_model_state else None,
    )


def _build_optimizer(model, config: TrainExperimentConfig):
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for optimizer construction.") from exc

    name = config.training.optimizer.name.lower()
    if name != "sgd":
        raise NotImplementedError("The starter scaffold currently implements SGD only.")
    return torch.optim.SGD(
        model.parameters(),
        lr=config.training.optimizer.learning_rate,
        weight_decay=config.training.optimizer.weight_decay,
        momentum=config.training.optimizer.momentum,
    )


def _evaluate_classifier(model, data_loader, device) -> dict[str, float]:
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("torch is required for model evaluation.") from exc

    model.eval()
    correct = 0
    total = 0
    cumulative_loss = 0.0
    criterion = torch.nn.CrossEntropyLoss()
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            logits = model(inputs)
            loss = criterion(logits, targets)
            cumulative_loss += float(loss.item()) * targets.size(0)
            predictions = logits.argmax(dim=1)
            correct += int((predictions == targets).sum().item())
            total += int(targets.size(0))

    accuracy = correct / max(1, total)
    average_loss = cumulative_loss / max(1, total)
    return {"accuracy": accuracy, "loss": average_loss}
""",
    "src/dp_audit_tightness/utils/__init__.py": """\"\"\"Utilities for logging, seeds, paths, and results.\"\"\"
""",
    "src/dp_audit_tightness/utils/logging_utils.py": """from __future__ import annotations

import logging
from typing import Any


def configure_logging(level: int = logging.INFO) -> logging.Logger:
    logging.basicConfig(
        level=level,
        format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    )
    return logging.getLogger("dp_audit_tightness")


def format_kv_fields(**fields: Any) -> str:
    return " ".join(f"{key}={value}" for key, value in fields.items())

""",
    "src/dp_audit_tightness/utils/paths.py": """from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import re


def utc_timestamp_slug() -> str:
    return datetime.now(tz=timezone.utc).strftime("%Y%m%dT%H%M%SZ")


def build_run_id(prefix: str, descriptor: str, seed: int | None = None) -> str:
    clean_descriptor = re.sub(r"[^a-zA-Z0-9]+", "_", descriptor).strip("_").lower()
    suffix = utc_timestamp_slug()
    if seed is None:
        return f"{prefix}_{clean_descriptor}_{suffix}"
    return f"{prefix}_{clean_descriptor}_seed{seed}_{suffix}"


def ensure_directory(path: str | Path) -> Path:
    resolved = Path(path)
    resolved.mkdir(parents=True, exist_ok=True)
    return resolved

""",
    "src/dp_audit_tightness/utils/results.py": """from __future__ import annotations

from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Mapping
import csv
import json

from dp_audit_tightness.utils.paths import ensure_directory
from dp_audit_tightness.utils.validation import (
    validate_audit_run_payload,
    validate_summary_payload,
    validate_training_run_payload,
)


CANARY_AUDIT_REGIME = "evaluator_controlled_canary_stress_test"
PASSIVE_AUDIT_REGIME = "passive_final_model_only"


@dataclass(slots=True)
class TrainingRunRecord:
    training_run_id: str
    experiment_name: str
    dataset: str
    split_seed: int
    training_seed: int
    model_name: str
    optimizer_name: str
    learning_rate: float
    weight_decay: float
    clipping_norm: float
    noise_multiplier: float
    batch_size: int
    epochs: int
    sampling_rate: float
    delta: float
    epsilon_upper_theory: float
    utility_metrics: dict[str, float]
    model_artifact_path: str | None
    epsilon_upper_pld: float | None = None
    pld_accounting_backend: str | None = None
    config_snapshot_path: str | None = None
    created_at_utc: str = field(default_factory=lambda: _utc_now_iso())
    notes: str | None = None

    @classmethod
    def from_dict(cls, payload: Mapping[str, Any]) -> "TrainingRunRecord":
        return cls(**dict(payload))

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    def to_flat_dict(self) -> dict[str, Any]:
        flat = {key: value for key, value in asdict(self).items() if key != "utility_metrics"}
        for key, value in self.utility_metrics.items():
            flat[f"utility_{key}"] = value
        return flat


@dataclass(slots=True)
class AuditRunRecord:
    audit_run_id: str
    training_run_id: str
    dataset: str
    split_seed: int
    training_seed: int
    model_name: str
    optimizer_name: str
    clipping_norm: float
    noise_multiplier: float
    batch_size: int
    epochs: int
    sampling_rate: float
    delta: float
    epsilon_upper_theory: float
    utility_metrics: dict[str, float]
    audit_regime: str
    auditor_variant: str
    auditor_strength_rank: int
    repeated_seeds: list[int]
    epsilon_lower_empirical: float
    empirical_ci_lower: float | None
    empirical_ci_upper: float | None
    privacy_loss_gap: float
    tightness_ratio: float | None
    saturation_detected: bool
    saturation_reason: str | None
    audit_mode: str | None = None
    audit_metadata: dict[str, Any] = field(default_factory=dict)
    created_at_utc: str = field(default_factory=lambda: _utc_now_iso())

    @classmethod
    def from_dict(cls, payload: Mapping[str, Any]) -> "AuditRunRecord":
        return cls(**dict(payload))

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    def to_flat_dict(self) -> dict[str, Any]:
        flat = {
            key: value
            for key, value in asdict(self).items()
            if key not in {"utility_metrics", "audit_metadata", "repeated_seeds"}
        }
        for key, value in self.utility_metrics.items():
            flat[f"utility_{key}"] = value
        flat["repeated_seeds"] = json.dumps(self.repeated_seeds)
        flat["audit_metadata"] = json.dumps(self.audit_metadata, sort_keys=True)
        return flat


@dataclass(slots=True)
class AggregatedSummaryRecord:
    summary_id: str
    dataset: str
    model_name: str
    audit_regime: str
    auditor_variant: str
    num_runs: int
    mean_epsilon_upper_theory: float
    mean_epsilon_lower_empirical: float
    std_epsilon_lower_empirical: float
    mean_privacy_loss_gap: float
    mean_tightness_ratio: float | None
    saturation_rate: float
    created_at_utc: str = field(default_factory=lambda: _utc_now_iso())

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    def to_flat_dict(self) -> dict[str, Any]:
        return asdict(self)


def save_training_result(record: TrainingRunRecord, results_root: str | Path) -> Path:
    root = Path(results_root)
    validate_training_run_payload(record.to_dict())
    json_path = root / "training" / f"{record.training_run_id}.json"
    save_json(json_path, record.to_dict())
    append_csv_row(root / "training" / "training_runs.csv", record.to_flat_dict())
    return json_path


def save_audit_result(record: AuditRunRecord, results_root: str | Path) -> Path:
    root = Path(results_root)
    validate_audit_run_payload(record.to_dict())
    subdirectory = _audit_subdirectory(record.audit_regime)
    json_path = root / "audits" / subdirectory / f"{record.audit_run_id}.json"
    save_json(json_path, record.to_dict())
    append_csv_row(root / "audits" / subdirectory / "audit_runs.csv", record.to_flat_dict())
    return json_path


def save_summary_result(record: AggregatedSummaryRecord, results_root: str | Path) -> Path:
    root = Path(results_root)
    validate_summary_payload(record.to_dict())
    json_path = root / "summaries" / f"{record.summary_id}.json"
    save_json(json_path, record.to_dict())
    append_csv_row(root / "summaries" / "aggregated_summaries.csv", record.to_flat_dict())
    return json_path


def save_json(path: str | Path, payload: Mapping[str, Any]) -> Path:
    target = Path(path)
    ensure_directory(target.parent)
    with target.open("w", encoding="utf-8") as handle:
        json.dump(_to_serializable(payload), handle, indent=2, sort_keys=True)
    return target


def load_json(path: str | Path) -> dict[str, Any]:
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def load_training_result(path: str | Path) -> TrainingRunRecord:
    payload = load_json(path)
    validate_training_run_payload(payload)
    return TrainingRunRecord.from_dict(payload)


def load_audit_result(path: str | Path) -> AuditRunRecord:
    payload = load_json(path)
    validate_audit_run_payload(payload)
    return AuditRunRecord.from_dict(payload)


def load_audit_results_for_training(
    results_root: str | Path,
    audit_regime: str,
    training_run_id: str,
) -> list[AuditRunRecord]:
    records: list[AuditRunRecord] = []
    audit_dir = Path(results_root) / "audits" / _audit_subdirectory(audit_regime)
    if not audit_dir.exists():
        return records
    for path in audit_dir.glob("*.json"):
        payload = load_json(path)
        validate_audit_run_payload(payload)
        if payload.get("training_run_id") == training_run_id:
            records.append(AuditRunRecord.from_dict(payload))
    return sorted(records, key=lambda record: (record.auditor_strength_rank, record.created_at_utc))


def discover_training_result_paths(results_root: str | Path) -> list[Path]:
    return sorted((Path(results_root) / "training").glob("*.json"))


def discover_audit_result_paths(results_root: str | Path) -> list[Path]:
    root = Path(results_root)
    return sorted((root / "audits" / "canary").glob("*.json")) + sorted(
        (root / "audits" / "passive").glob("*.json")
    )


def resolve_training_config_snapshot_path(
    training_result_path: str | Path,
    training_record: TrainingRunRecord,
) -> Path:
    if training_record.config_snapshot_path:
        return Path(training_record.config_snapshot_path)
    result_path = Path(training_result_path)
    return result_path.parent / "configs" / f"{training_record.training_run_id}_config.json"


def append_csv_row(path: str | Path, row: Mapping[str, Any]) -> Path:
    target = Path(path)
    ensure_directory(target.parent)
    serialized_row = {key: _csv_value(value) for key, value in row.items()}
    write_header = not target.exists()
    with target.open("a", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(serialized_row.keys()))
        if write_header:
            writer.writeheader()
        writer.writerow(serialized_row)
    return target


def _audit_subdirectory(audit_regime: str) -> str:
    if audit_regime == CANARY_AUDIT_REGIME:
        return "canary"
    if audit_regime == PASSIVE_AUDIT_REGIME:
        return "passive"
    raise ValueError(f"Unknown audit_regime: {audit_regime}")


def _to_serializable(payload: Any) -> Any:
    if isinstance(payload, Path):
        return str(payload)
    if isinstance(payload, dict):
        return {key: _to_serializable(value) for key, value in payload.items()}
    if isinstance(payload, list):
        return [_to_serializable(value) for value in payload]
    return payload


def _csv_value(value: Any) -> Any:
    if isinstance(value, (dict, list)):
        return json.dumps(value, sort_keys=True)
    return value


def _utc_now_iso() -> str:
    return datetime.now(tz=timezone.utc).isoformat()
""",
    "src/dp_audit_tightness/utils/seeds.py": """from __future__ import annotations

import os
import random

import numpy as np


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        return
""",
    "src/dp_audit_tightness/utils/validation.py": """from __future__ import annotations

from typing import Any, Mapping


_TRAINING_REQUIRED_KEYS = {
    "training_run_id",
    "experiment_name",
    "dataset",
    "split_seed",
    "training_seed",
    "model_name",
    "optimizer_name",
    "learning_rate",
    "weight_decay",
    "clipping_norm",
    "noise_multiplier",
    "batch_size",
    "epochs",
    "sampling_rate",
    "delta",
    "epsilon_upper_theory",
    "utility_metrics",
    "model_artifact_path",
    "created_at_utc",
}

_AUDIT_REQUIRED_KEYS = {
    "audit_run_id",
    "training_run_id",
    "dataset",
    "split_seed",
    "training_seed",
    "model_name",
    "optimizer_name",
    "clipping_norm",
    "noise_multiplier",
    "batch_size",
    "epochs",
    "sampling_rate",
    "delta",
    "epsilon_upper_theory",
    "utility_metrics",
    "audit_regime",
    "auditor_variant",
    "auditor_strength_rank",
    "repeated_seeds",
    "epsilon_lower_empirical",
    "privacy_loss_gap",
    "tightness_ratio",
    "saturation_detected",
    "created_at_utc",
}

_SUMMARY_REQUIRED_KEYS = {
    "summary_id",
    "dataset",
    "model_name",
    "audit_regime",
    "auditor_variant",
    "num_runs",
    "mean_epsilon_upper_theory",
    "mean_epsilon_lower_empirical",
    "std_epsilon_lower_empirical",
    "mean_privacy_loss_gap",
    "mean_tightness_ratio",
    "saturation_rate",
    "created_at_utc",
}


def validate_training_run_payload(payload: Mapping[str, Any]) -> None:
    _validate_required_keys(payload, _TRAINING_REQUIRED_KEYS, "training result")
    _validate_mapping_field(payload, "utility_metrics", "training result")


def validate_audit_run_payload(payload: Mapping[str, Any]) -> None:
    _validate_required_keys(payload, _AUDIT_REQUIRED_KEYS, "audit result")
    _validate_mapping_field(payload, "utility_metrics", "audit result")


def validate_summary_payload(payload: Mapping[str, Any]) -> None:
    _validate_required_keys(payload, _SUMMARY_REQUIRED_KEYS, "summary result")


def _validate_required_keys(
    payload: Mapping[str, Any],
    required_keys: set[str],
    artifact_name: str,
) -> None:
    missing = sorted(required_keys - set(payload.keys()))
    if missing:
        raise ValueError(
            f"{artifact_name} is missing required keys: {', '.join(missing)}"
        )


def _validate_mapping_field(payload: Mapping[str, Any], key: str, artifact_name: str) -> None:
    value = payload.get(key)
    if not isinstance(value, Mapping):
        raise ValueError(f"{artifact_name} field '{key}' must be a mapping.")
""",
}

for rel_path, content in FILES.items():
    target = REPO_DIR / rel_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')

os.chdir(REPO_DIR)
print('repo_dir =', REPO_DIR)
print('written_files =', len(FILES))


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall',
    'torch==2.2.2', 'torchvision==0.17.2', 'torchaudio==2.2.2',
    '--index-url', 'https://download.pytorch.org/whl/cu121'
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
    'numpy>=1.26', 'pandas>=2.2', 'PyYAML>=6.0', 'matplotlib>=3.8',
    'opacus>=1.5', 'dp-accounting>=0.4', 'scikit-learn>=1.4'
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), '--no-deps'], check=True)
print('install complete')


In [ ]:
import torch

print('torch_version =', torch.__version__)
print('cuda_available =', torch.cuda.is_available())
print('device_count =', torch.cuda.device_count())
print('arch_list =', torch.cuda.get_arch_list() if torch.cuda.is_available() else [])
if torch.cuda.is_available():
    print('device_name =', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('CUDA is not available. Enable GPU in the Kaggle notebook settings before running.')


In [ ]:
import subprocess
import sys

script_path = REPO_DIR / 'codex' / 'run_framework_validation_gpu_scale.py'
cmd = [sys.executable, str(script_path)]
print('running =', ' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


In [ ]:
results_dir = REPO_DIR / 'codex' / 'results' / 'framework_validation_gpu_scale'
expected = [
    results_dir / 'framework_validation_gpu_scale_summary.json',
    results_dir / 'framework_validation_gpu_scale_rows.csv',
    results_dir / 'framework_validation_gpu_scale_checks.json',
    results_dir / 'framework_validation_gpu_scale_report.md',
    results_dir / 'framework_validation_gpu_scale_artifacts.zip',
]
for path in expected:
    print(path.name, 'exists =', path.exists())
print('bundled_download =', results_dir / 'framework_validation_gpu_scale_artifacts.zip')


In [ ]:
import json

summary_path = REPO_DIR / 'codex' / 'results' / 'framework_validation_gpu_scale' / 'framework_validation_gpu_scale_summary.json'
checks_path = REPO_DIR / 'codex' / 'results' / 'framework_validation_gpu_scale' / 'framework_validation_gpu_scale_checks.json'
rows_path = REPO_DIR / 'codex' / 'results' / 'framework_validation_gpu_scale' / 'framework_validation_gpu_scale_rows.csv'

summary = json.loads(summary_path.read_text(encoding='utf-8'))
checks = json.loads(checks_path.read_text(encoding='utf-8'))
audit_rows = summary.get('audit_rows', [])
failed_checks = [check for check in checks if check.get('status') == 'fail']
provisional = [row for row in audit_rows if row.get('result_trust') == 'provisional']
invalidated = [row for row in audit_rows if row.get('result_trust') == 'invalidated']

print('=== Validation Summary ===')
print('training_rows =', len(summary.get('training_rows', [])))
print('audit_rows =', len(audit_rows))
print('failed_checks =', len(failed_checks))
print('provisional_rows =', len(provisional))
print('invalidated_rows =', len(invalidated))
print('rows_csv =', rows_path)

if provisional:
    ranked = sorted(provisional, key=lambda row: -(row.get('epsilon_lower_conservative') or 0.0))
    print('\n=== Best Provisional Rows ===')
    for row in ranked[:5]:
        print(f"{row['dataset']} + {row['attack_name']}: eps_lower={row.get('epsilon_lower_conservative')}, eps_upper={row.get('epsilon_upper_tighter')}, trust={row.get('result_trust')}")

if invalidated:
    print('\n=== Invalidated Rows ===')
    for row in invalidated:
        print(f"{row['dataset']} + {row['attack_name']}: tags={row.get('diagnostic_tags')}")

if failed_checks:
    print('\n=== Failed Checks ===')
    for check in failed_checks[:10]:
        print(f"{check['check_id']}: {check['details']}")
